# Radiology Reporting Harness - minimal template editing pipeline

**Task.** Turn a telegraphic radiologist dictation into a complete structured
report by *minimally editing* the supplied normal template. The leaderboard
metric (RES, lower is better) rewards template-edit fidelity, not free-form
report writing.

**Approach - structured generation (Approach B/D), in two stages.**

```
dictation
   |-- segment .......... sentences, "Bones shows ..." cues, technique/history
   |                      boilerplate, and the dictated summary block
   |-- normalise ........ shorthand expansion + corpus-based spell repair
   |-- split ............ "No acute fracture or dislocation" -> two clauses
   |-- route ............ cue > field label > curated anatomy > template text
   |                      overlap > statistics mined from train reports
   |-- edit template .... replace only the contradicted normal statement,
   |                      keep every untouched field byte-identical
   |-- impression ....... reuse the dictated summary, else condense the
   |                      abnormal findings and close with the template line
   |-- validate ......... negation / laterality / measurements / no invented
                          content / untouched fields / no dropped findings
```

**Stage 2 - LLM refinement (Approach D).** The deterministic draft is then
handed to a language model together with the template and the dictation. The
model is given one fixed instruction set (Section 6) and applies it uniformly
to every case: fix mis-routed clauses, repair dictation typos into standard
radiology terms, drop leaked section headers, and order the impression. It may
not add a finding, a diagnosis, a measurement or a laterality that is not in
the dictation, and it may not change the template's field labels or their
order.

Measured on 24 held-out training cases (references withheld from the model),
the refinement stage cuts word-level RES from **0.3937 to 0.2318 (-41%)** and
is better on 21 of the 24 cases.

**No API keys appear anywhere in this notebook.** The refinement stage reads
its key from an environment variable / Kaggle Secret and is skipped when none
is present. Its 132 outputs are cached in Section 6 as `refined_reports.json`,
so the notebook reproduces `submission.csv` byte-for-byte offline; supplying a
key re-runs the stage instead of reading the cache.

**What is and is not deterministic.** Stage 1 generates the same bytes on every
run (`check_determinism.py` verifies this across hash seeds). Stage 2 is a
language-model call and is not reproducible in that sense. What *is* reproducible
is the submission: the stage-2 outputs are cached in Section 6.2, so re-running
this notebook top to bottom regenerates `submission.csv` byte-for-byte with no
network access and no API key. Determinism here is a property of reproducing the
CSV, not of generating the refinements - running the live stage under
`RRH_RUN_LLM=1` overwrites the cache and yields a different submission.

One instruction set is applied to every case; there is no per-case manual editing
anywhere in the pipeline.


## 1. Environment and data

In [ ]:
import os, sys, json, csv, subprocess

INPUT_DIR = "/kaggle/input/radiology-reporting-harness"
if not os.path.isdir(INPUT_DIR):
    # local / repository checkout fallback
    for cand in ("data", "../data", "/kaggle/input"):
        if os.path.isdir(cand) and os.path.exists(os.path.join(cand, "train.csv")):
            INPUT_DIR = cand
            break
        if os.path.isdir(cand):
            for sub in sorted(os.listdir(cand)):
                p = os.path.join(cand, sub)
                if os.path.isdir(p) and os.path.exists(os.path.join(p, "train.csv")):
                    INPUT_DIR = p
                    break
print("input dir:", INPUT_DIR, os.listdir(INPUT_DIR)[:8])

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.chdir(WORK)
os.makedirs("rrh", exist_ok=True)
open(os.path.join("rrh", "__init__.py"), "w").close()
sys.path.insert(0, os.getcwd())

try:
    import rapidfuzz  # noqa: F401
    print("rapidfuzz available")
except ImportError:
    print("rapidfuzz not available - the pure-python fallback is used (slower, identical output)")


## 2. The pipeline

Each cell below writes one module of the `rrh` package, so the whole implementation is visible in the notebook *and* importable by it.

### 2.1 Text utilities - sentence splitting, similarity, edit distance

In [ ]:
%%writefile rrh/textutil.py
"""Low-level text utilities shared by the whole pipeline.

Everything here is deterministic and dependency-free (stdlib only) so the
pipeline reproduces byte-identically on Kaggle, offline.
"""
from __future__ import annotations

import re
import unicodedata
from difflib import SequenceMatcher
from functools import lru_cache

# ---------------------------------------------------------------- whitespace


def clean_ws(text: str) -> str:
    """Normalise unicode + collapse horizontal whitespace, keep newlines."""
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace(" ", " ").replace("–", "-").replace("—", "-")
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def squash(text: str) -> str:
    """Single-line, single-spaced version of a chunk of text."""
    return re.sub(r"\s+", " ", text or "").strip()


# ---------------------------------------------------------------- key forms

_NONWORD = re.compile(r"[^a-z0-9]+")


def key(text: str) -> str:
    """Aggressive normalisation used for equality / similarity comparisons."""
    return _NONWORD.sub(" ", (text or "").lower()).strip()


STOPWORDS = frozenset(
    """a an and are as at be been but by for from had has have in into is it its
    of on or that the there these this to was were with without which who whom
    within are demonstrates demonstrate shows show reveals reveal seen noted note
    identified appears appear present evident visualized visualised""".split()
)


def content_tokens(text: str) -> list[str]:
    return [t for t in key(text).split() if t not in STOPWORDS and len(t) > 2]


def token_set(text: str) -> frozenset[str]:
    return frozenset(content_tokens(text))


def jaccard(a: frozenset, b: frozenset) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def coverage(sub: frozenset, sup: frozenset) -> float:
    """Fraction of `sub` present in `sup`."""
    if not sub:
        return 0.0
    return len(sub & sup) / len(sub)


@lru_cache(maxsize=200_000)
def ratio(a: str, b: str) -> float:
    """Character-level similarity of two normalised strings."""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def sim(a: str, b: str) -> float:
    return ratio(key(a), key(b))


# ---------------------------------------------------------------- sentences

_SENT_END = re.compile(r"(?<=[.;!?])\s+")
_DECIMAL = re.compile(r"(?<=\d)\.(?=\d)")
_ABBREVS = (
    "dr", "vs", "approx", "e.g", "i.e", "no", "cf", "etc", "mr", "ms", "st",
    "fig", "ca", "wk", "yr", "mo",
)
# telegraphic dictations frequently drop the full stop between sentences
_SENT_FINAL = (
    "seen|noted|identified|present|intact|maintained|preserved|unremarkable|"
    "normal|negative|patent|visualized|visualised|appreciated|demonstrated|observed"
)
_SENT_START = (
    "The|There|No|Mild|Moderate|Severe|Minimal|Small|Large|Normal|Multiple|"
    "Otherwise|Marked|Trace|Focal|Diffuse|Both|Overall|Findings|Impression|An?|"
    "At|Few|Multilevel|Visualized|Rest|Status|Post|Mildly|Grade"
)
# the space is optional: dictations contain run-ons such as "maintainedMinimal"
_MISSING_STOP = re.compile(rf"\b({_SENT_FINAL}) ?(?=(?:{_SENT_START})\b)")


def split_sentences(text: str) -> list[str]:
    """Split prose into sentences without breaking decimals or level labels."""
    if not text:
        return []
    guarded = _DECIMAL.sub("\x00", text)
    guarded = re.sub(r"\b([A-Z])\.(?=\s*[A-Z]\.)", lambda m: m.group(1) + "\x00", guarded)
    for ab in _ABBREVS:
        guarded = re.sub(
            rf"\b({re.escape(ab)})\.", lambda m: m.group(1) + "\x00", guarded, flags=re.I
        )
    out: list[str] = []
    for line in guarded.split("\n"):
        line = line.strip()
        if not line:
            continue
        for piece in _SENT_END.split(line):
            piece = piece.strip()
            if not piece:
                continue
            piece = _MISSING_STOP.sub(lambda m: m.group(1) + ".\x01", piece)
            for sub in piece.split("\x01"):
                sub = sub.strip()
                if sub:
                    out.append(sub.replace("\x00", "."))
    return out


def cap_first(text: str) -> str:
    """Upper-case the first alphabetic character, leave the rest untouched."""
    for i, ch in enumerate(text):
        if ch.isalpha():
            # do not lower-case ALLCAPS acronyms that already start the string
            return text[:i] + ch.upper() + text[i + 1 :]
        if ch not in "([\"' ":
            break
    return text


def end_period(text: str) -> str:
    text = text.rstrip()
    if not text:
        return text
    if text[-1] in ".;:!?":
        return text[:-1] + "." if text[-1] == ";" else text
    return text + "."


def tidy_sentence(text: str) -> str:
    """Canonical sentence rendering: trimmed, capitalised, single final period."""
    text = squash(text)
    text = re.sub(r"^[\-•*\d]+[\.\)]\s*", "", text)  # strip list bullets
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"\.{2,}", ".", text)
    if not text:
        return ""
    return end_period(cap_first(text))


# ---------------------------------------------------------------- distances


try:  # optional accelerator; the pure-python fallback gives identical results
    from rapidfuzz.distance import Levenshtein as _RF

    def levenshtein(a, b) -> int:
        return _RF.distance(a, b)

except Exception:  # pragma: no cover - Kaggle images ship rapidfuzz, but be safe
    def levenshtein(a, b) -> int:
        return _levenshtein_py(a, b)


def _levenshtein_py(a, b) -> int:
    """Edit distance over any two sequences (str or list of tokens)."""
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ai != b[j - 1]))
        prev = cur
    return prev[lb]


### 2.2 Radiology lexicon - anatomy->field concepts, shorthand, spell repair

In [ ]:
%%writefile rrh/lexicon.py
"""Curated radiology knowledge used to route findings to template fields.

Kept small and explicit: it encodes *which anatomical field a term belongs to*,
never what a finding means clinically.  Nothing here invents content.
"""
from __future__ import annotations

import re

# canonical field concept -> terms that belong to that field
CONCEPT_TERMS: dict[str, tuple[str, ...]] = {
    "BONES": (
        "bone", "bones", "osseous", "fracture", "marrow", "cortex", "cortical",
        "lytic", "blastic", "sclerotic", "osteopenia", "osteoporosis", "mineralization",
        "vertebral body", "vertebral bodies", "endplate", "spondylosis", "spur",
        "osteophyte", "avulsion", "periosteal", "bone island", "enchondroma",
        "acromion", "tuberosity", "condyle", "malleolus", "calcaneus", "scaphoid",
        "clavicle", "rib", "sternum", "pedicle", "spinous process", "odontoid",
    ),
    "JOINTS": (
        "joint", "joints", "joint space", "articular", "dislocation", "subluxation",
        "arthrosis", "arthritis", "osteoarthrosis", "osteoarthritis", "degenerative change",
        "alignment", "effusion", "facet", "sacroiliac", "acromioclavicular",
        "glenohumeral", "carpometacarpal", "interphalangeal", "metacarpophalangeal",
    ),
    "SOFT TISSUES": (
        "soft tissue", "soft tissues", "swelling", "cellulitis", "edema", "oedema",
        "foreign body", "subcutaneous", "hematoma", "seroma", "abscess", "lipoma",
        "phlebolith", "calcified lymph node", "fat pad", "ganglion",
    ),
    "MUSCLES": (
        "muscle", "muscles", "musculature", "atrophy", "fatty infiltration",
        "myotendinous", "strain", "bulk",
    ),
    "TENDONS": (
        "tendon", "tendons", "tendinosis", "tendinopathy", "tendinitis", "tenosynovitis",
        "supraspinatus", "infraspinatus", "subscapularis", "teres minor", "biceps",
        "achilles", "peroneal", "quadriceps", "patellar tendon", "rotator cuff",
    ),
    "LIGAMENTS": (
        "ligament", "ligaments", "ligamentous", "cruciate", "collateral", "acl", "pcl",
        "mcl", "lcl", "lisfranc", "spring ligament", "deltoid ligament", "retinaculum",
        "syndesmosis", "talofibular",
    ),
    "MENISCI": ("meniscus", "menisci", "meniscal", "bucket-handle", "mucoid degeneration"),
    "CARTILAGE": ("cartilage", "chondral", "chondromalacia", "osteochondral"),
    "BURSAE": ("bursa", "bursae", "bursitis", "subacromial", "subdeltoid", "prepatellar"),
    "NERVES": ("nerve", "nerves", "neuroma", "median nerve", "ulnar nerve", "sciatic"),
    "LUNGS": (
        "lung", "lungs", "pulmonary", "airspace", "air-space", "consolidation",
        "opacity", "infiltrate", "atelectasis", "nodule", "emphysema", "bronchiectasis",
        "interstitial", "edema", "reticular", "ground-glass", "airway", "bronch",
    ),
    "PLEURA": ("pleura", "pleural", "effusion", "pneumothorax", "pleural thickening"),
    "HEART": ("heart", "cardiac", "cardiomegaly", "cardiac silhouette"),
    "MEDIASTINUM": (
        "mediastinum", "mediastinal", "hilum", "hila", "hilar", "aorta", "aortic",
        "great vessel", "trachea",
    ),
    "DIAPHRAGM": ("diaphragm", "diaphragmatic", "hemidiaphragm", "subdiaphragmatic", "free air"),
    "SUPPORT DEVICES": (
        "line", "tube", "catheter", "pacemaker", "device", "stent", "port",
        "endotracheal", "picc", "drain", "hardware", "screw", "plate", "prosthesis",
    ),
    "LIVER": ("liver", "hepatic", "hepatomegaly", "steatosis", "fatty liver"),
    "GALLBLADDER": ("gallbladder", "gallstone", "cholelithiasis", "biliary", "cbd"),
    "KIDNEYS": ("kidney", "kidneys", "renal", "hydronephrosis", "calculus", "nephrolithiasis"),
    "SPLEEN": ("spleen", "splenic", "splenomegaly"),
    "PANCREAS": ("pancreas", "pancreatic"),
    "BLADDER": ("bladder", "urinary bladder", "vesical"),
    "BOWEL": ("bowel", "colon", "small bowel", "ileus", "obstruction", "appendix"),
    "UTERUS": ("uterus", "uterine", "endometrium", "endometrial", "myometrium", "fibroid"),
    "OVARIES": ("ovary", "ovaries", "adnexa", "adnexal", "follicle"),
    "PROSTATE": ("prostate", "prostatic", "seminal vesicle"),
    "BRAIN": (
        "brain", "cerebral", "cerebellum", "parenchyma", "white matter", "gray matter",
        "infarct", "hemorrhage", "midline shift", "ventricle", "ventricular", "gliosis",
        "encephalomalacia", "mass effect",
    ),
    "SPINAL CORD": ("cord", "spinal cord", "myelomalacia", "syrinx", "cord signal"),
    "DISC": (
        "disc", "disk", "herniation", "protrusion", "extrusion", "bulge", "annular",
        "desiccation", "canal stenosis", "foraminal", "thecal sac", "osteophyte complex",
    ),
    "ALIGNMENT": (
        "alignment", "lordosis", "kyphosis", "scoliosis", "spondylolisthesis",
        "listhesis", "curvature", "straightening",
    ),
    "VESSELS": ("artery", "arterial", "vein", "venous", "stenosis", "aneurysm", "thrombus"),
    "SINUSES": ("sinus", "sinuses", "maxillary", "ethmoid", "sphenoid", "frontal sinus", "mucosal",
                "mastoid", "mastoiditis", "mastoid air cells"),
    "BILIARY": ("bile duct", "biliary", "cbd", "common bile duct", "common hepatic duct",
                "choledocholithiasis", "intrahepatic biliary", "biliary radicle",
                "biliary dilatation", "biliary stricture"),
    "PANCREATIC DUCT": ("pancreatic duct", "duct of wirsung", "main pancreatic duct"),
    "URETERS": ("ureter", "ureters", "ureteric", "ureteral", "hydroureter",
                "hydroureteronephrosis", "ureterovesical", "collecting system"),
    "SPINAL CANAL": ("spinal canal", "canal stenosis", "ap diameter", "central canal",
                     "canal diameter", "thecal"),
    "DISC SPACES": ("disc space", "disc height", "intervertebral", "disc space narrowing"),
    "NEURAL FORAMINA": ("neural foramen", "neural foramina", "foraminal", "foramina"),
    "FACET JOINTS": ("facet", "facets", "facet joint", "facet arthropathy", "zygapophyseal"),
    "ORBITS": ("orbit", "orbits", "globe", "optic nerve", "extraocular"),
}

# template label (upper-cased) -> canonical concept
LABEL_ALIASES: dict[str, str] = {
    "BONE": "BONES", "BONES": "BONES", "OSSEOUS STRUCTURES": "BONES",
    "OSSEOUS": "BONES", "BONES AND JOINTS": "BONES", "SKELETAL": "BONES",
    "VERTEBRAL BODIES": "BONES", "VERTEBRAL BODIES AND ALIGNMENT": "BONES",
    "OSSEOUS STRUCTURES AND ALIGNMENT": "BONES", "BONY STRUCTURES": "BONES",
    "JOINT": "JOINTS", "JOINTS": "JOINTS", "JOINT SPACES": "JOINTS",
    "JOINT SPACES AND ALIGNMENT": "JOINTS", "ARTICULATIONS": "JOINTS",
    "SOFT TISSUE": "SOFT TISSUES", "SOFT TISSUES": "SOFT TISSUES",
    "PARAVERTEBRAL SOFT TISSUES": "SOFT TISSUES", "PARASPINAL SOFT TISSUES": "SOFT TISSUES",
    "SURROUNDING SOFT TISSUES": "SOFT TISSUES",
    "MUSCLES": "MUSCLES", "MUSCULATURE": "MUSCLES", "MUSCLE": "MUSCLES",
    "TENDONS": "TENDONS", "TENDON": "TENDONS", "ROTATOR CUFF": "TENDONS",
    "LIGAMENTS": "LIGAMENTS", "LIGAMENT": "LIGAMENTS",
    "MENISCI": "MENISCI", "MENISCUS": "MENISCI",
    "ARTICULAR CARTILAGE": "CARTILAGE", "CARTILAGE": "CARTILAGE",
    "BURSAE": "BURSAE", "BURSA": "BURSAE",
    "NERVES": "NERVES", "NERVE": "NERVES",
    "LUNGS": "LUNGS", "LUNG": "LUNGS", "LUNGS/AIRWAYS": "LUNGS",
    "LUNGS AND AIRWAYS": "LUNGS", "LUNG PARENCHYMA": "LUNGS", "AIRWAYS": "LUNGS",
    "PLEURA": "PLEURA", "PLEURAL SPACES": "PLEURA", "PLEURAL SPACE": "PLEURA",
    "HEART": "HEART", "CARDIAC SILHOUETTE": "HEART", "CARDIOVASCULAR": "HEART",
    "CARDIOMEDIASTINAL SILHOUETTE": "HEART",
    "MEDIASTINUM": "MEDIASTINUM", "MEDIASTINUM/HILA": "MEDIASTINUM",
    "MEDIASTINUM AND HILA": "MEDIASTINUM", "HILA": "MEDIASTINUM",
    "DIAPHRAGM": "DIAPHRAGM", "DIAPHRAGMS": "DIAPHRAGM",
    "SUPPORT DEVICES": "SUPPORT DEVICES", "LINES/TUBES/SUPPORT DEVICES": "SUPPORT DEVICES",
    "LINES AND TUBES": "SUPPORT DEVICES", "DEVICES": "SUPPORT DEVICES",
    "LIVER": "LIVER", "GALLBLADDER": "GALLBLADDER", "GALLBLADDER AND BILIARY": "GALLBLADDER",
    "BILIARY SYSTEM": "GALLBLADDER", "KIDNEYS": "KIDNEYS", "KIDNEY": "KIDNEYS",
    "SPLEEN": "SPLEEN", "PANCREAS": "PANCREAS", "BLADDER": "BLADDER",
    "URINARY BLADDER": "BLADDER", "BOWEL": "BOWEL", "BOWEL GAS PATTERN": "BOWEL",
    "UTERUS": "UTERUS", "OVARIES": "OVARIES", "ADNEXA": "OVARIES", "PROSTATE": "PROSTATE",
    "BRAIN": "BRAIN", "BRAIN PARENCHYMA": "BRAIN", "PARENCHYMA": "BRAIN",
    "SPINAL CORD": "SPINAL CORD", "CORD": "SPINAL CORD",
    "ALIGNMENT": "ALIGNMENT", "SINUSES": "SINUSES", "PARANASAL SINUSES": "SINUSES",
    "ORBITS": "ORBITS", "VESSELS": "VESSELS", "VASCULAR": "VESSELS",
    "BILIARY TREE": "BILIARY", "BILE DUCTS": "BILIARY", "BILIARY": "BILIARY",
    "COMMON BILE DUCT": "BILIARY", "PANCREATIC DUCT": "PANCREATIC DUCT",
    "URETERS": "URETERS", "URETERS AND URINARY BLADDER": "URETERS",
    "SPINAL CANAL": "SPINAL CANAL", "CENTRAL CANAL": "SPINAL CANAL",
    "DISC SPACES": "DISC SPACES", "INTERVERTEBRAL DISCS": "DISC SPACES",
    "DISCS": "DISC SPACES", "NEURAL FORAMINA": "NEURAL FORAMINA",
    "FACET JOINTS": "FACET JOINTS", "SINUSES AND MASTOIDS": "SINUSES",
    "MASTOIDS": "SINUSES", "VERTEBRAE": "BONES", "VERTEBRAL BODIES/ALIGNMENT": "BONES",
}

_STEM_SUFFIXES = ("ies", "ives", "ing", "ous", "als", "ial", "ic", "es", "s", "al", "ar", "y")


def stem(word: str) -> str:
    w = word.lower()
    for suf in _STEM_SUFFIXES:
        if len(w) > len(suf) + 3 and w.endswith(suf):
            return w[: -len(suf)]
    return w


def concept_for_label(label: str) -> str | None:
    lab = re.sub(r"\s+", " ", (label or "").upper()).strip()
    if lab in LABEL_ALIASES:
        return LABEL_ALIASES[lab]
    for alias, concept in LABEL_ALIASES.items():
        if alias in lab:
            return concept
    return None


_TERM_INDEX: dict[str, set[str]] = {}
for _concept, _terms in CONCEPT_TERMS.items():
    for _t in _terms:
        _TERM_INDEX.setdefault(_t.lower(), set()).add(_concept)


def concept_hits(text: str) -> dict[str, int]:
    """How many curated terms of each concept occur in `text`."""
    low = " " + re.sub(r"[^a-z0-9 ]", " ", (text or "").lower()) + " "
    hits: dict[str, int] = {}
    for term, concepts in _TERM_INDEX.items():
        if f" {term} " in low or (" " in term and term in low):
            for c in concepts:
                hits[c] = hits.get(c, 0) + 1
    return hits


# ------------------------------------------------------------------ shorthand
# High-precision expansions of dictation shorthand and recurrent misspellings.
# Only unambiguous entries: nothing here changes clinical meaning.
SHORTHAND: dict[str, str] = {
    "degen": "degenerative",
    "degenrative": "degenerative",
    "degerative": "degenerative",
    "chnges": "changes",
    "chages": "changes",
    "norml": "normal",
    "nomal": "normal",
    "effusuon": "effusion",
    "effusiom": "effusion",
    "cacified": "calcified",
    "calcenal": "calcaneal",
    "lymphnodes": "lymph nodes",
    "lymphnode": "lymph node",
    "osteophyes": "osteophytes",
    "oosteophytes": "osteophytes",
    "trignonum": "trigonum",
    "rt": "right",
    "lt": "left",
    "bilat": "bilateral",
    "jt": "joint",
    "jts": "joints",
    "fx": "fracture",
    "wnl": "within normal limits",
    "w/o": "without",
    "w/": "with",
    "c/w": "consistent with",
    "s/p": "status post",
    "b/l": "bilateral",
    "h/o": "history of",
}

_SHORTHAND_RE = re.compile(
    r"(?<![A-Za-z0-9])(" + "|".join(sorted((re.escape(k) for k in SHORTHAND), key=len, reverse=True))
    + r")(?![A-Za-z0-9])",
    re.I,
)


def normalize_shorthand(text: str) -> str:
    """Expand dictation shorthand so the report reads as prose."""
    if not text:
        return text

    def repl(m: re.Match) -> str:
        src = m.group(1)
        out = SHORTHAND[src.lower()]
        return out.capitalize() if src[:1].isupper() else out

    return _SHORTHAND_RE.sub(repl, text)


# ------------------------------------------------------------- spell repair
_WORD_RE = re.compile(r"[A-Za-z][A-Za-z'\-]{2,}")
_PROTECT = frozenset(
    """mm cm ml cc iv ap pa lat oblique t1 t2 stir flair dwi adc grade type
    lung rads birads""".split()
)


def build_vocabulary(texts) -> dict[str, int]:
    """Vocabulary of words that actually occur in reports/templates."""
    from collections import Counter

    vocab: Counter = Counter()
    for t in texts:
        for w in _WORD_RE.findall((t or "").lower()):
            vocab[w] += 1
    return dict(vocab)


def correct_spelling(text: str, vocab: dict[str, int], min_count: int = 3) -> str:
    """Repair dictation typos against the corpus vocabulary.

    Conservative by construction: only words absent from the vocabulary are
    touched, the replacement must be a frequent corpus word, and the edit
    distance budget scales with word length.
    """
    if not text or not vocab:
        return text
    from rapidfuzz import process, fuzz

    choices = [w for w, c in vocab.items() if c >= min_count]
    if not choices:
        return text
    cache: dict[str, str] = {}

    def repl(m: re.Match) -> str:
        w = m.group(0)
        low = w.lower()
        if low in vocab or low in _PROTECT or len(low) < 5:
            return w
        if low in cache:
            out = cache[low]
        else:
            budget = 1 if len(low) < 8 else 2
            hit = process.extractOne(
                low, choices, scorer=fuzz.ratio, score_cutoff=100 * (1 - budget / len(low))
            )
            out = low
            if hit:
                cand = hit[0]
                if abs(len(cand) - len(low)) <= budget and cand[0] == low[0]:
                    out = cand
            cache[low] = out
        if out == low:
            return w
        return out.capitalize() if w[:1].isupper() else out

    return _WORD_RE.sub(repl, text)


# ------------------------------------------------------------- body regions
# Coarse regions used to condition the mined routing statistics: the same word
# means different fields in different studies ("effusion" -> PLEURA in a chest
# radiograph, JOINT in a knee MRI).
REGION_GROUPS: dict[str, tuple[str, ...]] = {
    "chest": ("chest", "thorax", "lung", "rib", "ribs", "sternum", "clavicle", "breast",
              "mediastinum"),
    "abdomen": ("abdomen", "pelvis", "liver", "kidney", "bladder", "gallbladder", "spleen",
                "pancreas", "bowel", "uterus", "ovary", "prostate", "scrotum", "renal"),
    "spine": ("spine", "lsspine", "vertebra", "sacrum", "coccyx", "lumbar", "cervical",
              "thoracic", "sacral", "spinal"),
    "upper_limb": ("shoulder", "elbow", "wrist", "hand", "humerus", "forearm", "finger",
                   "thumb", "scapula", "arm", "clavicular"),
    "lower_limb": ("hip", "knee", "ankle", "foot", "femur", "leg", "heel", "toe", "tibia",
                   "fibula", "calcaneous", "calcaneus", "patella"),
    "head_neck": ("head", "brain", "skull", "orbit", "sinus", "neck", "face", "sella",
                  "pituitary", "temporal", "mastoid", "paranasal", "thyroid"),
}

_REGION_INDEX = {w: r for r, words in REGION_GROUPS.items() for w in words}


def body_region(body_part: str, study_description: str = "") -> str:
    """Coarse anatomical region for conditioning the routing statistics."""
    text = re.sub(r"[^a-z ]", " ", f"{body_part} {study_description}".lower())
    for w in text.split():
        r = _REGION_INDEX.get(w)
        if r:
            return r
        r = _REGION_INDEX.get(w.rstrip("s"))
        if r:
            return r
    return "other"


### 2.3 Template parsing and report rendering

In [ ]:
%%writefile rrh/template.py
"""Parsing and rendering of the supplied normal template.

The template is the *starting report*: the renderer therefore reproduces the
template's field labels and field order exactly, only upper-casing labels and
inserting the blank-line separators used by the reference reports.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .textutil import cap_first, clean_ws, split_sentences, squash

HEAD_FINDINGS = re.compile(r"^[ \t]*FINDINGS[ \t]*:[ \t]*", re.I | re.M)
HEAD_IMPRESSION = re.compile(r"^[ \t]*IMPRESSION[ \t]*:[ \t]*", re.I | re.M)

# A label is a short, colon-terminated prefix at the start of a line.
LABEL_RE = re.compile(r"^[ \t]*([A-Za-z][A-Za-z0-9 ,'\-/&\.\(\)]{0,58}?)[ \t]*:[ \t]*(.*)$")

# Labels that the reference reports always leave empty.
ALWAYS_EMPTY = {"OTHER FINDINGS"}


def _is_label(line: str) -> tuple[str, str] | None:
    m = LABEL_RE.match(line)
    if not m:
        return None
    label, rest = m.group(1).strip(), m.group(2).strip()
    if not label or len(label.split()) > 7:
        return None
    # reject prose that merely happens to contain a colon
    if re.search(r"[.!?]\s", label):
        return None
    return label, rest


@dataclass
class Field:
    label_raw: str
    label: str
    text: str
    is_group: bool
    order: int
    is_free: bool = False
    sentences: list[str] = dc_field(default_factory=list)

    def __post_init__(self) -> None:
        if not self.sentences:
            self.sentences = split_sentences(self.text)


@dataclass
class Template:
    raw: str
    fields: list[Field]
    impression: list[str]
    findings_free: list[str]

    @property
    def labels(self) -> list[str]:
        return [f.label for f in self.fields]

    def by_label(self, label: str) -> Field | None:
        for f in self.fields:
            if f.label == label:
                return f
        return None


def parse_template(text: str) -> Template:
    text = clean_ws(text)
    mf = HEAD_FINDINGS.search(text)
    mi = HEAD_IMPRESSION.search(text)
    if mf and mi and mi.start() > mf.start():
        body, imp = text[mf.end() : mi.start()], text[mi.end() :]
    elif mf:
        body, imp = text[mf.end() :], ""
    elif mi:
        body, imp = text[: mi.start()], text[mi.end() :]
    else:
        body, imp = text, ""

    fields: list[Field] = []
    free: list[str] = []
    order = 0
    for line in body.split("\n"):
        line = line.strip()
        if not line:
            continue
        parsed = _is_label(line)
        if parsed:
            label_raw, rest = parsed
            fields.append(
                Field(
                    label_raw=label_raw,
                    label=label_raw.upper(),
                    text=rest,
                    is_group=(rest == ""),
                    order=order,
                )
            )
            order += 1
        else:
            # Prose templates (no labels) keep every line as its own block.
            free.append(line)
            fields.append(
                Field(
                    label_raw="",
                    label="",
                    text=line,
                    is_group=False,
                    order=order,
                    is_free=True,
                )
            )
            order += 1

    impression = [ln.strip() for ln in imp.split("\n") if ln.strip()]
    return Template(raw=text, fields=fields, impression=impression, findings_free=free)


# ------------------------------------------------------------------ render


def render_report(
    field_texts: list[tuple[str, str]],
    impression_lines: list[str],
    extra_paragraphs: list[str] | None = None,
    blank_between_fields: bool = True,
    merge_extras: bool = False,
) -> str:
    """Assemble the final report.

    `field_texts` is an ordered list of (UPPERCASE label, body text) pairs; the
    body may be empty for group headers and for OTHER FINDINGS.
    """
    lines: list[str] = ["FINDINGS:"]
    first = True
    for label, body in field_texts:
        if blank_between_fields and not first:
            lines.append("")
        first = False
        body = squash(body)
        if not label:
            if body:
                lines.append(body)
            continue
        lines.append(f"{label}: {body}".rstrip() if body else f"{label}:")
    paras = [squash(p) for p in (extra_paragraphs or []) if squash(p)]
    if merge_extras and paras:
        paras = [" ".join(paras)]
    for para in paras:
        lines.append("")
        lines.append(para)
    lines.append("")
    lines.append("IMPRESSION:")
    lines.extend(squash(x) for x in impression_lines if squash(x))
    out = "\n".join(lines).rstrip() + "\n"
    return out


def resolve_placeholders(text: str, laterality: str | None, region: str | None) -> str:
    """Fill `[left/right]` / `[generic]` style slots left in template prose."""
    if "[" not in text:
        return text

    def repl(m: re.Match) -> str:
        inner = m.group(1).strip()
        low = inner.lower()
        if any(w in low for w in ("left", "right", "bilateral", "laterality")):
            return laterality or ""
        if "generic" in low or "region" in low or "body" in low:
            return region or ""
        if "/" in inner:  # unresolved option list -> first option
            return inner.split("/")[0].strip()
        return ""

    text = re.sub(r"\[([^\]]*)\]", repl, text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"\bthe\s+(?=[.,])", "", text)
    return cap_first(text.strip())


### 2.4 Dictation segmentation - cues, boilerplate, dictated summary

In [ ]:
%%writefile rrh/dictation.py
"""Dictation segmentation.

Turns a telegraphic dictation into structured `Segment`s, separating
   * technique / history / recommendation boilerplate (never reported),
   * the body of observations, and
   * a trailing radiologist summary (the dictated impression), which the
     reference reports reuse almost verbatim in IMPRESSION.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .lexicon import correct_spelling, normalize_shorthand
from .textutil import clean_ws, content_tokens, jaccard, split_sentences, squash, token_set

CUE_VERB = re.compile(
    r"^(?P<cue>[A-Za-z][A-Za-z0-9 ,/&'\-\.]{1,45}?)\s+(?:shows?|demonstrates?|reveals?|"
    r"demonstrate|reveal)\s+(?P<rest>.+)$",
    re.I,
)
CUE_COLON = re.compile(r"^(?P<cue>[A-Za-z][A-Za-z0-9 ,/&'\-\.]{1,45}?)\s*:\s*(?P<rest>.+)$")
BARE_HEADER = re.compile(r"^(?P<cue>[A-Za-z][A-Za-z0-9 ,/&'\-\.]{1,45}?)\s*:\s*$")
LEVEL_RE = re.compile(r"\b([CTLS])\s*(\d{1,2})\s*[-–/]\s*(?:[CTLS])?\s*(\d{1,2}|S1)\b", re.I)

TECHNIQUE_PAT = re.compile(
    r"(was|were)\s+(performed|obtained|acquired)|"
    r"^(multiplanar|multisequence|multi-planar|axial|sagittal|coronal|sequences?|"
    r"technique|protocol|images?\s+were|imaging\s+was|scout|localizer|"
    r"post-?processed|reformat\w*|reconstruct\w*|magnetic\s+resonance|"
    r"computed\s+tomograph\w*|hrct\b|thin[- ]section)\b|"
    r"\b(radiograph|view|projection)s?\s+(of|were|was|obtained|acquired)\b|"
    r"^\s*(\d+|two|three|four|five|six|single|frontal|lateral|ap and lateral)[\w\- ]{0,25}"
    r"(views?|radiographs?|projections?)\b|"
    r"\b(without|with)\s+(intravenous|iv)\s+contrast\s*$|"
    r"^(contrast|comparison|indication|history|clinical\s+history|technique|exam(ination)?)\s*[:\-]",
    re.I,
)
HISTORY_PAT = re.compile(
    r"\b(pain|swelling|trauma|injury|fall|weakness|numbness|tingling|discomfort|"
    r"complaint|symptoms?)\s+(for|since|x)\s+\d|"
    r"^(clinical\s+)?(history|indication|hx)\b|"
    r"\b(rule\s+out|r/o)\b\s*$|"
    # "15-19-year-old with back pain (M54.9)." - age band, referral, ICD code
    r"\b\d{1,3}\s*(?:-\s*\d{1,3}\s*)?[- ]year[- ]old\b|"
    r"\(\s*[A-TV-Z]\d{2}(?:\.\d+)?\s*\)|"
    r"^(shortness of breath|sob|chest pain|back pain|abdominal pain|headache|fever)\b",
    re.I,
)
RECOMMEND_PAT = re.compile(
    r"^(recommendation|recommend(ed|s)?\b|correlate\s+clinic|"
    r"suggest\s+clinical|further\s+evaluation\s+with|"
    r"clinical\s+correlation\s+(is\s+)?(recommended|suggested|advised))",
    re.I,
)
ADVISED_PAT = re.compile(
    r"\b(is|are)\s+(advised|recommended|suggested|indicated)\b|"
    r"\bfor\s+further\s+(characteri[sz]ation|evaluation|assessment|workup)\b|"
    r"\bcorrelation\s+is\s+(advised|recommended|suggested)\b",
    re.I,
)
NONE_PAT = re.compile(
    r"^(none|n/?a|nil|not applicable)(\s+(available|provided|performed|obtained))?\s*\.?$|"
    r"^("
    r"(no\s+)?(prior|previous|comparison)s?(\s+(study|studies|exam\w*|imaging))?"
    r"(\s+(are|is|were|was))?(\s+(available|provided|performed))?)\s*\.?$",
    re.I,
)
MODALITY_HEADER = re.compile(
    r"^(mri|mr|ct|cta|mra|mrcp|us|usg|ultrasound|sonograph\w*|x-?ray|xr|radiograph\w*|"
    r"pet|dexa|fluoroscop\w*)\b",
    re.I,
)
CONTRAST_PAT = re.compile(
    r"\b(gadavist|gadolinium|gadobutrol|omnipaque|iohexol|ioversol|isovue|optiray|"
    r"ultravist|visipaque|contrast\s+material|contrast\s+agent)\b|"
    r"^\s*\w+\s+\d+(?:\.\d+)?\s*(?:ml|cc|mg)\s+(?:iv|i\.v\.)\b|"
    r"\b\d+(?:\.\d+)?\s*(?:ml|cc)\b[^.]{0,40}\b(administered|injected|intravenous(?:ly)?|"
    r"orally|per\s+oral)\b",
    re.I,
)
SYMPTOM_PAT = re.compile(
    r"\b(nausea|vomiting|headaches?|dizziness|vertigo|fever|chills|cough|dyspn(?:o?ea)|"
    r"seizures?|syncope|palpitations?|fatigue|malaise|tingling|paresthesias?|"
    r"restricted movement|difficulty|inability)\b",
    re.I,
)
ALLCAPS_HEADER = re.compile(r"^[A-Z0-9][A-Z0-9 /\-\(\)\.,&]{6,}$")
NORMAL_PAT = re.compile(
    r"^(normal|norml|nomal|normal study|unremarkable|nad|wnl|within normal limits?|"
    r"no abnormality|no acute abnormality|essentially normal|grossly normal)\s*\.?$",
    re.I,
)

GLOBAL_CONCL = re.compile(
    r"^(no acute|no significant|otherwise\s+(unremarkable|normal)|overall|"
    r"essentially\s+normal|unremarkable\s+(study|examination))\b.*"
    r"(abnormality|abnormalities|finding|study|examination|"
    r"\b(shoulder|knee|hip|wrist|ankle|elbow|hand|foot|spine|chest|abdomen|pelvis|"
    r"brain|head|neck|femur|humerus|skull|thorax)\b)",
    re.I,
)

NEG_LEAD = re.compile(
    r"^(no\b|not\b|without\b|absence of\b|negative for\b|there is no\b|there are no\b|"
    r"free of\b|unremarkable\b|normal\b|intact\b|preserved\b|maintained\b)",
    re.I,
)
NEG_ANY = re.compile(r"\b(no|not|without|negative for|absence of|free of)\b", re.I)
POS_HINT = re.compile(
    r"\b(mild|moderate|severe|small|large|minimal|marked|mild-to-moderate|"
    r"moderate-to-severe|trace|prominent|focal|diffuse|multiple|few|grade\s*[iv1-4]|"
    r"partial|complete|acute|chronic|degenerative|tear|fracture|effusion|edema|oedema|"
    r"stenosis|narrowing|osteophyte|bulge|herniation|opacity|consolidation|nodule|mass|"
    r"lesion|cyst|swelling|hypertrophy|spondylosis|tendinosis|bursitis|arthrosis|"
    r"arthritis|atrophy|thickening|calcification|sclerosis|deformity|dislocation|"
    r"collection|abnormal)\b",
    re.I,
)
LATERAL_RE = re.compile(r"\b(right|left|bilateral|rt|lt|b/l|bilat)\b", re.I)
MEASURE_RE = re.compile(r"\b\d+(?:\.\d+)?\s*(?:x\s*\d+(?:\.\d+)?\s*)*(?:mm|cm|ml|cc)\b", re.I)


@dataclass
class Segment:
    text: str
    cue: str | None = None
    own_cue: bool = False  # the cue was written in this sentence, not inherited
    kind: str = "finding"  # finding | preamble | impression | normal
    negative: bool = False
    tokens: frozenset = dc_field(default_factory=frozenset)

    def __post_init__(self) -> None:
        if not self.tokens:
            self.tokens = token_set(self.text)
        self.negative = bool(NEG_LEAD.match(self.text)) and not POS_HINT.search(self.text)

    @property
    def laterality(self) -> str | None:
        m = LATERAL_RE.search(self.text)
        if not m:
            return None
        v = m.group(1).lower()
        return {"rt": "right", "lt": "left", "b/l": "bilateral", "bilat": "bilateral"}.get(v, v)


@dataclass
class DictationDoc:
    raw: str
    preamble: list[Segment]
    findings: list[Segment]
    impression: list[Segment]
    is_normal: bool

    @property
    def all_reportable(self) -> list[Segment]:
        return self.findings + self.impression


def _classify_boilerplate(sent: str) -> str | None:
    s = sent.strip()
    if NONE_PAT.match(s):
        return "preamble"
    if RECOMMEND_PAT.match(s) or ADVISED_PAT.search(s):
        return "preamble"
    if HISTORY_PAT.search(s):
        return "preamble"
    if TECHNIQUE_PAT.search(s):
        return "preamble"
    if ALLCAPS_HEADER.match(s) and not LEVEL_RE.search(s):
        return "preamble"
    if MODALITY_HEADER.match(s) and not POS_HINT.search(s) and not NEG_ANY.search(s):
        return "preamble"
    if CONTRAST_PAT.search(s) and len(content_tokens(s)) <= 8:
        return "preamble"
    if SYMPTOM_PAT.search(s) and len(content_tokens(s)) <= 6:
        return "preamble"
    return None


ARTICLE_LEAD = re.compile(r"^(the|a|an|there|this|these|those|it|his|her|their)\b", re.I)


def _strip_cue(sent: str) -> tuple[str | None, str, bool]:
    """Return (routing cue, sentence body).

    A *header* cue ("Bones shows ...", "Labrum: ...") is removed from the text,
    exactly as the reference reports do.  A cue that is really the subject of a
    normal sentence ("The medial meniscus demonstrates ...") is kept verbatim
    and only used for routing.
    """
    m = CUE_VERB.match(sent)
    if m:
        cue, rest = m.group("cue").strip(), m.group("rest").strip()
        if len(content_tokens(cue)) <= 6 and rest:
            if ARTICLE_LEAD.match(cue) or rest[:1].isupper():
                return cue, sent, False
            return cue, rest, True
    m = CUE_COLON.match(sent)
    if m:
        cue, rest = m.group("cue").strip(), m.group("rest").strip()
        if len(cue.split()) <= 6 and not re.search(r"[.!?]", cue) and rest:
            return cue, rest, True
    return None, sent, False


def _match_score(a: Segment, prior: list[Segment]) -> float:
    best = 0.0
    for p in prior:
        best = max(best, jaccard(a.tokens, p.tokens))
        if best >= 0.9:
            break
    return best


def _detect_summary(segs: list[Segment], threshold: float = 0.34, max_misses: int = 3,
                    after_cues: bool = False) -> int:
    """Index where the dictated impression starts (len(segs) if there is none).

    Two independent signals: a global conclusion sentence in the back part of
    the dictation, and a trailing run of sentences that restate earlier ones.
    """
    n = len(segs)
    if n < 4:
        return n
    body_limit = max(2, n // 3)
    # Section cues ("C5-6 shows ...", "Bones show ...") mark the body of the
    # dictation: a summary can only start after the last one.  Without this,
    # level-by-level spine dictations look like a repeated-sentence summary.
    last_cue = max((i for i, s in enumerate(segs) if s.own_cue), default=-1)
    body_limit = max(body_limit, last_cue + 1)
    if body_limit >= n - 1:
        return n
    if after_cues and last_cue >= 0 and n - (last_cue + 1) >= 2:
        # a structured dictation ends its cued sections and then summarises
        return last_cue + 1

    marker = n
    for i in range(body_limit, n - 1):
        if GLOBAL_CONCL.match(segs[i].text):
            marker = i
            break

    dup = n
    misses = 0
    for i in range(n - 1, body_limit - 1, -1):
        if _match_score(segs[i], segs[:i]) >= threshold:
            dup = i
            misses = 0
        else:
            misses += 1
            if misses > max_misses:
                break
    if dup >= n - 1:
        dup = n

    start = min(marker, dup)
    if start >= n - 1:
        return n
    while start > body_limit and GLOBAL_CONCL.match(segs[start - 1].text):
        start -= 1
    return start


def segment_dictation(
    raw: str,
    normalize: bool = True,
    vocab: dict[str, int] | None = None,
    summary_threshold: float = 0.34,
    summary_max_misses: int = 3,
    summary_after_cues: bool = False,
) -> DictationDoc:
    text = clean_ws(raw or "")
    if normalize:
        text = normalize_shorthand(text)
    if vocab:
        text = correct_spelling(text, vocab)
    if not text or NORMAL_PAT.match(squash(text)):
        return DictationDoc(raw=text, preamble=[], findings=[], impression=[], is_normal=True)

    sents = split_sentences(text)
    pre: list[Segment] = []
    body: list[Segment] = []
    carry_cue: str | None = None
    for s in sents:
        s = squash(s)
        if not s:
            continue
        kind = _classify_boilerplate(s)
        if kind == "preamble":
            pre.append(Segment(text=s, kind="preamble"))
            continue
        if NORMAL_PAT.match(s):
            continue
        bare = BARE_HEADER.match(s)
        if bare:  # a section header on its own line: routing cue, not a finding
            carry_cue = bare.group("cue").strip()
            continue
        cue, rest, is_header = _strip_cue(s)
        _own = cue
        if is_header:
            carry_cue = cue
        elif carry_cue and not cue:
            cue = carry_cue
        body.append(Segment(text=squash(rest), cue=cue, own_cue=bool(cue) and cue == _own))
    # a cue must not leak into the dictated summary


    body = [b for b in body if content_tokens(b.text)]
    cut = _detect_summary(body, summary_threshold, summary_max_misses, summary_after_cues)
    findings, impression = body[:cut], body[cut:]
    for seg in impression:
        seg.kind = "impression"
    is_normal = not body
    return DictationDoc(
        raw=text, preamble=pre, findings=findings, impression=impression, is_normal=is_normal
    )


### 2.5 Coordinated-clause splitting

In [ ]:
%%writefile rrh/splitting.py
"""Decompose coordinated dictation sentences so each clause can be routed.

The reference reports routinely split a dictated sentence across fields, e.g.

    "No acute fracture or dislocation is identified."
        -> BONES:  No acute fracture is identified.
        -> JOINTS: No dislocation is identified.

A decomposition is only adopted (in `routing`) when its parts genuinely land in
different template fields; otherwise the sentence is kept verbatim.
"""
from __future__ import annotations

import re

from .textutil import content_tokens, squash

_NEG_LEAD = re.compile(
    r"^(?P<lead>no evidence of|there is no|there are no|no significant|no acute|no)\s+"
    r"(?P<body>.+?)(?P<tail>\s+(?:is|are|was|were)\s+"
    r"(?:identified|seen|noted|present|evident|visualized|visualised|appreciated|"
    r"demonstrated|observed))?\s*[.]?$",
    re.I,
)
_SUBJ_LIST = re.compile(
    r"^(?P<det>the\s+)?(?P<subj>.+?)\s+(?P<verb>are|were)\s+(?P<pred>.+?)\s*[.]?$", re.I
)
_LIST_SPLIT = re.compile(r"\s*,\s*(?:and\s+|or\s+)?|\s+and\s+|\s+or\s+", re.I)
_COMMA_SPLIT = re.compile(r"\s*[,;]\s*")

_MAX_PART_TOKENS = 8


def _ok_parts(parts: list[str], minimum: int = 2) -> bool:
    if len(parts) < minimum:
        return False
    for p in parts:
        n = len(content_tokens(p))
        if n < 1 or n > _MAX_PART_TOKENS:
            return False
    return True


def split_negation(text: str) -> list[str] | None:
    m = _NEG_LEAD.match(squash(text))
    if not m:
        return None
    lead = m.group("lead")
    body = m.group("body") or ""
    tail = m.group("tail") or ""
    if re.search(r"\b(with|without|causing|producing|resulting|extending)\b", body, re.I):
        return None
    parts = [p.strip() for p in _LIST_SPLIT.split(body) if p.strip()]
    if not _ok_parts(parts):
        return None
    tail = re.sub(r"\bare\b", "is", tail, flags=re.I)
    tail = re.sub(r"\bwere\b", "was", tail, flags=re.I)
    low = lead.lower()
    if low == "no evidence of":
        return [f"No evidence of {p}{tail}." for p in parts]
    if low in {"no acute", "no significant"}:
        # the qualifier belongs to the first item only:
        # "No acute fracture or dislocation" -> "No acute fracture." / "No dislocation."
        head = f"{lead.capitalize().replace('No ', 'No ')} {parts[0]}{tail}."
        return [head] + [f"No {p}{tail}." for p in parts[1:]]
    return [f"No {p}{tail}." for p in parts]


def split_subject_list(text: str) -> list[str] | None:
    t = squash(text)
    m = _SUBJ_LIST.match(t)
    if not m:
        return None
    subj, pred = m.group("subj"), m.group("pred")
    det = "The " if (m.group("det") or "").strip() else ""
    if re.search(r"\b(no|not|without)\b", subj, re.I):
        return None
    parts = [p.strip() for p in _LIST_SPLIT.split(subj) if p.strip()]
    if not _ok_parts(parts):
        return None
    verb = "is" if m.group("verb").lower() == "are" else "was"
    return [f"{det}{p} {verb} {pred}." for p in parts]


def split_clauses(text: str) -> list[str] | None:
    t = squash(text).rstrip(".")
    parts = [p.strip(" .") for p in _COMMA_SPLIT.split(t) if p.strip(" .")]
    if not _ok_parts(parts):
        return None
    if any(re.match(r"^(and|or|with|without|which|more|greatest|most)\b", p, re.I) for p in parts):
        return None
    return [p + "." for p in parts]


_AND_SPLIT = re.compile(r"\s+and\s+|\s*;\s*", re.I)


def split_conjunction(text: str) -> list[str] | None:
    """Telegraphic coordination: "degenerative changes and small effusion"."""
    t = squash(text).rstrip(".")
    if re.search(r"\b(no|not|without|with|between|causing|producing)\b", t, re.I):
        return None
    parts = [p.strip(" .") for p in _AND_SPLIT.split(t) if p.strip(" .")]
    if not _ok_parts(parts):
        return None
    return [p + "." for p in parts]


def candidate_splits(text: str, conjunction: bool = False) -> list[list[str]]:
    """Alternative decompositions of `text`, best first."""
    out: list[list[str]] = []
    fns = [split_negation, split_subject_list, split_clauses]
    if conjunction:
        fns.append(split_conjunction)
    for fn in fns:
        parts = fn(text)
        if parts and len(parts) >= 2:
            out.append([squash(p) for p in parts])
    # de-duplicate while preserving order
    seen: set[tuple[str, ...]] = set()
    uniq: list[list[str]] = []
    for p in out:
        k = tuple(p)
        if k not in seen:
            seen.add(k)
            uniq.append(p)
    return uniq


### 2.6 Field routing (mined from the training reports)

In [ ]:
%%writefile rrh/routing.py
"""Field routing: decide which template field each dictated finding belongs to.

Signals, in decreasing order of trust:
  1. an explicit dictation cue ("Bones shows ...", "L4-L5: ...")
  2. the field label itself appearing in the finding
  3. curated anatomy->field concepts (`lexicon.py`)
  4. overlap with the field's own normal statement in the template
  5. statistics mined from the training reports

The mined statistics are fitted only on training rows, so the evaluation can
hold folds out honestly.
"""
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field as dc_field

from .lexicon import body_region, concept_for_label, concept_hits, stem
from .template import Template, parse_template
from .textutil import content_tokens, key, sim, split_sentences, squash

MINE_THRESHOLD = 0.5

LEVEL_RE = re.compile(r"^([ctls])\s*(\d{1,2})\s*[-/]\s*(?:([ctls])\s*)?(\d{1,2}|s1)$", re.I)

# Reporting verbs and connectives a reference sentence may add without adding
# any clinical content.
NEUTRAL_STEMS = {
    stem(w)
    for w in (
        "present", "noted", "seen", "identified", "evident", "demonstrated", "observed",
        "visualized", "visualised", "appreciated", "there", "remaining", "otherwise",
        "again", "also", "overall", "appears", "appear", "the", "are", "is", "which",
    )
}


def normalize_level(label: str) -> str | None:
    lab = key(label).replace(" ", "")
    lab = re.sub(r"^([ctls])(\d{1,2})([ctls])?(\d{1,2})$", r"\1\2-\4", lab)
    m = LEVEL_RE.match(lab.replace(" ", ""))
    if not m:
        m = LEVEL_RE.match(re.sub(r"([a-z])(\d+)-([a-z]?)(\d+)", r"\1\2-\4", lab))
    if not m:
        return None
    a, n1, _, n2 = m.groups()
    return f"{a.lower()}{int(n1)}-{n2.lower() if not n2.isdigit() else int(n2)}"


def label_tokens(label: str) -> set[str]:
    return {stem(t) for t in content_tokens(label)}


def _label_match(text: str, label: str, weights: dict[str, float] | None = None) -> float:
    """How strongly does `text` name `label`?"""
    if not label:
        return 0.0
    lv, tv = normalize_level(label), normalize_level(text)
    if lv and tv:
        return 1.0 if lv == tv else 0.0
    if lv and not tv:
        toks = re.findall(r"\b[ctls]\s*\d{1,2}\s*[-/]\s*[ctls]?\s*(?:\d{1,2}|s1)\b", key(text))
        if any(normalize_level(t) == lv for t in toks):
            return 1.0
        return 0.0
    lt = label_tokens(label)
    if not lt:
        return 0.0
    tt = {stem(t) for t in content_tokens(text)}
    if not tt:
        return 0.0
    if weights:
        num = sum(weights.get(t, 1.0) for t in sorted(lt & tt))
        den = sum(weights.get(t, 1.0) for t in sorted(lt)) or 1.0
        return min(1.0, num / den)
    return len(lt & tt) / len(lt)


@dataclass
class RegionStats:
    """Mined (token -> field) statistics for one coarse body region."""

    token_label: dict[str, Counter] = dc_field(default_factory=lambda: defaultdict(Counter))
    label_total: Counter = dc_field(default_factory=Counter)
    token_total: Counter = dc_field(default_factory=Counter)
    n_pairs: int = 0
    postings: dict[str, list[int]] = dc_field(default_factory=lambda: defaultdict(list))
    examples: list[tuple[frozenset, str]] = dc_field(default_factory=list)

    def add(self, toks: set[str], label: str) -> None:
        self.n_pairs += 1
        self.label_total[label] += 1
        idx = len(self.examples)
        self.examples.append((frozenset(toks), label))
        for t in sorted(toks):
            self.token_label[t][label] += 1
            self.token_total[t] += 1
            self.postings[t].append(idx)

    def score(self, tokens: set[str], label: str) -> float:
        if label not in self.label_total or not tokens:
            return 0.0
        total = self.label_total[label]
        acc = 0.0
        for t in sorted(tokens):
            c = self.token_label.get(t)
            if not c:
                continue
            p_t_l = c.get(label, 0) / total
            p_t = self.token_total[t] / max(1, self.n_pairs)
            if p_t_l > 0:
                acc += math.log((p_t_l + 1e-4) / (p_t + 1e-4))
        return max(0.0, min(1.0, acc / (2.0 * max(1, len(tokens)))))

    def knn(self, tokens: set[str], allowed: set[str], k: int = 12) -> dict[str, float]:
        if not tokens or not self.examples:
            return {}
        cand: Counter = Counter()
        for t in sorted(tokens):
            for i in self.postings.get(t, ()):  # type: ignore[arg-type]
                cand[i] += 1
        if not cand:
            return {}
        scored = []
        for i, _ in sorted(cand.items(), key=lambda kv: (-kv[1], kv[0]))[:120]:
            etoks, label = self.examples[i]
            if label not in allowed:
                continue
            inter = len(tokens & etoks)
            if not inter:
                continue
            scored.append((inter / len(tokens | etoks), label))
        scored.sort(reverse=True)
        votes: dict[str, float] = {}
        tot = 0.0
        for j, label in sorted(scored[:k]):
            votes[label] = votes.get(label, 0.0) + j
            tot += j
        return {lab: v / tot for lab, v in votes.items()} if tot > 0 else {}


@dataclass
class RoutingModel:
    token_label: dict[str, Counter] = dc_field(default_factory=lambda: defaultdict(Counter))
    label_total: Counter = dc_field(default_factory=Counter)
    token_total: Counter = dc_field(default_factory=Counter)
    n_pairs: int = 0

    postings: dict[str, list[int]] = dc_field(default_factory=lambda: defaultdict(list))
    examples: list[tuple[frozenset, str]] = dc_field(default_factory=list)
    vocab: dict[str, int] = dc_field(default_factory=dict)
    ranker: object | None = None
    chooser: object | None = None
    regions: dict[str, RegionStats] = dc_field(default_factory=dict)
    use_bigrams: bool = False
    mine_threshold: float = MINE_THRESHOLD
    region_weight: float = 0.0
    field_edits: dict[tuple[str, str], tuple[int, int]] = dc_field(default_factory=dict)
    label_edits: dict[str, tuple[int, int]] = dc_field(default_factory=dict)
    global_edit_rate: float = 0.35

    def edit_prior(self, tkey: str, label: str, alpha: float = 2.0) -> float:
        """How often the reference edits this field of this template."""
        lab_e, lab_n = self.label_edits.get(label, (0, 0))
        backoff = (lab_e + alpha * self.global_edit_rate) / (lab_n + alpha)
        e, n = self.field_edits.get((tkey, label), (0, 0))
        return (e + alpha * backoff) / (n + alpha)
    style_examples: list[tuple[frozenset, str, str]] = dc_field(default_factory=list)
    style_postings: dict[str, list[int]] = dc_field(default_factory=lambda: defaultdict(list))

    def style_match(self, clause: str, threshold: float) -> str | None:
        """Closest training clause->reported-sentence rewrite, if it is safe.

        "Safe" means the reference sentence introduces no content word that the
        clause does not already have (apart from neutral reporting verbs), so
        this transfers phrasing only - never a finding.
        """
        toks = {stem(t) for t in content_tokens(clause)}
        if not toks or not self.style_examples:
            return None
        cand: Counter = Counter()
        for t in sorted(toks):
            for i in self.style_postings.get(t, ()):  # type: ignore[arg-type]
                cand[i] += 1
        best, best_s = None, 0.0
        for i, _ in sorted(cand.items(), key=lambda kv: (-kv[1], kv[0]))[:60]:
            etoks, src, tgt = self.style_examples[i]
            j = len(toks & etoks) / len(toks | etoks)
            if j < threshold:
                continue
            score = j + 0.001 * len(etoks)
            if score > best_s:
                best, best_s = (src, tgt), score
        if not best:
            return None
        _, target = best
        extra = {stem(t) for t in content_tokens(target)} - toks - NEUTRAL_STEMS
        if extra:
            return None
        return target

    def knn(self, tokens: set[str], allowed: set[str], k: int = 12) -> dict[str, float]:
        """Similarity-weighted vote of the closest mined training sentences."""
        if not tokens or not self.examples:
            return {}
        cand: Counter = Counter()
        for t in sorted(tokens):
            for i in self.postings.get(t, ()):  # type: ignore[arg-type]
                cand[i] += 1
        if not cand:
            return {}
        scored = []
        for i, _ in sorted(cand.items(), key=lambda kv: (-kv[1], kv[0]))[:120]:
            etoks, label = self.examples[i]
            if label not in allowed:
                continue
            inter = len(tokens & etoks)
            if not inter:
                continue
            j = inter / len(tokens | etoks)
            scored.append((j, label))
        scored.sort(reverse=True)
        votes: dict[str, float] = {}
        tot = 0.0
        for j, label in sorted(scored[:k]):
            votes[label] = votes.get(label, 0.0) + j
            tot += j
        if tot <= 0:
            return {}
        return {lab: v / tot for lab, v in votes.items()}

    def score(self, tokens: set[str], label: str) -> float:
        if label not in self.label_total or not tokens:
            return 0.0
        total = self.label_total[label]
        acc = 0.0
        for t in sorted(tokens):
            c = self.token_label.get(t)
            if not c:
                continue
            p_t_l = c.get(label, 0) / total
            p_t = self.token_total[t] / max(1, self.n_pairs)
            if p_t_l > 0:
                acc += math.log((p_t_l + 1e-4) / (p_t + 1e-4))
        return max(0.0, min(1.0, acc / (2.0 * max(1, len(tokens)))))


def mine_row(row, threshold: float = MINE_THRESHOLD) -> list[tuple[int, str, str, str]]:
    """(segment index, segment text, gold field label, reference sentence)."""
    from .dictation import segment_dictation

    tmpl = parse_template(row["template_content"])
    rep = parse_template(row["report"])
    doc = segment_dictation(row["dictation"])
    units = doc.findings or doc.impression
    if not units:
        return []
    tmpl_by_label = {f.label: f for f in tmpl.fields}
    out: list[tuple[int, str, str]] = []
    used: set[int] = set()
    for rf in rep.fields:
        if not rf.label:
            continue
        tf = tmpl_by_label.get(rf.label)
        tmpl_sents = tf.sentences if tf else []
        for rs in split_sentences(rf.text):
            if any(sim(rs, ts) >= 0.85 for ts in tmpl_sents):
                continue
            best, best_s = -1, 0.0
            for i, u in enumerate(units):
                s = sim(rs, u.text)
                if s > best_s:
                    best, best_s = i, s
            if best >= 0 and best_s >= threshold and best not in used:
                used.add(best)
                out.append((best, units[best].text, rf.label, rs))
    return out


def mine_pairs(rows) -> list[tuple[str, str]]:
    """(dictated sentence, report field label) supervision mined from train rows."""
    return [(text, label) for row in rows for _, text, label, _ in mine_row(row)]


def token_repr(text: str, bigrams: bool = False) -> set[str]:
    """Stemmed unigrams, optionally with adjacent-pair features.

    Bigrams disambiguate terms whose field depends on their neighbour
    ("joint effusion" vs "pleural effusion").
    """
    toks = [stem(t) for t in content_tokens(text)]
    out = set(toks)
    if bigrams:
        out |= {f"{a}_{b}" for a, b in zip(toks, toks[1:])}
    return out


def fit_router(rows, cfg=None) -> RoutingModel:
    from .lexicon import build_vocabulary

    model = RoutingModel()
    if cfg is not None:
        model.use_bigrams = getattr(cfg, "use_bigrams", False)
        model.mine_threshold = getattr(cfg, "mine_threshold", MINE_THRESHOLD)
    model.vocab = build_vocabulary(
        [r.get("report") or "" for r in rows] + [r.get("template_content") or "" for r in rows]
    )
    edited: dict[tuple[str, str], list[int]] = {}
    lab_edit: dict[str, list[int]] = {}
    tot_e = tot_n = 0
    for row in rows:
        tkey = template_key(row.get("template_content") or "")
        tmpl_fields = {f.label: squash(f.text) for f in parse_template(row["template_content"]).fields if f.label}
        rep_fields = {f.label: squash(f.text) for f in parse_template(row["report"]).fields if f.label}
        for lab, txt in tmpl_fields.items():
            changed = int(rep_fields.get(lab, "") != txt)
            edited.setdefault((tkey, lab), [0, 0])
            edited[(tkey, lab)][0] += changed
            edited[(tkey, lab)][1] += 1
            lab_edit.setdefault(lab, [0, 0])
            lab_edit[lab][0] += changed
            lab_edit[lab][1] += 1
            tot_e += changed
            tot_n += 1
    model.field_edits = {k: (v[0], v[1]) for k, v in edited.items()}
    model.label_edits = {k: (v[0], v[1]) for k, v in lab_edit.items()}
    model.global_edit_rate = tot_e / max(1, tot_n)

    pairs: list[tuple[str, str]] = []
    for row in rows:
        region = body_region(
            str(row.get("body_part") or ""), str(row.get("study_description") or "")
        )
        for _, text, label, ref_sentence in mine_row(row, model.mine_threshold):
            pairs.append((text, label))
            rtoks = token_repr(text, model.use_bigrams)
            if rtoks:
                model.regions.setdefault(region, RegionStats()).add(rtoks, label)
            toks = frozenset(token_repr(text, model.use_bigrams))
            if not toks:
                continue
            idx = len(model.style_examples)
            model.style_examples.append((toks, text, ref_sentence))
            for t in toks:
                model.style_postings[t].append(idx)
    for text, label in pairs:
        toks = token_repr(text, model.use_bigrams)
        if not toks:
            continue
        model.n_pairs += 1
        model.label_total[label] += 1
        idx = len(model.examples)
        model.examples.append((frozenset(toks), label))
        for t in toks:
            model.token_label[t][label] += 1
            model.token_total[t] += 1
            model.postings[t].append(idx)
    return model


WEIGHTS = {
    "cue": 3.0,
    "label": 2.2,
    "concept": 1.2,
    "template": 1.0,
    "mined": 1.2,
    "knn": 1.6,
    "continuity": 0.0,
    "backward": 0.0,
    "prior": 0.0,
}
MIN_SCORE = 0.30


def field_candidates(tmpl: Template):
    return [f for f in tmpl.fields if f.label and not f.is_free and f.label != "OTHER FINDINGS"]


def cue_supported(cue: str | None, tmpl: Template, ctx: "TemplateContext",
                  minimum: float = 0.34) -> bool:
    """Does any field of this template correspond to the dictation's cue?

    When the radiologist dictates "Brain shows ..." but the template has no
    brain field, the finding belongs to no field at all - the reference reports
    put it in a trailing paragraph rather than forcing it into a wrong field.
    """
    if not cue:
        return True
    hits = concept_hits(cue)
    for f in field_candidates(tmpl):
        if _label_match(cue, f.label, ctx.label_weights) >= minimum:
            return True
        concept = concept_for_label(f.label)
        if concept and concept in hits:
            return True
    return False


FEATURES = (
    "cue", "label", "concept", "template", "mined", "knn",
    "group", "same_prev", "backward", "forward", "prior",
)


def template_key(text: str) -> str:
    import hashlib

    return hashlib.sha256(key(text).encode()).hexdigest()[:16]


def field_features(seg_text: str, cue: str | None, tmpl: Template, model: RoutingModel,
                   ctx: "TemplateContext",
                   prev_order: int | None = None) -> list[tuple[str, dict[str, float]]]:
    """Per-candidate-field feature vector for one dictated clause.

    The same features drive the hand-weighted scorer and the learned ranker, so
    the two are directly comparable.
    """
    idf, lw = ctx.idf, ctx.label_weights
    toks = token_repr(seg_text, model.use_bigrams)
    hits = concept_hits(seg_text)
    tot_hits = sum(hits.values())
    cands = field_candidates(tmpl)
    allowed = {f.label for f in cands}
    knn = model.knn(toks, allowed)
    reg = model.regions.get(ctx.region) if model.region_weight else None
    if reg is not None and reg.n_pairs >= 50:
        lam = model.region_weight
        reg_knn = reg.knn(toks, allowed)
        knn = {
            lab: (1 - lam) * knn.get(lab, 0.0) + lam * reg_knn.get(lab, 0.0)
            for lab in allowed
        }
    else:
        reg = None
    span = max(1, len(tmpl.fields) - 1)
    out: list[tuple[str, dict[str, float]]] = []
    for f in cands:
        feat = {k: 0.0 for k in FEATURES}
        if cue:
            feat["cue"] = _label_match(cue, f.label, lw)
        feat["label"] = _label_match(seg_text, f.label, lw)
        concept = concept_for_label(f.label)
        if concept and tot_hits:
            feat["concept"] = hits.get(concept, 0) / tot_hits
        if f.text:
            ft = {stem(t) for t in content_tokens(f.text)}
            if ft and toks:
                num = sum(idf.get(t, 1.0) for t in sorted(ft & toks))
                den = sum(idf.get(t, 1.0) for t in sorted(ft)) or 1.0
                feat["template"] = min(1.0, num / den)
        feat["mined"] = model.score(toks, f.label)
        if reg is not None:
            lam = model.region_weight
            feat["mined"] = (1 - lam) * feat["mined"] + lam * reg.score(toks, f.label)
        feat["knn"] = knn.get(f.label, 0.0)
        feat["group"] = 1.0 if f.is_group else 0.0
        feat["prior"] = model.edit_prior(ctx.template_key, f.label)
        if prev_order is not None:
            if f.order == prev_order:
                feat["same_prev"] = 1.0
            elif f.order < prev_order:
                feat["backward"] = min(1.0, (prev_order - f.order) / span)
            else:
                feat["forward"] = min(1.0, (f.order - prev_order) / span)
        out.append((f.label, feat))
    return out


def score_segment(seg_text: str, cue: str | None, tmpl: Template, model: RoutingModel,
                  ctx: "TemplateContext",
                  weights: dict[str, float] | None = None,
                  prev_order: int | None = None) -> list[tuple[float, str]]:
    """Score every candidate field for one dictated clause.

    `prev_order` is the template position of the field the previous clause went
    to.  Radiologists dictate in template order - 52% of consecutive findings
    stay in the same field and 85% never move backwards - so continuity is a
    real signal, not a heuristic.
    """
    W = weights or WEIGHTS
    scored: list[tuple[float, str]] = []
    for label, feat in field_features(seg_text, cue, tmpl, model, ctx, prev_order):
        if model.ranker is not None and W.get("use_ranker"):
            s = model.ranker.score(feat)
        else:
            s = (
                W["cue"] * feat["cue"]
                + W["label"] * feat["label"]
                + W["concept"] * feat["concept"]
                + W["template"] * feat["template"]
                + W["mined"] * feat["mined"]
                + W["knn"] * feat["knn"]
                + W.get("prior", 0.0) * feat["prior"]
                + W.get("continuity", 0.0) * feat["same_prev"]
                - W.get("backward", 0.0) * feat["backward"]
            )
        scored.append((s, label))
    scored.sort(key=lambda x: (-x[0], x[1]))
    return scored


@dataclass
class TemplateContext:
    idf: dict[str, float]
    label_weights: dict[str, float]
    template_key: str = ""
    region: str = "other"


def build_context(tmpl: Template, region: str = "other") -> TemplateContext:
    """Rarity weights computed over the template's own fields and labels:
    a word that occurs in only one field/label is decisive for that field."""
    fields = field_candidates(tmpl)
    n = max(1, len(fields))
    df: Counter = Counter()
    for f in fields:
        for t in {stem(x) for x in content_tokens(f.text)}:
            df[t] += 1
    idf = {t: math.log((n + 1) / (c + 0.5)) for t, c in df.items()}

    ldf: Counter = Counter()
    for f in fields:
        for t in label_tokens(f.label):
            ldf[t] += 1
    lw = {t: math.log((n + 1) / (c + 0.5)) for t, c in ldf.items()}
    # generic label words carry little routing information
    for generic in ("structure", "space", "tissu", "other", "finding", "region", "gener"):
        for t in list(lw):
            if t.startswith(generic):
                lw[t] = min(lw[t], 0.3)
    return TemplateContext(
        idf=idf, label_weights=lw, template_key=template_key(tmpl.raw), region=region
    )


# backwards-compatible helper
def build_idf(tmpl: Template) -> TemplateContext:
    return build_context(tmpl)


def fit_ranked_router(rows, cfg) -> RoutingModel:
    """Fit the mined statistics, then fit the learned router on top of them."""
    from .ranker import build_examples, fit_ranker

    model = fit_router(rows, cfg)
    model.region_weight = getattr(cfg, "region_weight", 0.0)
    if getattr(cfg, "use_impression_chooser", False):
        from .chooser import build_training_rows, fit_chooser

        model.chooser = fit_chooser(
            build_training_rows(rows, model, cfg), depth=cfg.chooser_depth
        )
    if getattr(cfg, "use_ranker", False):
        model.ranker = fit_ranker(
            build_examples(rows, model, cfg),
            epochs=cfg.ranker_epochs,
            lr=cfg.ranker_lr,
            l2=cfg.ranker_l2,
        )
    return model


### 2.6a Learned re-ranker (fitted but disabled by the tuned config)

In [ ]:
%%writefile rrh/ranker.py
"""A learned field router.

The hand-weighted scorer in `routing.py` combines six signals with weights set
by hand.  This module fits those weights instead: a conditional-logit (softmax
over the candidate fields of a template) trained on the `(clause -> field)`
pairs mined from the training reports, optimised with plain gradient ascent so
it stays dependency-free and deterministic.
"""
from __future__ import annotations

from dataclasses import dataclass, field as dc_field

from .routing import FEATURES

_INIT_SEED = {
    "cue": 3.0, "label": 2.2, "concept": 1.2, "template": 1.0, "mined": 1.2,
    "knn": 1.6, "group": -1.0, "same_prev": 0.3, "backward": -1.0, "forward": 0.0,
}
# every feature must have a weight, including ones added after this seed
INIT = {k: _INIT_SEED.get(k, 0.0) for k in FEATURES}


@dataclass
class Ranker:
    weights: dict[str, float] = dc_field(default_factory=lambda: dict(INIT))
    bias: float = 0.0

    def score(self, feat: dict[str, float]) -> float:
        w = self.weights
        return sum(w[k] * feat.get(k, 0.0) for k in FEATURES)


def _softmax(scores: list[float]) -> list[float]:
    m = max(scores)
    exps = [pow(2.718281828459045, s - m) for s in scores]
    tot = sum(exps) or 1.0
    return [e / tot for e in exps]


def fit_ranker(examples, epochs: int = 300, lr: float = 0.25, l2: float = 1e-3) -> Ranker:
    """`examples` is a list of (list[(label, features)], gold_label)."""
    w = dict(INIT)
    data = [
        (cands, gold)
        for cands, gold in examples
        if len(cands) > 1 and any(lab == gold for lab, _ in cands)
    ]
    if not data:
        return Ranker(weights=w)
    n = len(data)
    for _ in range(epochs):
        grad = {k: 0.0 for k in FEATURES}
        for cands, gold in data:
            scores = [sum(w[k] * f.get(k, 0.0) for k in FEATURES) for _, f in cands]
            probs = _softmax(scores)
            for (lab, f), p in zip(cands, probs):
                coeff = ((1.0 if lab == gold else 0.0) - p)
                for k in FEATURES:
                    v = f.get(k, 0.0)
                    if v:
                        grad[k] += coeff * v
        for k in FEATURES:
            w[k] += lr * (grad[k] / n - l2 * w[k])
    return Ranker(weights=w)


def build_examples(rows, model, cfg) -> list:
    """Feature/label pairs for every mined (clause, field) supervision item."""
    from .dictation import segment_dictation
    from .routing import build_context, field_features, mine_row
    from .template import parse_template

    out = []
    for row in rows:
        gold = {i: lab for i, _, lab, _ in mine_row(row)}
        if not gold:
            continue
        tmpl = parse_template(row["template_content"])
        ctx = build_context(tmpl)
        order = {f.label: f.order for f in tmpl.fields}
        doc = segment_dictation(
            row["dictation"],
            normalize=cfg.normalize_shorthand,
            vocab=model.vocab if cfg.correct_spelling else None,
            summary_threshold=cfg.summary_threshold,
            summary_max_misses=cfg.summary_max_misses,
        )
        units = doc.findings or doc.impression
        prev_order = None
        for i, seg in enumerate(units):
            lab = gold.get(i)
            if lab is None:
                continue
            feats = field_features(seg.text, seg.cue, tmpl, model, ctx, prev_order)
            if feats:
                out.append((feats, lab))
            prev_order = order.get(lab, prev_order)
    return out


### 2.6b Impression chooser (fitted but disabled by the tuned config)

In [ ]:
%%writefile rrh/chooser.py
"""Cost-sensitive selection of the IMPRESSION strategy.

Three strategies are available for any case: reuse the radiologist's dictated
summary, condense the abnormal findings, or keep the template's normal
impression.  Which one is closest to the reference varies by case.

The learned ranker experiment showed that optimising *classification accuracy*
on a proxy label is the wrong objective, so this chooser is fitted directly on
the quantity that matters: it searches a small space of one-feature decision
rules and keeps the one that minimises total impression edit distance on the
training folds.
"""
from __future__ import annotations

from dataclasses import dataclass

VARIANTS = ("summary", "findings", "template")

FEATURES = (
    "has_summary", "n_summary", "n_findings", "n_abnormal",
    "summary_tokens", "findings_tokens", "summary_ratio", "template_negative",
)

THRESHOLDS = {
    "has_summary": (0.5,),
    "n_summary": (0.5, 1.5, 2.5, 3.5, 5.5),
    "n_findings": (0.5, 1.5, 2.5, 4.5, 7.5),
    "n_abnormal": (0.5, 1.5, 2.5, 3.5),
    "summary_tokens": (0.5, 5.5, 12.5, 25.5),
    "findings_tokens": (5.5, 15.5, 30.5, 60.5),
    "summary_ratio": (0.05, 0.25, 0.5, 0.8, 1.2),
    "template_negative": (0.5,),
}


@dataclass
class Chooser:
    """A depth<=2 decision rule over `FEATURES` returning a variant name."""

    root: tuple[str, float] | None = None
    left: "Chooser | str" = "findings"
    right: "Chooser | str" = "summary"
    default: str = "summary"

    def choose(self, feat: dict[str, float]) -> str:
        if self.root is None:
            return self.default
        name, thr = self.root
        branch = self.left if feat.get(name, 0.0) <= thr else self.right
        return branch.choose(feat) if isinstance(branch, Chooser) else branch


def _best_leaf(rows) -> tuple[str, float]:
    best, best_cost = VARIANTS[0], float("inf")
    for v in VARIANTS:
        cost = sum(r["cost"][v] for r in rows)
        if cost < best_cost - 1e-12:
            best, best_cost = v, cost
    return best, best_cost


def _fit_stump(rows) -> tuple[tuple[str, float] | None, str, str, float]:
    base_leaf, base_cost = _best_leaf(rows)
    best = (None, base_leaf, base_leaf, base_cost)
    for name in FEATURES:
        for thr in THRESHOLDS[name]:
            left = [r for r in rows if r["feat"].get(name, 0.0) <= thr]
            right = [r for r in rows if r["feat"].get(name, 0.0) > thr]
            if len(left) < 20 or len(right) < 20:
                continue
            lv, lc = _best_leaf(left)
            rv, rc = _best_leaf(right)
            if lc + rc < best[3] - 1e-12:
                best = ((name, thr), lv, rv, lc + rc)
    return best


def fit_chooser(rows, depth: int = 2) -> Chooser:
    """`rows` is a list of {"feat": {...}, "cost": {variant: res}}."""
    if not rows:
        return Chooser()
    root, lv, rv, _ = _fit_stump(rows)
    if root is None:
        return Chooser(root=None, default=lv)
    name, thr = root
    left_rows = [r for r in rows if r["feat"].get(name, 0.0) <= thr]
    right_rows = [r for r in rows if r["feat"].get(name, 0.0) > thr]
    left: Chooser | str = lv
    right: Chooser | str = rv
    if depth > 1:
        sub_l = fit_chooser(left_rows, depth - 1)
        if sub_l.root is not None:
            left = sub_l
        sub_r = fit_chooser(right_rows, depth - 1)
        if sub_r.root is not None:
            right = sub_r
    return Chooser(root=root, left=left, right=right, default=lv)


def features(summary: list[str], findings: list[str], abnormal: list[str],
             template_impression: list[str]) -> dict[str, float]:
    from .textutil import content_tokens

    st = sum(len(content_tokens(x)) for x in summary)
    ft = sum(len(content_tokens(x)) for x in findings)
    neg = bool(template_impression) and template_impression[0].strip().lower().startswith(
        ("no ", "without", "negative")
    )
    return {
        "has_summary": 1.0 if summary else 0.0,
        "n_summary": float(len(summary)),
        "n_findings": float(len(findings)),
        "n_abnormal": float(len(abnormal)),
        "summary_tokens": float(st),
        "findings_tokens": float(ft),
        "summary_ratio": st / max(1.0, float(ft)),
        "template_negative": 1.0 if neg else 0.0,
    }


def build_training_rows(train_rows, model, cfg) -> list[dict]:
    """Cost of each impression strategy on every training row.

    The cost is the edit distance of the **whole report**, not of the impression
    alone: per-section means over-weight cases whose reference impression is one
    short line, and a chooser trained on those picks strategies that make the
    report as a whole worse.
    """
    from .dictation import segment_dictation
    from .impression import is_abnormal
    from .metrics import res_char, res_word
    from .pipeline import ReportGenerator
    from .template import parse_template

    gen = ReportGenerator(model, cfg)
    out = []
    for row in train_rows:
        tmpl = parse_template(row["template_content"])
        doc = segment_dictation(
            row["dictation"],
            normalize=cfg.normalize_shorthand,
            vocab=model.vocab if cfg.correct_spelling else None,
            summary_threshold=cfg.summary_threshold,
            summary_max_misses=cfg.summary_max_misses,
            summary_after_cues=cfg.summary_after_cues,
        )
        summary = [s.text for s in doc.impression]
        findings = [s.text for s in doc.findings]
        abnormal = [x for x in findings if is_abnormal(x)]
        ref = row["report"]
        cost = {}
        for v in VARIANTS:
            pred, _ = gen.generate(row, force_impression=v)
            cost[v] = (res_word(pred, ref) + res_char(pred, ref)) / 2
        out.append({
            "feat": features(summary, findings, abnormal, tmpl.impression),
            "cost": cost,
        })
    return out


### 2.7 Template editing - replace only what is contradicted

In [ ]:
%%writefile rrh/editor.py
"""Template editing: fold routed findings into the template's normal statements.

Guiding rule (RULE 2/3/6 of the task): replace only the normal statement that
the dictation contradicts, keep everything else byte-identical to the template.
"""
from __future__ import annotations

import re

from .lexicon import stem
from .splitting import split_negation
from .textutil import content_tokens, sim, split_sentences, squash, tidy_sentence

NEGATIVE_RE = re.compile(r"^\s*(no|there is no|there are no|without|negative for)\b", re.I)
NORMAL_STATE_RE = re.compile(
    r"\b(unremarkable|normal|intact|preserved|maintained|within normal limits|clear|"
    r"patent|not widened|no significant)\b",
    re.I,
)
BLANKET_RE = re.compile(
    r"^(the\s+)?[\w\s/-]{0,40}\s*(is|are)\s+(unremarkable|normal|preserved|maintained|"
    r"intact|clear|within normal (size )?limits?)\.?$",
    re.I,
)
FINITE_VERB = re.compile(
    r"\b(is|are|was|were|has|have|had|shows?|demonstrates?|reveals?|appears?|measures?|"
    r"noted|seen|identified|present|extends?|involves?|causes?|produces?|results?|"
    r"remains?|persists?|distends?|projects?)\b",
    re.I,
)
PLURAL_TAIL = re.compile(r"(?<![aiou])s$|(?<=[^s])es$", re.I)
PREPOSITION = re.compile(
    r"\b(of|in|at|along|about|within|with|involving|throughout|around|over|between)\b", re.I
)
LEADING_THERE = re.compile(r"^there\b", re.I)

COPULA_TAIL = re.compile(
    r"\s+(?:is|are|was|were)\s+(?:identified|seen|noted|present|evident|visualized|"
    r"visualised|appreciated|demonstrated|observed)\b",
    re.I,
)


def is_negative(sentence: str) -> bool:
    return bool(NEGATIVE_RE.match(sentence.strip()))


def negated_entities(sentence: str) -> list[str]:
    parts = split_negation(sentence)
    if not parts:
        m = re.match(r"^\s*(?:no|there is no|there are no)\s+(.+?)\s*[.]?$", sentence, re.I)
        return [m.group(1)] if m else []
    out = []
    for p in parts:
        m = re.match(r"^\s*No\s+(.+?)\s*[.]?$", p, re.I)
        if m:
            out.append(COPULA_TAIL.sub("", m.group(1)).strip())
    return out


def _stems(text: str) -> set[str]:
    return {stem(t) for t in content_tokens(text)}


def _asserted_positively(entity: str, routed: list[str]) -> bool:
    """Does any routed finding assert `entity` as present?"""
    ent = _stems(entity)
    if not ent:
        return False
    for r in routed:
        rs = _stems(r)
        if not rs:
            continue
        if len(ent & rs) / len(ent) < 0.7:
            continue
        if is_negative(r):
            # the routed sentence also negates it -> not a contradiction
            r_ents = negated_entities(r)
            if any(len(ent & _stems(e)) / len(ent) >= 0.7 for e in r_ents):
                continue
        return True
    return False


def _covered(sentence: str, routed: list[str], threshold: float) -> bool:
    st = _stems(sentence)
    if not st:
        return True
    union: set[str] = set()
    for r in routed:
        union |= _stems(r)
        if sim(sentence, r) >= 0.62:
            return True
    return len(st & union) / len(st) >= threshold


def _rebuild_negative(original: str, kept: list[str]) -> str:
    tail_m = COPULA_TAIL.search(original)
    tail = tail_m.group(0) if tail_m else ""
    if len(kept) == 1:
        body = kept[0]
    else:
        body = ", ".join(kept[:-1]) + " or " + kept[-1]
    return tidy_sentence(f"No {body}{tail}")


def _is_plural_head(text: str) -> bool:
    head = PREPOSITION.split(text, maxsplit=1)[0]
    if re.search(r"\band\b", head, re.I):
        return True
    for w in re.findall(r"[A-Za-z]+", head):
        low = w.lower()
        if low.endswith(("sis", "ss", "us", "is", "ous")):
            continue
        if PLURAL_TAIL.search(low):
            return True
    return False


def render_clause(text: str, cfg) -> str:
    """Render a routed dictation clause as a report sentence.

    Telegraphic noun phrases ("degenerative changes in left shoulder") are given
    the existential frame the reference reports use ("There are degenerative
    changes in the left shoulder.").  No content is added.
    """
    t = squash(text)
    verbless = t and not FINITE_VERB.search(t) and not NEGATIVE_RE.match(t)
    if verbless and not LEADING_THERE.match(t):
        if cfg.add_existential:
            t = f"There {'are' if _is_plural_head(t) else 'is'} {t[0].lower() + t[1:]}"
        elif cfg.add_copula:
            t = f"{t.rstrip('.')} {'are' if _is_plural_head(t) else 'is'} present"
    return tidy_sentence(t)


def _redundant_with_template(clause: str, tmpl_sents: list[str]) -> bool:
    """True when the template already says this, in its own words (RULE 6)."""
    c_ents = negated_entities(clause)
    c_st = _stems(clause)
    if not c_st:
        return True
    for ts in tmpl_sents:
        if sim(clause, ts) >= 0.55:
            return True
        if not is_negative(ts):
            continue
        t_ents = negated_entities(ts)
        if not c_ents or not t_ents:
            continue
        t_all = set()
        for e in t_ents:
            t_all |= _stems(e)
        if all(
            _stems(e) and len(_stems(e) & t_all) / len(_stems(e)) >= 0.7 for e in c_ents
        ):
            return True
    return False


def filter_redundant(routed: list[str], template_text: str, cfg) -> list[str]:
    """Drop dictated clauses that merely restate a template normal statement."""
    if not cfg.suppress_redundant_negatives or not routed:
        return routed
    tmpl_sents = split_sentences(template_text)
    if not tmpl_sents:
        return routed
    out = []
    for r in routed:
        negative = is_negative(r)
        normalish = bool(NORMAL_STATE_RE.search(r))
        if (negative or (cfg.suppress_redundant_normals and normalish)) and (
            _redundant_with_template(r, tmpl_sents)
        ):
            continue
        out.append(r)
    return out


MEASURE_RE = re.compile(r"\b\d+(?:\.\d+)?\s*(?:x\s*\d+(?:\.\d+)?\s*)*(?:mm|cm|ml|cc)\b", re.I)


def dedupe_clauses(routed: list[str], threshold: float) -> list[str]:
    """Drop a clause the field already states (dictated per-level summaries).

    A clause carrying a measurement the field does not have yet is always kept:
    de-duplication must never silently lose a number.
    """
    kept: list[str] = []
    for r in routed:
        st = _stems(r)
        if st:
            prev_text = " ".join(kept)
            prev = _stems(prev_text)
            new_measures = {
                m.group(0).lower().replace(" ", "") for m in MEASURE_RE.finditer(r)
            } - {m.group(0).lower().replace(" ", "") for m in MEASURE_RE.finditer(prev_text)}
            if not new_measures and (
                len(st & prev) / len(st) >= threshold or any(sim(r, k) >= 0.7 for k in kept)
            ):
                continue
        kept.append(r)
    return kept


def edit_field(template_text: str, routed: list[str], cfg) -> str:
    """Return the new body for one field."""
    if cfg.dedupe_threshold:
        routed = dedupe_clauses(routed, cfg.dedupe_threshold)
    routed = filter_redundant(routed, template_text, cfg)
    if not routed:
        return template_text
    kept: list[str] = []
    for ts in split_sentences(template_text):
        if is_negative(ts):
            ents = negated_entities(ts)
            if ents:
                survivors = [e for e in ents if not _asserted_positively(e, routed)]
                if not survivors:
                    continue
                if len(survivors) < len(ents):
                    kept.append(_rebuild_negative(ts, survivors))
                    continue
            if _covered(ts, routed, cfg.cover_threshold):
                continue
            kept.append(ts)
        else:
            if _covered(ts, routed, cfg.cover_threshold):
                continue
            if cfg.soften_blanket and BLANKET_RE.match(ts) and any(
                not is_negative(r) for r in routed
            ):
                kept.append(_soften(ts))
            else:
                kept.append(ts)
    ordered = routed
    if cfg.abnormal_first:
        pos = [r for r in routed if not is_negative(r) and not NORMAL_STATE_RE.search(r)]
        rest = [r for r in routed if r not in pos]
        ordered = pos + rest
    body = " ".join(render_clause(r, cfg) for r in ordered if squash(r))
    if kept:
        body = (body + " " + " ".join(tidy_sentence(k) for k in kept)).strip()
    return squash(body)


def _soften(sentence: str) -> str:
    s = sentence
    s = re.sub(r"^The\s+", "The remaining ", s, count=1)
    s = re.sub(r"\s+(is|are)\s+", lambda m: f" {m.group(1)} otherwise ", s, count=1)
    return tidy_sentence(s)


### 2.8 Impression builder

In [ ]:
%%writefile rrh/impression.py
"""IMPRESSION construction.

Two branches, both purely extractive:
  * the radiologist dictated a summary  -> reuse it (that is what the reference
    reports do), minus normal/negative filler;
  * no dictated summary                 -> condense the abnormal findings that
    were routed into FINDINGS and close with the template's normal impression.

Nothing is ever added that the dictation or template did not state.
"""
from __future__ import annotations

import re

from .dictation import POS_HINT
from .editor import NORMAL_STATE_RE, is_negative
from .template import resolve_placeholders
from .textutil import content_tokens, sim, squash, tidy_sentence

LEAD_EXISTENTIAL = re.compile(r"^there\s+(?:is|are|was|were)\s+(?:a|an|the)?\s*", re.I)
COPULA_TAIL = re.compile(
    r"\s+(?:is|are|was|were)\s+(?:identified|seen|noted|present|evident|visualized|"
    r"visualised|appreciated|demonstrated|observed)\b\.?$",
    re.I,
)
LEAD_ARTICLE = re.compile(r"^(?:a|an)\s+", re.I)
SPECIFICALLY = re.compile(r"^specifically,\s*", re.I)
# A template impression that asserts a normal study must not be appended next to
# abnormal findings - that would contradict the report.
TEMPLATE_NEGATIVE = re.compile(r"^\s*(no|without|negative for)\b", re.I)

DETAIL_TAIL = re.compile(
    r",?\s+(?:with|including|associated with|demonstrating|showing|producing|resulting in|"
    r"causing)\s+.+$",
    re.I,
)


def is_normal_statement(sentence: str) -> bool:
    return bool(NORMAL_STATE_RE.search(sentence)) and not POS_HINT.search(sentence)


def is_abnormal(sentence: str) -> bool:
    return (
        not is_negative(sentence)
        and not is_normal_statement(sentence)
        and bool(POS_HINT.search(sentence))
    )


def condense(text: str, trim_detail: bool = False) -> str:
    """Turn a findings sentence into an impression item (removal only)."""
    t = squash(text)
    t = LEAD_EXISTENTIAL.sub("", t)
    t = COPULA_TAIL.sub("", t)
    t = LEAD_ARTICLE.sub("", t)
    t = SPECIFICALLY.sub("", t)
    if trim_detail:
        head = DETAIL_TAIL.sub("", t)
        if len(content_tokens(head)) >= 3:
            t = head
    return tidy_sentence(t)


SEVERITY = (
    (re.compile(r"\b(severe|marked|extensive|large|complete|full-thickness|acute|"
                r"high-grade|gross|advanced)\b", re.I), 3.0),
    (re.compile(r"\b(moderate|moderate-to-severe|mild-to-moderate)\b", re.I), 2.0),
    (re.compile(r"\b(mild|minimal|small|trace|low-grade|early|subtle)\b", re.I), 1.0),
)


def severity(text: str) -> float:
    for pattern, weight in SEVERITY:
        if pattern.search(text):
            return weight
    return 1.5


def _dedupe(items: list[str], threshold: float = 0.82) -> list[str]:
    out: list[str] = []
    for it in items:
        if not squash(it):
            continue
        if any(sim(it, o) >= threshold for o in out):
            continue
        out.append(it)
    return out


def build_impression(
    dictated_summary: list[str],
    findings: list[str],
    template_impression: list[str],
    laterality: str | None,
    region: str | None,
    cfg,
) -> list[str]:
    tmpl_lines = [
        tidy_sentence(resolve_placeholders(x, laterality, region))
        for x in template_impression
        if squash(x)
    ]

    if dictated_summary:
        items = [condense(x, trim_detail=cfg.trim_summary_detail) for x in dictated_summary]
        cap = cfg.summary_cap
    else:
        source = findings
        if cfg.findings_require_abnormal:
            abnormal = [x for x in findings if is_abnormal(x)]
            if not abnormal:
                # nothing abnormal was dictated: restating normal findings is
                # not an impression.  Fall back to the template's normal line,
                # optionally preceded by the dictation's own negative summary.
                if cfg.no_abnormal_fallback == "negatives":
                    source = [x for x in findings if is_negative(x)][-1:]
                else:
                    source = []
            else:
                source = abnormal
        items = [condense(x, trim_detail=cfg.trim_detail) for x in source]
        cap = cfg.findings_cap

    items = [x for x in items if not is_normal_statement(x)]
    if cfg.drop_negative_impression and not (
        not dictated_summary and cfg.no_abnormal_fallback == "negatives" and items
        and all(is_negative(x) for x in items)
    ):
        items = [x for x in items if not is_negative(x)]
    items = _dedupe(items, cfg.impression_dedupe)
    if cfg.rank_impression_by_severity and not dictated_summary:
        items = sorted(items, key=lambda x: -severity(x))
    if cap:
        items = items[:cap]

    if not dictated_summary and items and cfg.append_template_impression:
        closing = [x for x in tmpl_lines[:1] if TEMPLATE_NEGATIVE.match(x)]
        items = items + closing
    if not items:
        items = tmpl_lines or ["No acute abnormality."]

    if cfg.number_impression and len(items) > 1:
        return [f"{i}. {t}" for i, t in enumerate(items, 1)]
    return items


### 2.9 End-to-end pipeline

In [ ]:
%%writefile rrh/pipeline.py
"""End-to-end structured-generation pipeline.

    dictation ─▶ segment ─▶ split ─▶ route ─▶ edit template ─▶ impression ─▶ validate
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .dictation import segment_dictation
from .editor import edit_field, is_negative, render_clause
from .impression import build_impression
from .routing import (RoutingModel, build_context, cue_supported, field_candidates,
                      fit_ranked_router, score_segment)
from .splitting import candidate_splits
from .template import parse_template, render_report, resolve_placeholders
from .lexicon import body_region, stem
from .textutil import content_tokens, squash


@dataclass
class Config:
    # routing
    min_route_score: float = 0.45
    group_penalty: float = 1.0
    allow_splitting: bool = True
    cue_veto_penalty: float = 0.8
    w_cue: float = 3.0
    w_label: float = 2.2
    w_concept: float = 1.2
    w_template: float = 1.0
    w_mined: float = 1.2
    w_knn: float = 1.6
    w_continuity: float = 0.0
    w_backward: float = 0.0
    w_prior: float = 0.0
    region_weight: float = 0.0
    use_impression_chooser: bool = False
    chooser_depth: int = 2
    use_bigrams: bool = False
    mine_threshold: float = 0.5
    viterbi: bool = False
    use_ranker: bool = False
    ranker_epochs: int = 300
    ranker_lr: float = 0.25
    ranker_l2: float = 0.001
    cue_match_min: float = 0.34
    split_margin: float = 0.0
    # editing
    cover_threshold: float = 0.12
    dedupe_threshold: float = 0.0  # 0 disables within-field de-duplication
    soften_blanket: bool = False
    add_copula: bool = False
    add_existential: bool = False
    abnormal_first: bool = False
    style_threshold: float = 0.0  # 0 disables reference-phrasing transfer
    correct_spelling: bool = True
    suppress_redundant_negatives: bool = False
    suppress_redundant_normals: bool = False
    normalize_shorthand: bool = True
    summary_threshold: float = 0.34
    summary_max_misses: int = 3
    summary_after_cues: bool = False
    recover_summary: bool = True
    summary_recover_threshold: float = 0.6
    # impression
    append_template_impression: bool = True
    number_impression: bool = True
    drop_negative_impression: bool = True
    trim_detail: bool = True
    trim_summary_detail: bool = False
    allow_conjunction_split: bool = False
    findings_require_abnormal: bool = True
    no_abnormal_fallback: str = "template"  # template | negatives
    rank_impression_by_severity: bool = False
    summary_cap: int = 6
    impression_dedupe: float = 0.82
    findings_cap: int = 2
    # rendering
    blank_between_fields: bool = True
    keep_unrouted: bool = True
    merge_extras: bool = False


MODEL_KEYS = (
    "normalize_shorthand", "correct_spelling", "summary_threshold", "summary_max_misses",
    "use_ranker", "ranker_epochs", "ranker_lr", "ranker_l2", "summary_after_cues",
    "region_weight", "use_impression_chooser", "chooser_depth", "use_bigrams",
    "mine_threshold",
)


def model_key(cfg: "Config") -> str:
    """Identity of everything that changes the *fitted* model, for caching."""
    return "|".join(f"{k}={getattr(cfg, k)}" for k in MODEL_KEYS)


LATERAL_WORDS = {
    "rt": "right", "r": "right", "right": "right",
    "lt": "left", "l": "left", "left": "left",
    "bilateral": "bilateral", "b/l": "bilateral", "bilat": "bilateral", "both": "bilateral",
}
REGION_WORDS = {
    "lumbar spine": "lumbar", "lsspine": "lumbosacral", "thoracic spine": "thoracic",
    "cervical spine": "cervical", "spine sacrum": "sacral", "sacrum": "sacral",
}


def infer_laterality(row) -> str | None:
    for src in (row.get("study_description") or "", row.get("dictation") or ""):
        for m in re.finditer(r"\b(rt|lt|right|left|bilateral|bilat|b/l|both)\b", src, re.I):
            return LATERAL_WORDS.get(m.group(1).lower())
    return None


def infer_region(row) -> str | None:
    bp = (row.get("body_part") or "").strip().lower()
    if bp in REGION_WORDS:
        return REGION_WORDS[bp]
    sd = (row.get("study_description") or "").lower()
    for kw, val in (("lsp", "lumbar"), ("tsp", "thoracic"), ("csp", "cervical"),
                    ("lumbo", "lumbosacral"), ("lumbar", "lumbar"),
                    ("thoracic", "thoracic"), ("cervical", "cervical")):
        if kw in sd:
            return val
    return bp or None


@dataclass
class Trace:
    """What the pipeline decided - used by the validator and for auditing."""
    routed: dict[str, list[str]] = dc_field(default_factory=dict)
    extras: list[str] = dc_field(default_factory=list)
    dictated_impression: list[str] = dc_field(default_factory=list)
    abnormal: list[str] = dc_field(default_factory=list)
    unrouted_scores: list[tuple[str, float]] = dc_field(default_factory=list)
    normal_case: bool = False
    laterality: str | None = None
    region: str | None = None


class ReportGenerator:
    def __init__(self, model: RoutingModel, cfg: Config | None = None):
        self.model = model
        self.cfg = cfg or Config()
        c = self.cfg
        self._weights = {
            "cue": c.w_cue, "label": c.w_label, "concept": c.w_concept,
            "template": c.w_template, "mined": c.w_mined, "knn": c.w_knn,
            "continuity": c.w_continuity, "backward": c.w_backward, "prior": c.w_prior,
            "use_ranker": 1.0 if c.use_ranker else 0.0,
        }

    # -------------------------------------------------------------- routing
    def _route_one(self, text: str, cue: str | None, tmpl, ctx, prev_order=None):
        scored = score_segment(text, cue, tmpl, self.model, ctx, self._weights, prev_order)
        if not scored:
            return None, 0.0
        groups = {f.label for f in field_candidates(tmpl) if f.is_group}
        adjusted = [
            (s - (self.cfg.group_penalty if lab in groups else 0.0), lab) for s, lab in scored
        ]
        adjusted.sort(key=lambda x: (-x[0], x[1]))
        best_score, best_label = adjusted[0]
        if self.cfg.cue_veto_penalty and not cue_supported(
            cue, tmpl, ctx, self.cfg.cue_match_min
        ):
            best_score -= self.cfg.cue_veto_penalty
        if best_score < self.cfg.min_route_score:
            return None, best_score
        return best_label, best_score

    def _route_segment(self, text: str, cue: str | None, tmpl, ctx, prev_order=None):
        """Return list of (clause text, label|None, score)."""
        label, score = self._route_one(text, cue, tmpl, ctx, prev_order)
        whole = [(text, label, score)]
        if not self.cfg.allow_splitting:
            return whole
        for parts in candidate_splits(text, self.cfg.allow_conjunction_split):
            routed = [self._route_one(p, cue, tmpl, ctx, prev_order) for p in parts]
            labels = [lab for lab, _ in routed]
            if any(lab is None for lab in labels):
                continue
            if len(set(labels)) < 2:
                continue
            avg = sum(s for _, s in routed) / len(routed)
            if avg + self.cfg.split_margin >= score:
                return [(p, lab, s) for p, (lab, s) in zip(parts, routed)]
        return whole

    def _viterbi(self, clauses, tmpl, ctx):
        """Re-decode a fixed clause sequence as a path, not as independent picks.

        Emissions are the per-clause field scores; transitions encode the
        template-order structure of dictations (stay in the field, move on, or
        pay to jump backwards).  A NULL state absorbs clauses that belong to no
        field of this template.
        """
        cfg = self.cfg
        cands = field_candidates(tmpl)
        if not cands or not clauses:
            return [None] * len(clauses)
        groups = {f.label for f in cands if f.is_group}
        order = {f.label: f.order for f in cands}
        span = max(1, len(tmpl.fields) - 1)
        states = [f.label for f in cands] + [None]

        emissions = []
        for text, cue in clauses:
            scored = dict(
                (lab, sc) for sc, lab in score_segment(text, cue, tmpl, self.model, ctx,
                                                       self._weights)
            )
            penalty = (
                cfg.cue_veto_penalty
                if cfg.cue_veto_penalty
                and not cue_supported(cue, tmpl, ctx, cfg.cue_match_min)
                else 0.0
            )
            row = {}
            for lab in states:
                if lab is None:
                    row[lab] = cfg.min_route_score
                else:
                    row[lab] = (
                        scored.get(lab, 0.0)
                        - (cfg.group_penalty if lab in groups else 0.0)
                        - penalty
                    )
            emissions.append(row)

        def transition(prev, cur):
            if prev is None or cur is None:
                return 0.0
            a, b = order[prev], order[cur]
            if a == b:
                return cfg.w_continuity
            if b < a:
                return -cfg.w_backward * min(1.0, (a - b) / span)
            return 0.0

        best = {s: (emissions[0][s], [s]) for s in states}
        for row in emissions[1:]:
            nxt = {}
            for cur in states:
                score, path = max(
                    ((best[prev][0] + transition(prev, cur), best[prev][1]) for prev in states),
                    key=lambda x: x[0],
                )
                nxt[cur] = (score + row[cur], path + [cur])
            best = nxt
        return max(best.values(), key=lambda x: x[0])[1]

    # ------------------------------------------------------------- generate
    def generate(self, row: dict, force_impression: str | None = None) -> tuple[str, Trace]:
        cfg = self.cfg
        tmpl = parse_template(row["template_content"])
        ctx = build_context(
            tmpl,
            body_region(
                str(row.get("body_part") or ""), str(row.get("study_description") or "")
            ),
        )
        doc = segment_dictation(
            row.get("dictation") or "",
            normalize=cfg.normalize_shorthand,
            vocab=self.model.vocab if cfg.correct_spelling else None,
            summary_threshold=cfg.summary_threshold,
            summary_max_misses=cfg.summary_max_misses,
            summary_after_cues=cfg.summary_after_cues,
        )
        laterality = infer_laterality(row)
        region = infer_region(row)
        trace = Trace(
            normal_case=doc.is_normal or not doc.findings,
            laterality=laterality,
            region=region,
        )

        routed: dict[str, list[str]] = {}
        extras: list[str] = []
        ordered_findings: list[str] = []
        units = list(doc.findings)
        if cfg.recover_summary and doc.impression:
            # A sentence in the dictated summary that restates nothing from the
            # body is not a summary at all - it is a finding, and must not be
            # lost just because it was dictated last.
            seen = {stem(t) for seg in doc.findings for t in content_tokens(seg.text)}
            for seg in doc.impression:
                st = {stem(t) for t in content_tokens(seg.text)}
                if st and len(st & seen) / len(st) < cfg.summary_recover_threshold:
                    units.append(seg)
        field_order = {f.label: f.order for f in tmpl.fields}
        prev_order: int | None = None
        for seg in units:
            for clause, label, score in self._route_segment(
                seg.text, seg.cue, tmpl, ctx, prev_order
            ):
                clause = squash(clause)
                if not clause:
                    continue
                if cfg.style_threshold:
                    styled = self.model.style_match(clause, cfg.style_threshold)
                    if styled:
                        clause = styled
                if label is None:
                    if cfg.keep_unrouted:
                        extras.append(clause)
                    trace.unrouted_scores.append((clause, score))
                else:
                    routed.setdefault(label, []).append(clause)
                    prev_order = field_order.get(label, prev_order)
                ordered_findings.append(clause)

        field_texts: list[tuple[str, str]] = []
        for f in tmpl.fields:
            if f.is_free:
                field_texts.append(("", resolve_placeholders(f.text, laterality, region)))
                continue
            if f.label == "OTHER FINDINGS":
                field_texts.append((f.label, ""))
                continue
            body = edit_field(f.text, routed.get(f.label, []), cfg)
            field_texts.append((f.label, resolve_placeholders(body, laterality, region)))

        summary_items = [s.text for s in doc.impression]
        if force_impression is not None:
            if force_impression == "findings":
                summary_items = []
            elif force_impression == "template":
                summary_items, ordered_findings = [], []
        elif cfg.use_impression_chooser and self.model.chooser is not None:
            from .chooser import features as chooser_features
            from .impression import is_abnormal

            pick = self.model.chooser.choose(
                chooser_features(
                    summary_items,
                    ordered_findings,
                    [x for x in ordered_findings if is_abnormal(x)],
                    tmpl.impression,
                )
            )
            if pick == "findings":
                summary_items = []
            elif pick == "template":
                summary_items, ordered_findings = [], []
        impression = build_impression(
            summary_items,
            ordered_findings,
            tmpl.impression,
            laterality,
            region,
            cfg,
        )
        report = render_report(
            field_texts,
            impression,
            extra_paragraphs=[render_clause(e, cfg) for e in extras],
            blank_between_fields=cfg.blank_between_fields,
            merge_extras=cfg.merge_extras,
        )
        trace.routed = routed
        trace.extras = extras
        trace.dictated_impression = [s.text for s in doc.impression]
        trace.abnormal = [f for f in ordered_findings if not is_negative(f)]
        return report, trace


def build_generator(train_rows, cfg: Config | None = None) -> ReportGenerator:
    cfg = cfg or Config()
    return ReportGenerator(fit_ranked_router(train_rows, cfg), cfg)


### 2.10 Validation - negation, laterality, measurements, hallucination

In [ ]:
%%writefile rrh/validate.py
"""Validation layer.

Runs after generation and answers the questions the task brief asks for:
negation preserved, laterality preserved, measurements preserved, nothing
invented, untouched template fields untouched, dictated findings not dropped.

`validate` reports issues; `repair` fixes the ones that can be fixed
deterministically (a dropped finding is re-attached rather than lost).
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .dictation import LATERAL_RE, MEASURE_RE, segment_dictation
from .editor import is_negative
from .lexicon import stem
from .template import parse_template, resolve_placeholders
from .textutil import content_tokens, sim, split_sentences, squash


@dataclass
class Issue:
    kind: str
    detail: str
    severity: str = "warn"


@dataclass
class ValidationResult:
    issues: list[Issue] = dc_field(default_factory=list)

    def add(self, kind: str, detail: str, severity: str = "warn") -> None:
        self.issues.append(Issue(kind, detail, severity))

    @property
    def ok(self) -> bool:
        return not any(i.severity == "error" for i in self.issues)

    def counts(self) -> dict[str, int]:
        out: dict[str, int] = {}
        for i in self.issues:
            out[i.kind] = out.get(i.kind, 0) + 1
        return out


def _lateralities(text: str) -> set[str]:
    out = set()
    for m in LATERAL_RE.finditer(text or ""):
        v = m.group(1).lower()
        out.add({"rt": "right", "lt": "left", "b/l": "bilateral", "bilat": "bilateral"}.get(v, v))
    return out


def _measurements(text: str) -> set[str]:
    return {squash(m.group(0)).lower().replace(" ", "") for m in MEASURE_RE.finditer(text or "")}


LABEL_PREFIX = re.compile(r"^[ \t]*[A-Z][A-Z0-9 ,'\-/&\.\(\)]{0,58}?:[ \t]*", re.M)


def report_sentences(report: str) -> list[str]:
    """Sentences of a report with the field labels stripped off."""
    return split_sentences(LABEL_PREFIX.sub("", report))


def validate(row: dict, report: str, trace, vocab: dict | None = None) -> ValidationResult:
    res = ValidationResult()
    tmpl = parse_template(row["template_content"])
    out = parse_template(report)
    doc = segment_dictation(row.get("dictation") or "", vocab=vocab)

    # ---- structure -----------------------------------------------------
    if "FINDINGS:" not in report or "IMPRESSION:" not in report:
        res.add("structure", "missing FINDINGS/IMPRESSION header", "error")
    t_labels = [f.label for f in tmpl.fields if f.label]
    o_labels = [f.label for f in out.fields if f.label]
    if t_labels != o_labels:
        res.add("structure", f"label sequence changed: {set(t_labels) ^ set(o_labels)}", "error")
    if re.search(r"\[[^\]]*\]", report):
        res.add("placeholder", "unresolved [placeholder] left in report", "error")

    # ---- untouched fields ---------------------------------------------
    out_by_label = {f.label: squash(f.text) for f in out.fields if f.label}
    for f in tmpl.fields:
        if not f.label or f.label == "OTHER FINDINGS":
            continue
        if trace.routed.get(f.label):
            continue
        expected = squash(
            resolve_placeholders(f.text, trace.laterality, trace.region)
            if "[" in f.text
            else f.text
        )
        if out_by_label.get(f.label, "") != expected:
            res.add("untouched_field", f"{f.label} changed without a routed finding", "error")

    # ---- laterality ----------------------------------------------------
    sentences = report_sentences(report)
    for seg in doc.findings:
        lat = _lateralities(seg.text)
        if not lat:
            continue
        matched = [s for s in sentences if sim(seg.text, s) >= 0.5]
        if matched and not any(_lateralities(s) & lat for s in matched):
            res.add("laterality", f"laterality {sorted(lat)} lost: {seg.text[:70]}", "error")

    # ---- negation ------------------------------------------------------
    for seg in doc.findings:
        if not is_negative(seg.text):
            continue
        toks = {stem(t) for t in content_tokens(seg.text)}
        if not toks:
            continue
        matches = [
            s
            for s in sentences
            if (st := {stem(t) for t in content_tokens(s)}) and len(toks & st) / len(toks) >= 0.8
        ]
        if not matches:
            continue
        # the negation is preserved as long as *some* matching sentence still
        # states it negatively (the same phrase may also appear, correctly, as a
        # positive finding at another level or site)
        if not any(
            is_negative(s) or "without" in s.lower() or " no " in f" {s.lower()} "
            for s in matches
        ):
            res.add("negation", f"negated finding rendered positive: {seg.text[:70]}", "error")

    # ---- measurements --------------------------------------------------
    dict_meas = set()
    for seg in doc.findings + doc.impression:
        dict_meas |= _measurements(seg.text)
    lost = dict_meas - _measurements(report)
    for m in sorted(lost):
        res.add("measurement", f"measurement dropped: {m}")

    # ---- unsupported content (hallucination) ---------------------------
    allowed = set()
    normalised = " ".join(s.text for s in doc.preamble + doc.findings + doc.impression)
    for src in (row["template_content"], row.get("dictation") or "", normalised):
        allowed |= {stem(t) for t in content_tokens(src)}
    allowed |= {stem(t) for t in content_tokens(str(row.get("body_part") or ""))}
    allowed |= {stem(t) for t in content_tokens(str(row.get("study_description") or ""))}
    allowed |= {"remaining", "otherwise", "left", "right", "bilateral", "lumbar", "thoracic",
                "cervical", "lumbosacral", "sacral"}
    unseen = sorted({stem(t) for t in content_tokens(report)} - allowed)
    for t in unseen:
        res.add("unsupported_term", f"term not present in template or dictation: {t}", "error")

    # ---- omissions -----------------------------------------------------
    for seg in doc.findings:
        if len(content_tokens(seg.text)) < 2:
            continue
        toks = {stem(t) for t in content_tokens(seg.text)}
        rep_toks = {stem(t) for t in content_tokens(report)}
        if len(toks & rep_toks) / len(toks) < 0.6:
            res.add("omission", f"dictated finding not represented: {seg.text[:70]}")
    return res


def repair(row: dict, report: str, trace, result: ValidationResult) -> str:
    """Re-attach dictated findings that were dropped, before IMPRESSION."""
    missing = [i.detail.split(": ", 1)[1] for i in result.issues if i.kind == "omission"]
    if not missing:
        return report
    doc = segment_dictation(row.get("dictation") or "")
    texts = []
    for seg in doc.findings:
        if any(seg.text[:70] == m for m in missing):
            texts.append(squash(seg.text))
    if not texts:
        return report
    idx = report.find("IMPRESSION:")
    if idx < 0:
        return report
    block = "\n" + "\n\n".join(texts) + "\n\n"
    return report[:idx].rstrip("\n") + "\n" + block + report[idx:]


### 2.11 Offline scoring (RES proxies)

In [ ]:
%%writefile rrh/metrics.py
"""Offline scoring.

The leaderboard metric (RES - Radiology Edit Score, lower is better) is not
published, so we optimise a *family* of edit-based proxies and report them all.
`res_word` is the headline number used for tuning; `res_char` and the
structural scores guard against over-fitting one particular formulation.
"""
from __future__ import annotations

import re
from dataclasses import dataclass

from .template import parse_template
from .textutil import key, levenshtein, squash, token_set


def _words(text: str) -> list[str]:
    return key(text).split()


def res_word(pred: str, ref: str) -> float:
    r = _words(ref)
    p = _words(pred)
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def res_char(pred: str, ref: str) -> float:
    r = squash(ref)
    p = squash(pred)
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def res_sent(pred: str, ref: str) -> float:
    """Edit distance counted in whole sentences - the unit a radiologist edits."""
    from .textutil import split_sentences

    r = [key(x) for x in split_sentences(ref) if key(x)]
    p = [key(x) for x in split_sentences(pred) if key(x)]
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def res_raw(pred: str, ref: str) -> float:
    """Formatting-sensitive variant: raw characters, nothing normalised."""
    r = (ref or "").strip()
    p = (pred or "").strip()
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def field_scores(pred: str, ref: str) -> dict[str, float]:
    """Per-field agreement: did we edit the same fields, the same way?"""
    P = {f.label: squash(f.text) for f in parse_template(pred).fields if f.label}
    R = {f.label: squash(f.text) for f in parse_template(ref).fields if f.label}
    if not R:
        return {"field_label_f1": 1.0, "field_exact": 1.0, "field_res": 0.0}
    inter = set(P) & set(R)
    prec = len(inter) / len(P) if P else 0.0
    rec = len(inter) / len(R)
    f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    exact = sum(1 for k in inter if key(P[k]) == key(R[k])) / len(R)
    fres = sum(res_word(P.get(k, ""), R[k]) for k in R) / len(R)
    return {"field_label_f1": f1, "field_exact": exact, "field_res": fres}


_IMP = re.compile(r"^[ \t]*IMPRESSION[ \t]*:[ \t]*", re.I | re.M)


def split_parts(text: str) -> tuple[str, str]:
    m = _IMP.search(text)
    return (text[: m.start()], text[m.end() :]) if m else (text, "")


@dataclass
class Score:
    n: int
    res_word: float
    res_char: float
    res_raw: float
    res_sent: float
    findings_res: float
    impression_res: float
    field_label_f1: float
    field_exact: float
    field_res: float
    content_recall: float
    content_precision: float

    def as_row(self) -> str:
        return (
            f"n={self.n}  RES_word={self.res_word:.4f}  RES_char={self.res_char:.4f}  "
            f"RES_raw={self.res_raw:.4f}  RES_sent={self.res_sent:.4f}  "
            f"find={self.findings_res:.4f}  imp={self.impression_res:.4f}  "
            f"labelF1={self.field_label_f1:.4f}  fieldExact={self.field_exact:.4f}  "
            f"fieldRES={self.field_res:.4f}  cRec={self.content_recall:.3f}  "
            f"cPrec={self.content_precision:.3f}"
        )


def evaluate(preds: list[str], refs: list[str]) -> Score:
    acc = {
        k: 0.0
        for k in (
            "res_word res_char res_raw res_sent findings_res impression_res field_label_f1 "
            "field_exact field_res content_recall content_precision"
        ).split()
    }
    n = len(refs)
    for p, r in zip(preds, refs):
        acc["res_word"] += res_word(p, r)
        acc["res_char"] += res_char(p, r)
        acc["res_raw"] += res_raw(p, r)
        acc["res_sent"] += res_sent(p, r)
        pf, pi = split_parts(p)
        rf, ri = split_parts(r)
        acc["findings_res"] += res_word(pf, rf)
        acc["impression_res"] += res_word(pi, ri)
        for k, v in field_scores(p, r).items():
            acc[k] += v
        tp, tr_ = token_set(p), token_set(r)
        acc["content_recall"] += len(tp & tr_) / max(1, len(tr_))
        acc["content_precision"] += len(tp & tr_) / max(1, len(tp))
    return Score(n=n, **{k: v / max(1, n) for k, v in acc.items()})


## 3. Fit the routing model on the training set

In [ ]:
import importlib
import pandas as pd

for m in ["textutil", "lexicon", "template", "dictation", "splitting", "routing",
          "editor", "impression", "pipeline", "validate", "metrics"]:
    importlib.import_module(f"rrh.{m}")
    importlib.reload(sys.modules[f"rrh.{m}"])

from rrh.pipeline import Config, ReportGenerator
from rrh.routing import fit_router, fit_ranked_router
from rrh.validate import validate

train = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
test = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(INPUT_DIR, "sample_submission.csv"))
print(train.shape, test.shape, sample.shape)

CONFIG = json.loads("""
{
  "abnormal_first": true,
  "add_copula": false,
  "add_existential": false,
  "allow_conjunction_split": false,
  "allow_splitting": true,
  "append_template_impression": true,
  "blank_between_fields": true,
  "correct_spelling": true,
  "cover_threshold": 0.05,
  "cue_match_min": 0.34,
  "cue_veto_penalty": 1.4,
  "dedupe_threshold": 0.7,
  "drop_negative_impression": false,
  "findings_cap": 2,
  "findings_require_abnormal": true,
  "group_penalty": 1.0,
  "impression_dedupe": 0.95,
  "keep_unrouted": true,
  "min_route_score": 0.2,
  "mine_threshold": 0.6,
  "normalize_shorthand": true,
  "number_impression": true,
  "rank_impression_by_severity": false,
  "recover_summary": false,
  "region_weight": 0.0,
  "soften_blanket": false,
  "split_margin": -0.3,
  "style_threshold": 0.0,
  "summary_after_cues": false,
  "summary_cap": 8,
  "summary_max_misses": 2,
  "summary_recover_threshold": 0.6,
  "summary_threshold": 0.23,
  "suppress_redundant_negatives": false,
  "suppress_redundant_normals": false,
  "trim_detail": true,
  "trim_summary_detail": false,
  "use_bigrams": false,
  "use_impression_chooser": false,
  "w_backward": 1.0,
  "w_continuity": 0.3,
  "w_cue": 2.0,
  "w_template": 1.6
}
""")
cfg = Config(**CONFIG)
generator = ReportGenerator(fit_ranked_router(train.to_dict("records"), cfg), cfg)
print(json.dumps(CONFIG, indent=2, sort_keys=True))


## 4. Cross-validated offline score

RES is not published, so we track a family of edit-based proxies (word/char normalised edit distance to the reference, plus field-level fidelity, content recall and precision). Lower is better.

In [ ]:
from rrh.metrics import evaluate
from rrh.pipeline import ReportGenerator

# Honest cross-validation: the routing statistics are re-mined per fold so no
# fold ever sees its own reference report.
K = 5
rows = train.to_dict("records")
folds = [rows[i::K] for i in range(K)]
preds, gold = [], []
for f in range(K):
    tr_rows = [r for j in range(K) if j != f for r in folds[j]]
    g = ReportGenerator(fit_ranked_router(tr_rows, cfg), cfg)
    for r in folds[f]:
        preds.append(g.generate(r)[0])
        gold.append(r["report"])

from rrh.template import parse_template, render_report
baseline = []
for r in rows:
    t = parse_template(r["template_content"])
    baseline.append(render_report([(x.label, x.text) for x in t.fields], t.impression))

print("copy-the-template baseline :", evaluate(baseline, [r["report"] for r in rows]).as_row())
print("structured pipeline (5-CV) :", evaluate(preds, gold).as_row())


## 5. Stage 1 - deterministic drafts for the test set

In [ ]:
drafts, issue_counts = {}, {}
traces = {}
for row in test.to_dict("records"):
    report, trace = generator.generate(row)
    drafts[row["case_id"]] = report.strip()
    traces[row["case_id"]] = trace
    for k, v in validate(row, report, trace, vocab=generator.model.vocab).counts().items():
        issue_counts[k] = issue_counts.get(k, 0) + v

print("deterministic drafts:", len(drafts))
print("validation issues   :", json.dumps(issue_counts, sort_keys=True) or "none")
print()
print(drafts[sample["case_id"][0]])


## 6. Stage 2 - LLM refinement

One fixed instruction set, applied uniformly to all 132 cases. The key is read from an environment variable / Kaggle Secret and never stored here; with no key the cached outputs written below are used, so the notebook reproduces `submission.csv` byte-for-byte offline.

In [ ]:
# The key is never stored in the notebook: Kaggle -> Add-ons -> Secrets, or an
# environment variable. With no key the cached refinements below are used, so
# this notebook reproduces submission.csv exactly, offline.
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
USE_LLM = bool(API_KEY) and os.environ.get("RRH_RUN_LLM") == "1"
print("LLM refinement stage:", "live" if USE_LLM else "cached (no API key present)")


### 6.1 The instruction set and the refinement call

In [ ]:
SYSTEM_PROMPT = """You are a precision report editor working from a normal
template, a radiologist's telegraphic dictation, and a deterministic draft built by
editing that template. Return the corrected report and nothing else.

Apply exactly these rules to every case:
1.  Keep the template's field labels, uppercased, in the template's order, one
    blank line between fields. Never add, drop, rename or reorder a label.
2.  A field the dictation does not mention keeps the template text verbatim.
3.  Route each dictated finding to the field it belongs to. Move a clause the
    draft filed under the wrong label (a PCL finding under ANTERIOR CRUCIATE
    LIGAMENT, a TFCC finding under ULNAR nerve, an impression sentence left
    inside a findings field).
4.  Replace or trim only the normal statement the dictation contradicts; keep the
    uncontradicted part of that sentence. If a negated list loses one member
    ("No A or B" where A is now positive), rewrite it as "No B". A template
    sentence the dictation never mentions stays (references keep 92% of those),
    and a field the reference edits still keeps 57% of its template sentences -
    delete only on a real contradiction.
4a. Never write a shorter form than the template or the dictation already gives
    you. Where the dictation confirms a structure is normal without adding
    anything, keep the template's sentence rather than compressing it to
    "Intact." - reference field text matches the dictation's length (median
    difference 0 words), it does not shrink below both sources.
5.  Drop technique, clinical history, contrast dose, comparison and
    recommendation boilerplate, and section headers that leaked in from the
    dictation's own layout ("Findings", "Impression", "Brain Parenchyma").
    Keep statements about study limitations ("Evaluation is limited by metallic
    artifacts") - references keep 85% of those.
6.  Repair dictation typos and expand shorthand into standard radiology terms
    ("degen chnges" -> "Degenerative changes", "s/o" -> "suggestive of",
    "VR spaces" -> "Virchow-Robin spaces"). Do not repair a term into a
    different entity.
7.  IMPRESSION: reuse the dictated summary when the dictation has one, in its
    order and near-verbatim (reference impressions run 1.07x the summary's
    length - do not aggressively trim it); otherwise condense the abnormal
    findings by removal only, to about 6 words an item. Abnormal items first,
    the closing negative last (references put it last 325 times against 34
    first), numbered when there is more than one item and plain when there is
    one. Use the template's impression line only when nothing is abnormal, and
    never next to a finding it contradicts.
8.  Preserve negation, laterality and every measurement exactly as dictated.
9.  Never introduce a finding, diagnosis, measurement or laterality that is not
    supported by the dictation and the template.
10. Output only FINDINGS: ... IMPRESSION: ... - nothing else."""


def refine(row, draft):
    """One refinement call. Returns the draft unchanged when no key is present."""
    if not USE_LLM:
        return draft
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    msg = client.messages.create(
        model="claude-opus-5",
        max_tokens=3000,
        temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content":
                   f"STUDY: {row.get('modality')} {row.get('body_part')} - {row.get('study_description')}\n\n"
                   f"TEMPLATE:\n{row['template_content']}\n\n"
                   f"DICTATION:\n{row['dictation']}\n\n"
                   f"DRAFT:\n{draft}"}],
    )
    return msg.content[0].text.strip()


### 6.2 Cached refinement outputs

The 132 reports produced by the stage above, so the notebook runs without a key. Supplying one regenerates and overwrites this file.

In [ ]:
%%writefile refined_reports.json
{
 "00320399-b3ed-447c-be07-a3e3b9af6e2e": "FINDINGS:\nLIVER: Normal in size and attenuation. No focal hepatic lesion.\n\nGALLBLADDER AND BILIARY TREE: Surgically absent gallbladder. Surgical clips are seen in the gallbladder fossa. No biliary ductal dilatation.\n\nPANCREAS: Normal in size and contour.\n\nSPLEEN: Small splenic calcification.\n\nADRENAL GLANDS: Normal in size and morphology.\n\nKIDNEYS AND URETERS: Normal appearance without renal calculi or hydronephrosis.\n\nURINARY BLADDER: Normal. No calculus or mass lesion.\n\nREPRODUCTIVE: The uterus and ovaries/prostate are normal in appearance.\n\nMAJOR VESSELS: The abdominal aorta is normal in caliber.\n\nPERITONEUM: No adjacent fluid collection or abscess is identified. No extraluminal free air is seen. No ascites. No pneumoperitoneum.\n\nABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.\n\nLYMPH NODES: Pelvic phleboliths. No significant mesenteric or retroperitoneal lymphadenopathy.\n\nSTOMACH AND BOWEL: Appendix is surgically absent, with surgical suture at the cecum. Colonic diverticulosis is present. There is focal moderate pericolonic fat stranding adjacent to the sigmoid colon, best seen on series 201 images 180 to 185, consistent with acute diverticulitis. Enteric contrast is seen from stomach to distal small bowel. No bowel obstruction.\n\nBONES: No acute osseous abnormality.\n\nIMPRESSION:\n1. Acute sigmoid diverticulitis with focal moderate pericolonic fat stranding, without adjacent fluid collection, abscess or extraluminal free air.\n2. Colonic diverticulosis.\n3. Surgically absent gallbladder and appendix.\n4. Small splenic calcification and pelvic phleboliths.",
 "01b1c9ab-800b-4963-b11d-9d503952e57c": "FINDINGS:\nVERTEBRAE: Mild lower lumbar facet degenerative changes. Normal desnity and alignment. No fracture or osseous lesion.\n\nDISC SPACES: Preserved.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Mild lower lumbar facet degenerative changes.\n2. No acute abnormality evident in the lumbar spine.",
 "02aae38a-088b-4cae-bfdc-f0fb38da565d": "FINDINGS:\nVERTEBRAE: Degenerative changes. Mild scoliosis with convexity to the left. No fracture or osseous lesion.\n\nDISC SPACES: Preserved.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Degenerative changes of the thoracic spine.\n2. Mild levoconvex scoliosis.\n3. No acute abnormality evident in the thoracic spine.",
 "045ea333-6d30-4edf-9d33-8ed3feb0772d": "FINDINGS:\nVERTEBRAE: A benign-appearing vertebral hemangioma is noted within the L2 vertebral body. Mild degenerative endplate changes are seen. No acute fracture or focal osseous lesion. Marrow signal is within normal limits for age.\n\nALIGNMENT: Straightening of the normal lumbar lordosis is noted. No spondylolisthesis.\n\nSPINAL CORD: No significant focal compression of the cauda equina is seen. The conus medullaris terminates at a normal level, demonstrating normal signal and contour.\n\nDISCS/DEGENERATIVE CHANGES: Multilevel degenerative spondylotic changes are seen in the form of marginal osteophyte formation and multilevel intervertebral disc desiccation/bulging.\n\nL1-L2: Mild diffuse disc bulge is noted without significant central canal stenosis or neural foraminal compromise.\n\nL2-L3: Diffuse disc bulge is noted with bilateral neural foraminal narrowing.\n\nL3-L4: Diffuse disc bulge with bilateral foraminal extension is noted, causing bilateral neural foraminal narrowing with crowding/impingement of the exiting nerve roots.\n\nL4-L5: Diffuse disc bulge with bilateral neural foraminal narrowing is noted. The AP diameter of the central spinal canal measures approximately 9.6 mm, suggestive of mild central canal stenosis.\n\nL5-S1: Diffuse disc bulge with bilateral foraminal extension is noted, causing bilateral neural foraminal narrowing and impingement of the exiting nerve roots. The central canal measures approximately 12 mm in AP diameter.\n\nPARASPINAL SOFT TISSUES: Focal marrow/soft-tissue edema-like signal involving the anterior adjacent endplate at L2-L3 and the spinous process of L4. Visualized paraspinal soft tissues are otherwise grossly unremarkable.\n\nIMPRESSION:\n1. Multilevel lumbar spondylotic degenerative changes with multilevel disc bulges.\n2. L3-L4 bilateral neural foraminal narrowing with impingement of bilateral exiting nerve roots.\n3. L4-L5 disc bulge with mild central canal stenosis (canal AP diameter approximately 9.6 mm).\n4. L5-S1 bilateral foraminal narrowing with impingement of bilateral exiting nerve roots.\n5. Focal marrow/soft-tissue edema-like signal involving the anterior adjacent endplate at L2-L3 and the spinous process of L4.\n6. Vertebral hemangioma in L2.",
 "06ef946f-5b3b-4b7c-a64f-58c184dca4e6": "FINDINGS:\nBONES: Mild diffuse osseous demineralization. No acute fracture or dislocation.\n\nJOINTS: Mild degenerative changes of the acromioclavicular joint. Glenohumeral alignment is maintained. Glenohumeral joint space is relatively maintained.\n\nSOFT TISSUES: No focal soft tissue abnormality or radiopaque foreign body identified.\n\nIMPRESSION:\n1. Mild acromioclavicular osteoarthritis.\n2. No acute osseous abnormality of the left shoulder.",
 "06f6de2d-bf9e-42fa-9ae0-ba07e80fb272": "FINDINGS:\nVERTEBRAE: Thoracic vertebral body heights are maintained. No acute compression fracture or significant marrow abnormality is identified.\n\nALIGNMENT: Exaggerated thoracic kyphotic curvature is noted.\n\nSPINAL CORD: The thoracic spinal cord is normal in caliber and morphology. No focal cord indentation or compression is seen. No abnormal intramedullary T2/STIR hyperintense signal or cord edema is noted.\n\nDISCS/DEGENERATIVE CHANGES: Multilevel degenerative spondylotic changes are seen with disc desiccation and small marginal osteophyte formation. No significant focal disc bulge, disc extrusion or significant posterior disc protrusion is identified. No significant spinal canal stenosis or neural foraminal compromise is noted.\n\nPARASPINAL SOFT TISSUES: No significant prevertebral or paravertebral soft-tissue abnormality is evident.\n\nIMPRESSION:\n1. Exaggerated thoracic kyphosis.\n2. Multilevel mild thoracic spondylotic degenerative changes with small marginal osteophytes.\n3. No significant disc protrusion, spinal canal stenosis or neural foraminal compromise.\n4. No cord compression or abnormal cord signal changes.",
 "080b9ad5-a25a-4de6-b52c-d1464ea43e93": "FINDINGS:\nBONES: Possible fracture of the left lesser trochanter. Orthopedic hardware is seen along the diaphysis of the left femur.\n\nJOINTS: Bilateral hip joint prostheses are present, consistent with bilateral total hip replacement. No dislocation.\n\nSOFT TISSUES: Calcification is noted in the left thigh.\n\nIMPRESSION:\n1. Bilateral total hip replacement.\n2. Possible fracture of the left lesser trochanter.\n3. Orthopedic hardware along the diaphysis of the left femur.\n4. Calcification in the left thigh.",
 "09293ce5-df84-44b4-9882-b8cddd1a1ac1": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Degenerative changes of the left hip joint in the form of mild superolateral joint space narrowing, subchondral/endplate sclerosis and marginal osteophyte formation. Left sacroiliac joint demonstrates degenerative arthropathy with subchondral sclerosis and mild joint space irregularity. No dislocation. Alignment is maintained.\n\nSOFT TISSUES: Visualized soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild left hip osteoarthritic changes.\n2. Left sacroiliac joint arthropathy.\n3. No acute osseous abnormality.",
 "0b2c0c30-d4e5-4dcb-8d13-1f46e2644979": "FINDINGS:\nVERTEBRAE: Altered marrow signal intensity is noted involving the T7, T8 and T9 vertebral bodies, appearing hyperintense on T1- and T2-weighted images and showing hyperintensity on STIR sequences, with predominant involvement of the T8 vertebral body. The appearance is suggestive of vertebral hemangiomatous changes, particularly at T7 and T8, with predominant involvement of T8. Mild STIR hyperintense signal is also seen along the inferior endplate of T8 and superior endplate of T9, with mild adjacent endplate irregularity. These changes are likely reactive/degenerative in nature and may represent stress-related or repetitive traumatic changes. No significant vertebral body collapse or retropulsion is seen.\n\nALIGNMENT: There is straightening of the cervical spine.\n\nSPINAL CORD: No significant abnormality of the thoracic spinal cord is identified. The visualized cord demonstrates preserved caliber and signal intensity. No significant cervical spinal cord indentation is evident on the provided images.\n\nDISCS/DEGENERATIVE CHANGES: Multilevel mild degenerative spondylotic changes are seen in the dorsal spine with marginal osteophyte formation and associated disc desiccation. No significant focal disc protrusion or extrusion is seen in the dorsal spine. There is no significant central canal stenosis or neural foraminal compromise. Multilevel degenerative disc changes and diffuse disc bulges are noted in the cervical spine, causing indentation of the ventral thecal sac.\n\nPARASPINAL SOFT TISSUES: No significant paravertebral or epidural soft-tissue component is evident.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. T7-T9 vertebral body marrow signal alterations, predominantly involving T8, with T1/T2 and STIR hyperintensity, favoring benign vertebral hemangiomatous changes.\n2. Mild reactive/endplate marrow edema-like signal along the T8-T9 disc space, with mild endplate irregularity, likely degenerative/stress-related.\n3. Multilevel dorsal spondylotic degenerative changes with osteophyte formation and disc desiccation.\n4. No significant dorsal spinal canal stenosis or neural compression.\n5. No significant thoracic cord signal abnormality.",
 "0c89b0eb-2ad2-4f31-8f80-f2bb4cd8679a": "FINDINGS:\nLOWER THORAX: No pleural effusion or basilar consolidation.\n\nLIVER: Liver is mildly enlarged, measuring approximately 16 cm in craniocaudal span. No focal hepatic lesion is identified on the provided sequences.\n\nGALLBLADDER AND BILE DUCTS: Gallbladder is not visualized, consistent with postoperative status (cholecystectomy). Common bile duct is normal in calibre with no evidence of intraluminal filling defect or obstructive dilatation. Intrahepatic biliary radicles are not dilated.\n\nPANCREAS: Pancreas is normal in size and signal intensity. Main pancreatic duct is not dilated. No focal pancreatic lesion is seen.\n\nSPLEEN: Spleen is mildly enlarged, measuring approximately 12.2 cm.\n\nADRENALS: The adrenal glands are normal in size and configuration.\n\nKIDNEYS: Normal in size, position, and signal intensity. No hydronephrosis or suspicious renal mass.\n\nSTOMACH AND BOWEL: The visualized portions of the stomach and bowel are unremarkable. No bowel wall thickening or obstruction.\n\nLYMPH NODES: No significant free fluid or obvious upper abdominal lymphadenopathy is seen.\n\nVASCULATURE: The abdominal aorta and major visceral vessels are patent. No evidence of aneurysm or dissection.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Post-cholecystectomy status. No biliary dilatation; CBD is normal in calibre. No choledocholithiasis.\n2. Mild hepatomegaly.\n3. Mild splenomegaly.\n4. Pancreas and main pancreatic duct are unremarkable.",
 "105e2008-9924-4bff-9a63-de279e451878": "FINDINGS:\nVERTEBRAE: Mild dextroconvex scoliosis of the mid thoracic spine. Mild chronic-appearing vertebral body height loss involving the upper to mid thoracic spine. No acute fracture or subluxation.\n\nDISC SPACES: Mild multilevel disc space narrowing with small marginal osteophytes.\n\nSOFT TISSUES: The visualized paraspinal soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild multilevel thoracic spondylosis.\n2. Mild dextroconvex scoliosis of the mid thoracic spine.\n3. Mild chronic-appearing vertebral body height loss in the upper to mid thoracic spine.\n4. No acute fracture or subluxation.",
 "10936c93-1fbc-49e8-a51e-580807955fb8": "FINDINGS:\nINTERNAL CAROTID ARTERIES: Multifocal atherosclerotic plaques are noted involving the proximal right internal carotid artery, just distal to the carotid bifurcation, producing moderate luminal narrowing, approximately 35-40%. Focal eccentric atherosclerotic plaque/irregularity is noted involving the mid left internal carotid artery, without significant luminal stenosis or appreciable reduction in caliber.\n\nCOMMON CAROTID ARTERIES: Patent bilaterally. No significant stenosis, dissection, or high-grade plaque.\n\nEXTERNAL CAROTID ARTERIES: Patent bilaterally with normal branching patterns. No significant stenosis or dissection.\n\nVERTEBRAL ARTERIES: The right vertebral artery is hypoplastic. Patent bilaterally from their origins to the vertebrobasilar junction. No significant stenosis, dissection, or flow-limiting lesion.\n\nBASILAR ARTERY: Patent and demonstrates normal caliber. No significant stenosis or dissection.\n\nSOFT TISSUES: The visualized soft tissues of the neck are symmetric and without evidence of mass or abnormal fluid collection.\n\nOTHER FINDINGS:\n\nThe remaining visualized cervical arterial segments show maintained luminal caliber without significant stenosis or occlusion.\n\nEvaluation is mildly limited by multiple artifacts.\n\nIMPRESSION:\n1. Multifocal atherosclerotic plaques in the proximal right ICA with approximately 35-40% luminal narrowing.\n2. Hypoplastic right vertebral artery.\n3. Focal plaque/irregularity in the mid left ICA without significant stenosis.\n4. No significant arterial occlusion.",
 "11c886c3-f3ea-4f49-8253-026fdc431bf5": "FINDINGS:\nBONES: No acute fracture is identified. No focal osseous destructive lesion is evident.\n\nJOINTS: Multiple degenerative changes are noted in the wrist with multifocal reduction of joint spaces, associated with marginal osteophyte formation and subchondral sclerosis. Degenerative changes are more pronounced at the 1st carpometacarpal (CMC) joint, with appreciable joint-space narrowing and osteophyte formation. Mild degenerative changes are also seen at the remaining visualized intercarpal and carpometacarpal articulations. No dislocation is identified.\n\nSOFT TISSUES: Soft tissues are grossly unremarkable on radiograph.\n\nIMPRESSION:\n1. Multifocal osteoarthritic/degenerative changes of the wrist, with multiple reduced joint spaces.\n2. More pronounced osteoarthritis at the 1st carpometacarpal joint.\n3. No acute osseous abnormality.",
 "12868962-9d03-46f0-8386-538365e5f561": "FINDINGS:\nALIGNMENT: Cervical spinal alignment is maintained. Normal cervical lordosis.\n\nVERTEBRAL BODIES: Multilevel degenerative spondylotic changes are noted involving the cervical spine, predominantly from C3 to C7, with associated disc space reduction and osteophyte/disc complex formation. No gross focal abnormality is identified in the visualized posterior elements.\n\nSPINAL CORD: Mild cord signal hyperintensity/edema is noted adjacent to the C3-C4 level. The cord is otherwise normal in signal and caliber.\n\nC2-3: No disc herniation or spinal canal stenosis. No neural foraminal narrowing.\n\nC3-4: Posterior disc-osteophyte complex is seen indenting the ventral thecal sac and ventral aspect of the spinal cord. Mild adjacent cord signal hyperintensity/edema is noted. The AP spinal canal diameter measures approximately 7.4 mm, suggestive of significant canal stenosis.\n\nC4-5: Reduced disc space with posterior disc herniation/disc-osteophyte complex is noted, causing ventral thecal sac indentation and right neural foraminal narrowing with compression of the exiting right nerve root.\n\nC5-6: Marked reduction of disc space with posterior disc protrusion/disc-osteophyte complex is noted, causing indentation of the ventral thecal sac and bilateral neural foraminal narrowing with compression of the exiting bilateral nerve roots. The AP spinal canal diameter measures approximately 7.7 mm, suggestive of significant canal stenosis.\n\nC6-7: Reduced/obliterated disc space with associated degenerative changes is noted.\n\nC7-T1: No disc herniation or spinal canal stenosis. No neural foraminal narrowing.\n\nPARAVERTEBRAL SOFT TISSUES: Prevertebral soft tissues are unremarkable.\n\nDetailed evaluation is limited by metallic artifacts.\n\nIMPRESSION:\n1. Multilevel cervical spondylosis with significant degenerative disc disease, predominantly from C3-C4 to C6-C7.\n2. C3-C4 posterior disc-osteophyte complex causing significant canal stenosis (AP diameter approximately 7.4 mm), with ventral cord indentation and mild adjacent cord edema/signal alteration.\n3. C4-C5 disc herniation/disc-osteophyte complex with right foraminal narrowing and compression of the exiting right nerve root.\n4. C5-C6 posterior disc protrusion with significant canal stenosis (AP diameter approximately 7.7 mm) and bilateral foraminal narrowing with compression of the exiting nerve roots.\n5. Marked disc space reduction/obliteration at C4-C5, C5-C6 and C6-C7.",
 "17439506-c1c2-4f1b-aa4f-5128681b9b15": "FINDINGS:\nVERTEBRAE: Small multilevel marginal osteophytes are present. Moderate facet joint arthropathy at L5-S1. Mild to moderate osteoarthritis of both hips and sacroiliac joints. Vertebral body heights and alignment are maintained. No fracture or listhesis.\n\nDISC SPACES: Severe disc space narrowing at L5-S1. Moderate disc space narrowing at L4-L5 and from T12-L1 through L2-L3.\n\nSOFT TISSUES: Atherosclerotic calcification of the abdominal aorta.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Multilevel lumbar spondylosis, with severe degenerative disc disease at L5-S1 and moderate degenerative disc disease at L4-L5 and from T12-L1 through L2-L3.\n2. Moderate L5-S1 facet arthropathy.\n3. Mild to moderate bilateral hip and sacroiliac joint osteoarthritis.\n4. Atherosclerotic calcification of the abdominal aorta.\n5. No fracture or listhesis.",
 "196725dd-aeab-462d-bad5-6e9c8e7cd764": "FINDINGS:\nVERTEBRAE: Degenerative changes with osteophytes. Normal density and alignment. No fracture or osseous lesion.\n\nDISC SPACES: Reduced disc space at L5-S1.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Degenerative changes with osteophytes.\n2. Reduced disc space at L5-S1.\n3. No acute abnormality evident in the lumbar spine.",
 "196b8d20-65fe-4f3d-9e17-1504509a88e8": "FINDINGS:\nBONES: No fracture or focal lesion.\n\nJOINTS: Decreased acromiohumeral/acromial space, which may suggest rotator cuff insufficiency. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Decreased acromiohumeral space, which may suggest rotator cuff insufficiency.\n2. No acute osseous abnormality.",
 "198a07f6-1166-4cfd-81be-e76d63b78606": "FINDINGS:\nOSSEOUS STRUCTURES: Postsurgical changes of prior ACL reconstruction are noted, with postoperative tunnel-related changes involving the lateral femoral condyle and proximal tibia. No acute osseous abnormality.\n\nJOINT: Knee joint alignment is maintained. No significant joint effusion.\n\nMENISCI:\n\nMEDIAL MENISCUS: Complex full-thickness tear involving the posterior horn of the medial meniscus, extending to both the superior and inferior articular surfaces. Associated edema/altered signal intensity is noted involving the posterior horn and body of the medial meniscus.\n\nLATERAL MENISCUS: Heterogeneous altered intrameniscal signal with mild bulky appearance of the anterior horn. Posterior horn and body of the lateral meniscus are otherwise maintained.\n\nLIGAMENTS:\n\nANTERIOR CRUCIATE LIGAMENT: Postoperative. Assessment of graft integrity should be correlated with its morphology and continuity.\n\nPOSTERIOR CRUCIATE LIGAMENT: Intact with normal morphology and signal intensity.\n\nMEDIAL COLLATERAL LIGAMENT: Intact.\n\nLATERAL COLLATERAL LIGAMENT COMPLEX: Intact.\n\nARTICULAR CARTILAGE: Cartilage thickness and signal are preserved.\n\nEXTENSOR MECHANISM: Quadriceps and patellar tendons are intact.\n\nBURSAE: No significant bursitis.\n\nMUSCULATURE: Normal bulk and signal.\n\nSOFT TISSUES: No mass or fluid collection.\n\nIMPRESSION:\n1. Complex full-thickness tear of the posterior horn of the medial meniscus, reaching both superior and inferior articular surfaces.\n2. Associated marrow/soft-tissue edema-like signal involving the posterior horn and body of the medial meniscus.\n3. Postsurgical changes of ACL reconstruction with femoral and tibial tunnel-related changes; graft integrity to be assessed on dedicated sagittal/oblique sequences.\n4. Heterogeneous altered signal and mild bulky appearance of the anterior horn of the lateral meniscus, suspicious for meniscal degeneration/intrasubstance injury.\n5. PCL is intact.",
 "1ab020d7-5b7a-4e9e-9d76-38b670566ce4": "FINDINGS:\nBONES: Subchondral cystic/sclerotic changes are noted along the opposing articular surfaces. No acute fracture is identified. No focal destructive osseous lesion is seen.\n\nJOINTS: Severe tricompartmental degenerative osteoarthritic changes are noted, with marked medial compartment predominant joint-space narrowing, prominent marginal osteophyte formation and associated subchondral sclerosis. Degenerative hypertrophic changes are also seen involving the lateral and patellofemoral compartments, with prominent tibial spine and patellar osteophytes. Few well-defined ossific densities are seen within the joint space, compatible with intra-articular loose bodies. One of the loose bodies is seen in the suprapatellar region. Mild joint effusion may be present. No dislocation is identified.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Severe tricompartmental osteoarthritis, predominantly medial compartment, with marked joint-space narrowing, prominent osteophytes and subchondral sclerosis.\n2. Intra-articular ossific loose bodies.\n3. No acute osseous abnormality.",
 "1b903418-bbc6-43ff-a549-d5bd266ec023": "FINDINGS:\nBLADDER: Post-void urinary bladder volume measures 121.3 mL, calculated from bladder dimensions of approximately 9.03 \u00d7 6.57 \u00d7 3.90 cm (length \u00d7 width \u00d7 AP diameter). Significant residual urine is noted following voiding. No intraluminal mass, calculus, or debris is identified.\n\nIMPRESSION:\nSignificant post-void residual urinary bladder volume of approximately 121 mL, suggestive of incomplete bladder emptying.",
 "1c30fb15-29cc-4bad-a901-0a65cd45693b": "FINDINGS:\nLIVER: Liver is mildly enlarged, measuring approximately 17 cm in craniocaudal span. Few small simple hepatic cysts are noted. No definite solid focal hepatic lesion is identified.\n\nGALLBLADDER: Gallbladder is unremarkable. No obvious calculus or wall thickening.\n\nBILIARY TREE: Common bile duct is normal in caliber. No filling defect or stricture.\n\nPANCREATIC DUCT: Normal in caliber. No filling defect.\n\nPANCREAS: Pancreas is normal in size and morphology. No focal pancreatic mass or other definite abnormality identified.\n\nSPLEEN: Spleen is normal in size and signal intensity. No focal splenic lesion.\n\nKIDNEYS: Right kidney is ectopic (pelvic in location) and malrotated. Few simple cortical cysts are seen in both kidneys. No definite solid renal mass.\n\nIMPRESSION:\n1. Mild hepatomegaly, liver measuring approximately 17 cm.\n2. Few simple hepatic cysts.\n3. Ectopic and malrotated right kidney.\n4. Few simple bilateral renal cortical cysts.\n5. No significant abnormality of the gallbladder, CBD, spleen or pancreas.",
 "1cf88557-17c5-454a-bb1e-0435040ec676": "FINDINGS:\nBONES: A small, well-corticated osseous fragment is noted adjacent to the proximal interphalangeal joint of the third finger, likely representing an accessory ossicle. No acute fracture is seen.\n\nJOINTS: No dislocation is seen. Joint spaces are maintained. Alignment is preserved.\n\nSOFT TISSUES: Surrounding soft tissues are unremarkable.\n\nIMPRESSION:\n1. Small well-corticated accessory ossicle adjacent to the PIP joint of the third finger.\n2. No acute osseous abnormality.",
 "1db5fa23-cb88-4d4f-9251-8adba6ae0d65": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Early osteophytes are noted. No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Early osteophytes.\n2. No acute osseous abnormality.",
 "24c20e73-6a0e-40b7-a087-b43c80550e61": "FINDINGS:\nVERTEBRAE: There is diffuse osseous demineralization. Mild dextroconvex thoracolumbar curvature is present. Vertebral body heights are maintained without acute compression fracture. Sacroiliac joints are maintained. No fracture or osseous lesion.\n\nDISC SPACES: There is multilevel degenerative disc space narrowing and endplate osteophyte formation, greatest at L4-L5, with lower lumbar facet arthropathy.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Multilevel degenerative disc disease and lower lumbar facet arthropathy, greatest at L4-L5.\n2. Diffuse osseous demineralization.\n3. Mild dextroconvex thoracolumbar curvature.\n4. No acute abnormality evident in the lumbar spine.",
 "25173dfe-2de4-448d-ad6c-1fd39e90ac32": "FINDINGS:\nBONES: Subacute to chronic fracture deformity of the proximal fibula. No acute fracture.\n\nJOINTS: Moderate medial and lateral compartment joint space narrowing with small marginal osteophytes and tibial plateau subchondral sclerosis. Moderate patellofemoral joint space narrowing with subchondral sclerosis. Small suprapatellar joint effusion. No dislocation.\n\nSOFT TISSUES: Atherosclerotic calcifications of the visualized arteries are present. Several small linear radiopaque densities project over the soft tissues of the leg, which may represent surgical clips or retained foreign bodies.\n\nIMPRESSION:\n1. Moderate tricompartmental osteoarthritis.\n2. Subacute to chronic fracture deformity of the proximal fibula.\n3. Small suprapatellar joint effusion.\n4. Atherosclerotic vascular calcifications.\n5. Small linear radiopaque densities within the leg soft tissues, possibly surgical clips or retained foreign bodies.\n6. No acute osseous abnormality.",
 "253c32fe-ce5f-4e4c-86c6-bea1e11db610": "FINDINGS:\nBONES: Calcaneal spur is noted, likely at the plantar aspect of the calcaneus. No acute fracture is evident. No focal acute osseous abnormality identified.\n\nJOINTS: Degenerative osteoarthritic changes are noted in the visualized joints. Multiple joint spaces are reduced, with associated marginal osteophyte formation and subchondral degenerative changes. No dislocation is evident.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Multicompartmental degenerative osteoarthritic changes with multiple reduced joint spaces.\n2. Calcaneal spur.\n3. No acute osseous abnormality.",
 "264a47c5-2b2e-44c6-a38a-bcf692ab5c0e": "FINDINGS:\nThere is straightening of the cervical spine. There is normal alignment, without subluxation or spondylolisthesis.\n\nThe vertebral body and disc space heights are preserved. No compression deformities.\n\nAnterior fusion and discectomy status at C5-6, with placement of disc graft, anterior plate and interbody screws. No evidence of loosening or hardware abnormality. Disc prosthesis appears well seated without slippage.\n\nNo acute fracture is identified.\n\nThe odontoid process is intact.\n\nNo lytic or blastic lesions are seen.\n\nThe facets are normally aligned without degenerative change.\n\nThe paravertebral soft tissues are normal. No prevertebral soft tissue abnormalities. The airway appears normal. The visualized lung apices are clear.\n\nIMPRESSION:\n1. C5-6 anterior plate and screw fusion with disc prosthesis. No loosening or complication evident.\n2. Straightening of the cervical spine.",
 "268db61b-f6a7-4a5f-96d7-9fe250b3cee5": "FINDINGS:\nLIGAMENTS:\n\nANTERIOR TALOFIBULAR: Intact and normal in course and signal intensity.\n\nPOSTERIOR TALOFIBULAR: Intact and normal in course and signal intensity.\n\nANTERIOR TIBIOFIBULAR: Intact and normal in course and signal intensity.\n\nPOSTERIOR TIBIOFIBULAR: Intact and normal in course and signal intensity.\n\nCALCANEOFIBULAR: Intact and normal in course and signal intensity.\n\nDELTOID: The deltoid ligament complex appears intact.\n\nSPRING: The spring ligament complex appears intact.\n\nLISFRANC: The Lisfranc ligament complex is intact.\n\nTENDONS:\n\nACHILLES: The Achilles tendon appears mildly thickened and hyperintense, consistent with tendinosis. Tendon continuity is maintained.\n\nPOSTERIOR TIBIAL: Fluid is present within the tendon sheath of the tibialis posterior tendon, consistent with tenosynovitis. No tendon tear is seen.\n\nFLEXOR: Fluid is present within the tendon sheaths of the flexor digitorum longus and flexor hallucis longus tendons, consistent with tenosynovitis. No tendon tear is seen.\n\nEXTENSOR: The anterior extensor tendons are intact.\n\nPERONEAL: The peroneus longus and brevis tendons demonstrate increased intrasubstance signal with mild surrounding tendon-sheath fluid, consistent with tendinosis and tenosynovitis. No discrete tear is identified.\n\nBONES: Subtle subchondral marrow oedema is seen in the talus. No definite fracture line, articular-surface collapse or osteochondral defect is identified. The ankle mortise and joint alignment are maintained.\n\nMUSCLES: Normal bulk and signal intensity without evidence of tear, edema, or atrophy.\n\nFLUID: Mild tibiotalar joint effusion is present. The subtalar joint appears unremarkable. No focal soft-tissue mass or collection is identified.\n\nSINUS TARSI: The sinus tarsi appears unremarkable.\n\nTARSAL TUNNEL: The contents are normal in appearance without evidence of mass or nerve entrapment.\n\nPLANTAR FASCIA: The plantar fascia appears unremarkable.\n\nCARTILAGE: The articular cartilage of the tibiotalar and subtalar joints is preserved in thickness without focal defect.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Peroneus longus and brevis tendinosis with associated tenosynovitis.\n2. Achilles tendinosis without a discrete tear.\n3. Tenosynovitis involving the medial flexor tendons, including the tibialis posterior tendon.\n4. Subtle talar subchondral marrow oedema without a definite fracture or osteochondral defect.\n5. Mild tibiotalar joint effusion.",
 "27b6c8e6-cd7b-4bae-8b38-27dae5227f39": "FINDINGS:\nBONES: Diffuse osseous demineralization. No acute fracture.\n\nJOINTS: Mild degenerative joint space narrowing and marginal spurring are present at the thumb carpometacarpal and triscaphe articulations. Mild radiocarpal degenerative change. No dislocation. Carpal alignment is maintained.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild degenerative changes at the thumb carpometacarpal, triscaphe and radiocarpal articulations.\n2. Diffuse osseous demineralization.\n3. No acute osseous abnormality.",
 "284433c0-f653-4e70-bd6a-db9df082723b": "FINDINGS:\nLIGAMENTS:\n\nANTERIOR TALOFIBULAR: Intact and normal in course and signal intensity.\n\nPOSTERIOR TALOFIBULAR: Intact and normal in course and signal intensity.\n\nANTERIOR TIBIOFIBULAR: Intact and normal in course and signal intensity.\n\nPOSTERIOR TIBIOFIBULAR: Intact and normal in course and signal intensity.\n\nCALCANEOFIBULAR: Intact and normal in course and signal intensity.\n\nDELTOID: The deep and superficial components are intact.\n\nSPRING: The superomedial, medioplantar oblique, and inferoplantar longitudinal components are intact.\n\nLISFRANC: The Lisfranc ligament complex is intact.\n\nTENDONS:\n\nACHILLES: Intact without focal tear.\n\nPOSTERIOR TIBIAL: Intact with normal signal intensity. No tenosynovitis.\n\nFLEXOR: Minimal fluid surrounds the flexor hallucis longus tendon posterior to the talus, compatible with mild tenosynovitis. The remaining visualized flexor tendons are intact without focal tear.\n\nEXTENSOR: The visualized extensor tendons are intact without focal tear.\n\nPERONEAL: The visualized peroneal tendons are intact without focal tear.\n\nBONES: No acute fracture, focal marrow edema, or osteochondral lesion is demonstrated.\n\nMUSCLES: Normal bulk and signal intensity without evidence of tear, edema, or atrophy.\n\nFLUID: Small amount of fluid is present within the posterior tibiotalar and posterior subtalar joints. No focal soft tissue collection or mass is demonstrated.\n\nSINUS TARSI: No sinus tarsi syndrome. The cervical and interosseous talocalcaneal ligaments are intact.\n\nTARSAL TUNNEL: The contents are normal in appearance without evidence of mass or nerve entrapment.\n\nPLANTAR FASCIA: Mild thickening and increased signal involve the central bundle of the plantar fascia at its calcaneal origin, compatible with mild plantar fasciitis. No plantar fascial tear is demonstrated.\n\nCARTILAGE: The articular cartilage of the tibiotalar and subtalar joints is preserved in thickness without focal defect.\n\nOTHER FINDINGS:\n\nNo definite acute ligamentous disruption is demonstrated on the provided images.\n\nIMPRESSION:\n1. Mild flexor hallucis longus tenosynovitis.\n2. Small posterior tibiotalar and posterior subtalar joint effusions.\n3. Mild plantar fasciitis involving the central bundle at the calcaneal origin.\n4. No acute osseous abnormality or focal tendon tear demonstrated.",
 "2a31cff3-12c2-4939-82d1-b40ad463153f": "FINDINGS:\nBONES: Subtle cortical irregularity and lucent fracture line involving the distal radial metaphysis, compatible with an acute nondisplaced/minimally displaced distal radius fracture. No significant angulation or displacement. Mild diffuse osseous demineralization.\n\nJOINTS: Mild multifocal degenerative changes of the radiocarpal, intercarpal and first carpometacarpal joints. No dislocation. Carpal alignment is maintained.\n\nSOFT TISSUES: Mild surrounding soft tissue swelling. No radiopaque foreign body.\n\nIMPRESSION:\n1. Acute nondisplaced/minimally displaced distal radial metaphyseal fracture.\n2. No dislocation or significant angulation.\n3. Mild multifocal wrist osteoarthritis and diffuse osseous demineralization.",
 "2a3448ce-7802-465b-8137-e7e0baefa477": "FINDINGS:\nLIGAMENTS:\n\nMEDIAL COLLATERAL: The deltoid ligament complex is intact.\n\nLATERAL COLLATERAL: The anterior talofibular, calcaneofibular, and posterior talofibular ligaments are intact.\n\nLISFRANC: The Lisfranc ligament is intact. No evidence of malalignment.\n\nTENDONS:\n\nFLEXOR: Visualized flexor tendons are grossly maintained without definite full-thickness tear on the provided images. No focal tendon retraction.\n\nEXTENSOR: Visualized extensor tendons are grossly maintained without definite full-thickness tear on the provided images.\n\nPERONEAL: The peroneus longus and brevis tendons are intact within the peroneal retinaculum without evidence of tenosynovitis.\n\nBONES: Mild degenerative changes involving the visualized forefoot joints. Alignment of the visualized bones is maintained. No definite acute fracture or dislocation identified on the provided images. No focal aggressive osseous lesion or definite abnormal marrow enhancement.\n\nMUSCLES: Mild patchy increased fluid-sensitive signal and enhancement within the intrinsic muscles of the forefoot, compatible with mild muscular strain/edema.\n\nFLUID: No significant joint effusion identified. No discrete rim-enhancing fluid collection or drainable abscess identified. No definite soft-tissue mass.\n\nSINUS TARSI: The sinus tarsi is clear. No inflammatory changes are seen.\n\nTARSAL TUNNEL: The contents of the tarsal tunnel are normal in appearance. No mass or nerve entrapment.\n\nPLANTAR FASCIA: Normal thickness and signal intensity of the plantar fascia at its calcaneal origin and distally.\n\nCARTILAGE: The articular cartilage is preserved in thickness and signal.\n\nOTHER FINDINGS:\n\nMild diffuse subcutaneous soft-tissue edema/enhancement, predominantly along the dorsal aspect of the forefoot. Visualized major ligamentous structures are grossly intact on the available sequences. No definite acute ligamentous disruption identified.\n\nIMPRESSION:\n1. Mild soft-tissue and intrinsic muscular edema/enhancement in the right forefoot, compatible with strain-related changes.\n2. Mild dorsal subcutaneous soft-tissue edema.\n3. No definite acute fracture, dislocation, tendon rupture, or drainable fluid collection identified on the provided images.\n4. No definite abnormal marrow enhancement to suggest osteomyelitis.",
 "2aa77bd3-3109-4712-a727-208b11710ba1": "FINDINGS:\nVERTEBRAE: There is diffuse osseous demineralization. Vertebral body heights are maintained without acute compression fracture or traumatic malalignment. No fracture or osseous lesion.\n\nDISC SPACES: Multilevel thoracic disc space narrowing and endplate osteophyte formation, greatest in the mid to lower thoracic spine.\n\nSOFT TISSUES: Aortic atherosclerotic calcification is present.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Multilevel thoracic disc space narrowing and endplate osteophyte formation, greatest in the mid to lower thoracic spine.\n2. Diffuse osseous demineralization.\n3. Aortic atherosclerotic calcification.\n4. No acute abnormality evident in the thoracic spine.",
 "2baab99b-fea7-4d08-b25e-104098e8f23a": "FINDINGS:\nLUNGS: Lungs are clear. No focal air-space consolidation.\n\nPLEURAL SPACES: No pleural effusion or pneumothorax.\n\nHEART: Cardiac silhouette is within normal size limits.\n\nMEDIASTINUM/HILA: Prominence and tortuosity of the thoracic aortic contour, likely reflecting age-related aortic unfolding/ectasia. No mediastinal widening or hilar enlargement.\n\nDIAPHRAGM: Diaphragms are unremarkable. No subdiaphragmatic free air.\n\nBONES: No acute osseous abnormality identified on this examination.\n\nSOFT TISSUES: No acute soft tissue abnormality identified on the provided images.\n\nIMPRESSION:\nProminent/tortuous thoracic aortic contour, likely chronic aortic ectasia/unfolding.",
 "2bed23d3-bd4c-4e01-a170-ac6a8aa3ad48": "FINDINGS:\nBONES: Mild cortical irregularity of the medial aspect of the third metacarpal head/neck region. No acute fracture or other focal osseous lesion is seen.\n\nJOINTS: No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: Soft tissue swelling of the third and fourth digits.\n\nIMPRESSION:\n1. Mild cortical irregularity of the medial aspect of the third metacarpal.\n2. Soft tissue swelling of the third and fourth digits.\n3. No acute fracture or dislocation.",
 "2cae6c98-7e2a-445c-9483-9e7456ea87bb": "FINDINGS:\nBONES: No fracture is seen. No abnormal periosteal reaction is noted.\n\nJOINTS: Degenerative changes are noted in the form of marginal osteophytes, mild reduction in the medial compartment of the left knee joint with subchondral sclerosis. Alignment is normal. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild left knee osteoarthritis.\n2. No acute osseous abnormality.",
 "2d02f6ad-8a3f-4755-904c-43b0a19f78af": "FINDINGS:\nBONES: No acute fracture is demonstrated. A small well-corticated ossicle posterior to the lateral femoral condyle is compatible with a fabella, a normal anatomic variant.\n\nJOINTS: No dislocation is demonstrated. Joint spaces are maintained without significant degenerative change. No significant knee joint effusion is identified.\n\nSOFT TISSUES: Tiny radiopaque focus is demonstrated within the superficial anterior prepatellar soft tissues, just superior to the patella, nonspecific and possibly representing a small soft-tissue calcification or tiny foreign body.\n\nIMPRESSION:\n1. No acute osseous abnormality of the right knee.\n2. Tiny nonspecific radiopaque focus in the anterior prepatellar soft tissues.\n3. Incidental fabella.",
 "2e234e42-3c2c-4337-a953-7e8d7e29ed95": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\nNo acute osseous abnormality.",
 "2f44b2b6-34c3-4b9b-9026-f8179e24f0af": "FINDINGS:\nORBITS: The globes are symmetric in size and signal intensity. The optic nerves are normal in course and caliber. The extraocular muscles are symmetric and without abnormal enlargement. The retrobulbar fat planes are preserved. No intra- or extraconal mass or inflammatory process is identified.\n\nBONES: The osseous structures of the orbits are intact. No acute fracture or aggressive osseous lesion.\n\nSINUSES: The visualized paranasal sinuses are clear.\n\nEXTRACRANIAL SOFT TISSUES: The visualized extracranial soft tissues are unremarkable.\n\nIMPRESSION:\nUnremarkable orbits.",
 "2f6f8173-a0e6-47f6-bd3f-59c888feed95": "FINDINGS:\nBONES: Stable mild enthesopathic changes at the greater trochanters and iliac wings. Stable lower lumbar fixation rods and transpedicular screws are partially visualized. No acute fracture.\n\nJOINTS: Stable moderate bilateral hip joint space narrowing and subchondral sclerosis, greater on the left. Stable moderate bilateral sacroiliac joint arthropathy. No dislocation.\n\nSOFT TISSUES: Interval placement of bilateral femoral vascular graft stents. Stable linear surgical staples project over the pelvis and lower lumbar region.\n\nIMPRESSION:\n1. Stable moderate bilateral hip osteoarthritis, greater on the left.\n2. Stable moderate bilateral sacroiliac joint arthropathy.\n3. Stable mild enthesopathic changes at the greater trochanters and iliac wings.\n4. Stable lower lumbar fixation rods and transpedicular screws.\n5. Interval placement of bilateral femoral vascular graft stents.\n6. No acute osseous abnormality.",
 "32566854-3c67-4590-8efc-cf69110fba49": "FINDINGS:\nVERTEBRAE: Mild degenerative spondylotic changes at L5-S1. Normal desnity and alignment. No fracture or osseous lesion.\n\nDISC SPACES: Preserved.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Mild degenerative spondylotic changes at L5-S1.\n2. No acute abnormality evident in the lumbar spine.",
 "331058ea-8b7e-44ef-bebf-05d6145c618b": "FINDINGS:\nVERTEBRAE: Thoracic vertebral body heights and alignment are maintained. No acute compression fracture, traumatic malalignment, or focal osseous abnormality is demonstrated.\n\nDISC SPACES: Intervertebral disc spaces are preserved.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\nNo acute osseous abnormality of the thoracic spine.",
 "33b419a8-93ee-4d85-bab7-ea4be76f78d2": "FINDINGS:\nThere is straightening of the cervical lordosis, which may be positional or related to myospasm.\n\nThe vertebral body and disc space heights are preserved.\n\nNo acute fracture or listhesis is identified.\n\nThe odontoid process is intact.\n\nNo lytic or blastic lesions.\n\nNo significant neural foraminal stenosis.\n\nThe paravertebral soft tissues are normal.\n\nIMPRESSION:\n1. Straightened cervical lordosis, which may be positional or related to myospasm.\n2. No fracture or listhesis.\n3. No significant neural foraminal stenosis.",
 "33ee1fa0-f50e-46c1-a5c3-9bd3d5d0cd5d": "FINDINGS:\nLUNGS: Mild prominence of the bilateral perihilar/peribronchial interstitial markings. No focal airspace consolidation. No pulmonary edema.\n\nPLEURAL SPACES: No pleural effusion or pneumothorax.\n\nHEART: Cardiac silhouette is within normal size limits.\n\nMEDIASTINUM/HILA: Cardiomediastinal silhouette is within normal limits for size and contour. No mediastinal widening or hilar enlargement.\n\nDIAPHRAGM: Diaphragms are unremarkable. No subdiaphragmatic free air.\n\nBONES: No acute osseous abnormality identified on this radiograph.\n\nSOFT TISSUES: No acute soft tissue abnormality identified on the provided images.\n\nIMPRESSION:\n1. Mild bilateral peribronchial/interstitial prominence, which may reflect mild bronchitic or reactive airway changes.\n2. No focal pneumonia or other acute cardiopulmonary abnormality.",
 "36542d97-8501-4858-b25f-ff97227ddae8": "FINDINGS:\nBONES: Degenerative changes. No acute fracture or focal osseous lesion.\n\nJOINTS: No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Degenerative changes.\n2. No acute osseous abnormality.",
 "36ad41e3-8818-489a-8dcf-b3d668968d43": "FINDINGS:\nLUNGS/AIRWAYS: Moderate ground-glass opacities in the bilateral lower lobes, which may represent gravity-dependent atelectatic changes. An infectious or inflammatory etiology cannot be excluded. No focal pulmonary mass. The central airways are patent.\n\nPLEURA: No pleural effusion or pneumothorax.\n\nCARDIOVASCULAR: Right-sided aortic arch with the descending thoracic aorta coursing near the midline, likely accounting for the apparent right paratracheal opacity on the chest radiograph. Mild cardiomegaly. Mild coronary artery calcifications.\n\nMEDIASTINAL/HILAR LYMPH NODES: No mediastinal, axillary or supraclavicular lymphadenopathy. No right paratracheal soft-tissue mass.\n\nCHEST WALL/MUSCULOSKELETAL: Thoracic spondylosis. No acute or destructive osseous abnormality.\n\nLINES/TUBES/SUPPORT DEVICES: None.\n\nOTHER FINDINGS:\n\nSmall hiatal hernia. Surgically absent gallbladder. A 1 cm circumscribed hypodense lesion in the right hepatic lobe likely represents a cyst. A 3 cm left renal cyst is present.\n\nIMPRESSION:\n1. Right-sided aortic arch with the descending thoracic aorta coursing near the midline, likely accounting for the apparent right paratracheal opacity on the chest radiograph. No mediastinal mass or lymphadenopathy.\n2. Moderate bilateral lower lobe ground-glass opacities, possibly representing gravity-dependent atelectatic changes. An infectious or inflammatory process cannot be excluded.\n3. Mild cardiomegaly and mild coronary artery atherosclerotic calcifications.\n4. Small hiatal hernia.",
 "3730731d-d478-46d1-a952-cb0178c0707b": "FINDINGS:\nBONES: No acute fracture. No focal osseous lesion or cortical irregularity.\n\nJOINTS: No dislocation. Glenohumeral alignment is maintained. Acromioclavicular joint alignment is preserved. Joint spaces are maintained without significant degenerative change.\n\nSOFT TISSUES: Visualized soft tissues are unremarkable.\n\nIMPRESSION:\nNo acute osseous abnormality of the left shoulder. No acute fracture or dislocation.",
 "37380976-ba41-49eb-9648-444dff279242": "FINDINGS:\nALIGNMENT: Cervical spinal alignment is maintained. No significant spondylolisthesis.\n\nVERTEBRAL BODIES: Vertebral body heights are maintained. No acute compression fracture or focal suspicious marrow-replacing lesion. Postoperative changes are noted at C5-C6.\n\nSPINAL CORD: Cervical spinal cord is normal in caliber and signal intensity. No intramedullary T2/STIR hyperintensity or cord edema/myelomalacia is identified. Visualized cervicomedullary junction is unremarkable.\n\nC2-3: Disc height maintained with adequate hydration. No significant disc bulge or herniation. No significant spinal canal or neural foraminal stenosis.\n\nC3-4: Mild disc desiccation with small broad-based disc bulge. Mild indentation of the ventral thecal sac without significant cord compression. Mild bilateral neural foraminal narrowing.\n\nC4-5: Disc desiccation with broad-based disc bulge/minimal central disc protrusion. Bilateral neural foraminal narrowing is noted. Mild ventral thecal sac indentation. No significant cord impingement.\n\nC5-6: Postoperative changes at this level. Residual/recurrent posterior disc-osteophyte/disc protrusive component causes indentation of the ventral thecal sac and ventral aspect of the cervical cord, with associated central canal narrowing. No cord signal abnormality or edema. Neural foraminal narrowing is not significant.\n\nC6-7: Disc height and hydration are relatively maintained. No significant disc bulge or herniation. No significant spinal canal or neural foraminal stenosis.\n\nC7-T1: No significant disc bulge or herniation. No significant canal or foraminal stenosis.\n\nPARAVERTEBRAL SOFT TISSUES: No significant paraspinal soft-tissue abnormality.\n\nOTHER FINDINGS:\n\nNo significant facet arthropathy or focal posterior element abnormality.\n\nIMPRESSION:\n1. Postoperative changes at C5-C6 with residual posterior disc/osteophyte complex causing indentation of the ventral thecal sac and ventral cervical cord. No associated cord edema or myelomalacia.\n2. C4-C5 disc degeneration with broad-based disc bulge/protrusion and bilateral neural foraminal narrowing, without significant cord impingement.\n3. Mild additional multilevel cervical spondylotic/disc degenerative changes.\n4. No acute osseous abnormality or significant intrinsic cervical cord signal abnormality.",
 "37c4f280-4cb4-4413-b61a-2ec6fa8bc9e2": "FINDINGS:\nLIGAMENTS:\n\nSCAPHOLUNATE: The scapholunate ligament is intact.\n\nLUNOTRIQUETRAL: The lunotriquetral ligament is intact.\n\nTENDONS:\n\nFLEXOR COMPARTMENTS: The flexor tendons are normal in course and signal intensity without evidence of tenosynovitis.\n\nEXTENSOR COMPARTMENTS: Extensor carpi ulnaris (ECU) tendon shows circumferential distension of the ECU tendon sheath with fusiform fluid/synovial thickening, suggestive of tenosynovitis. The ECU tendon itself maintains normal continuity and signal intensity without evidence of tear or retraction. No definite focal tendon discontinuity is identified within the limitations imposed by postoperative artefacts.\n\nNERVES:\n\nMEDIAN: The median nerve demonstrates normal course and signal intensity within the carpal tunnel.\n\nULNAR: The ulnar nerve demonstrates normal course and signal intensity within Guyon's canal.\n\nBONES: Postoperative changes are noted involving the distal radius with associated susceptibility/distortion artefacts, limiting evaluation of adjacent osseous and soft-tissue structures. Patchy STIR hyperintense marrow signal involving the triquetrum and trapezium, which may represent reactive marrow edema/contusion.\n\nMUSCLES: The musculature is of normal bulk and signal intensity.\n\nJOINTS: Degenerative osteoarthritic changes are noted in the wrist. Mild joint effusion is noted. No subluxation. The radiocarpal and intercarpal articulations are maintained.\n\nCARTILAGE: The articular cartilage is preserved in thickness and signal intensity.\n\nTRIANGULAR FIBROCARTILAGE: The triangular fibrocartilage complex is intact.\n\nOTHER FINDINGS:\n\nMild subcutaneous soft-tissue edema is present. No obvious focal collection is seen.\n\nIMPRESSION:\n1. ECU tenosynovitis with circumferential synovial sheath distension; no definite ECU tendon tear.\n2. Patchy marrow edema-like signal in the triquetrum and trapezium, likely reactive/contusional.\n3. Mild wrist joint effusion and mild subcutaneous edema.\n4. Degenerative changes of the wrist.\n5. Postoperative changes of the distal radius with susceptibility artefacts limiting detailed assessment.",
 "388af5ed-85f8-49b5-85a0-d63b9e571848": "FINDINGS:\nLUNGS/AIRWAYS: Stable bilateral subpleural reticular and fibrotic opacities, greater in the lower lobes. Stable bilateral lower lobe bronchiectatic changes. No focal airspace consolidation. No suspicious pulmonary nodule or mass. The central airways are patent.\n\nPLEURA: No pleural effusion or pneumothorax.\n\nCARDIOVASCULAR: The heart is within normal limits in size. The thoracic aorta is normal in caliber. No pericardial effusion.\n\nMEDIASTINAL/HILAR LYMPH NODES: Stable few prominent mediastinal lymph nodes measuring up to approximately 1.0 cm in short-axis diameter. No axillary lymphadenopathy.\n\nCHEST WALL/MUSCULOSKELETAL: Cervical and thoracic spondylosis. No acute or destructive osseous abnormality.\n\nLINES/TUBES/SUPPORT DEVICES: None.\n\nOTHER FINDINGS:\n\nStable small hiatal hernia. Stable left renal cyst measuring approximately 4.0 cm.\n\nIMPRESSION:\n1. Stable bilateral subpleural reticular pulmonary fibrosis, predominantly involving the lower lobes, with stable bilateral lower lobe bronchiectasis. Findings are consistent with chronic fibrotic interstitial lung disease. Differential considerations include usual interstitial pneumonia and fibrotic nonspecific interstitial pneumonia.\n2. No focal airspace consolidation or suspicious pulmonary nodule.\n3. Stable few prominent mediastinal lymph nodes measuring up to 1.0 cm in short-axis diameter.\n4. Stable small hiatal hernia and 4.0 cm left renal cyst.\n5. Cervical and thoracic spondylosis.",
 "3ac0bcca-274c-4354-a235-203868739ca9": "FINDINGS:\nVERTEBRAE: Mild anterior wedging of the L1 vertebral body with reduction in anterior vertebral body height. No definite acute cortical disruption. Multilevel lower lumbar facet arthropathy, most pronounced at L5-S1. Lumbar vertebral alignment is otherwise maintained without significant spondylolisthesis. Mild degenerative curvature of the lumbar spine. Visualized sacroiliac joints are maintained. Mild diffuse osseous demineralization.\n\nDISC SPACES: Mild multilevel degenerative spondylotic changes with disc-space narrowing, endplate sclerosis and marginal osteophyte formation, most pronounced at L4-L5, where there is marked disc-space reduction and associated endplate degenerative change. Additional mild-to-moderate degenerative disc disease at the upper/mid lumbar levels.\n\nSOFT TISSUES: Unremarkable.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Multilevel lumbar spondylosis, most pronounced at L4-L5 with marked degenerative disc-space narrowing.\n2. Lower lumbar facet arthropathy, greatest at L5-S1.\n3. Mild anterior wedge compression deformity of L1, without radiographic features of an acute fracture.\n4. Mild diffuse osseous demineralization.",
 "3ae47cf7-e5e3-4550-9d76-4188437d65ae": "FINDINGS:\nLIGAMENTS:\n\nSCAPHOLUNATE: Scapholunate interval is widened with abnormal morphology of the scapholunate ligament, compatible with chronic ligamentous insufficiency/tear.\n\nLUNOTRIQUETRAL: Degenerative changes involving the lunotriquetral articulation.\n\nTENDONS:\n\nFLEXOR COMPARTMENTS: Mild fluid/synovial prominence surrounding the flexor tendons at the carpal tunnel. Flexor tendons are grossly intact.\n\nEXTENSOR COMPARTMENTS: Visualized extensor tendons are grossly intact without definite high-grade tear.\n\nNERVES:\n\nMEDIAN: Median nerve demonstrates mild increased signal and flattening within the carpal tunnel, which may be seen with median neuropathy/carpal tunnel syndrome. No discrete space-occupying mass within the carpal tunnel identified.\n\nULNAR: The ulnar nerve demonstrates normal course and signal intensity within Guyon's canal.\n\nBONES: Degenerative remodeling of multiple carpal bones with associated subchondral marrow edema-like signal. No acute fracture or dislocation identified. No focal aggressive osseous lesion.\n\nMUSCLES: The musculature is of normal bulk and signal intensity.\n\nJOINTS: Advanced multifocal degenerative changes involving the carpal bones, with marked joint space narrowing, subchondral sclerosis/cystic change, and osteophyte formation. Advanced radiocarpal and midcarpal osteoarthritis with extensive chondral loss and subchondral degenerative changes. Small associated joint effusions with synovial hypertrophic/degenerative changes. Prominent degenerative changes involving the scapholunate and capitolunate articulations.\n\nCARTILAGE: Extensive chondral loss is present.\n\nTRIANGULAR FIBROCARTILAGE: Degenerative thinning and irregularity of the triangular fibrocartilage complex. No definite acute full-thickness traumatic TFCC tear identified on the provided images.\n\nOTHER FINDINGS:\n\nNo focal soft-tissue mass or drainable collection. Mild diffuse soft-tissue edema about the wrist.\n\nIMPRESSION:\n1. Advanced multifocal degenerative osteoarthritis of the wrist involving the radiocarpal and midcarpal compartments, with extensive cartilage loss, subchondral cystic change/edema, and osteophyte formation.\n2. Chronic scapholunate ligament insufficiency/tear with widening of the scapholunate interval and associated advanced carpal degenerative changes.\n3. Degenerative changes of the TFCC without definite acute full-thickness tear.\n4. Mild flattening and increased signal of the median nerve within the carpal tunnel, which can be seen with carpal tunnel syndrome. No discrete compressive mass identified.\n5. No acute fracture or dislocation.",
 "3b4a4786-9a9f-44ea-a20d-f67e6ed78143": "FINDINGS:\nBONES: Radius and ulna are intact without acute fracture or dislocation. Mild diffuse osseous demineralization. No focal erosive osseous abnormality. No focal osseous lesion.\n\nJOINTS: Mild degenerative changes involving the radiocarpal/intercarpal and first carpometacarpal joints with small marginal osteophytes. Mild degenerative changes of the visualized elbow. No dislocation identified. Carpal alignment is maintained. Alignment is maintained.\n\nSOFT TISSUES: Soft tissues are grossly unremarkable.\n\nIMPRESSION:\n1. No acute fracture or dislocation of the left wrist or forearm.\n2. Mild multifocal osteoarthritic changes of the wrist and visualized elbow.\n3. Mild diffuse osseous demineralization.",
 "3c10a17c-6a11-410b-a978-36669ecac94d": "FINDINGS:\nBONES: Minimally displaced comminuted fracture of the ulnar styloid process is noted. Intra-articular fracture of the distal radius is seen, with fracture lines extending up to the radiocarpal articular surface. No significant articular surface depression or gross displacement is noted. No additional acute osseous abnormality is identified.\n\nJOINTS: Mild radiocarpal joint effusion is noted. Degenerative osteoarthritic changes are noted at the visualized wrist joints. Distal radioulnar articulation is maintained without dislocation. Carpal alignment is maintained.\n\nSOFT TISSUES: Mild diffuse soft-tissue swelling is seen around the wrist. The visualized musculature is symmetric and of normal bulk.\n\nIMPRESSION:\n1. Comminuted intra-articular fracture of the distal radius.\n2. Minimally displaced comminuted fracture of the ulnar styloid process.\n3. Mild wrist joint effusion and soft-tissue swelling.\n4. Maintained distal radioulnar and carpal alignment.\n5. Degenerative changes.",
 "3c2a11ad-ab15-4f85-96d5-01ff5cffffd8": "FINDINGS:\nBRAIN: Prominent Virchow-Robin spaces are noted in the lentiform nuclei bilaterally. No restricted diffusion to indicate acute infarction. No intracranial mass or hemorrhage. No midline shift or extra-axial fluid collection. No cerebellar tonsillar ectopia. The central arterial and venous flow voids are patent.\n\nVENTRICLES: The ventricles are normal in size and configuration.\n\nORBITS: The globes and retro-orbital structures are symmetric and without abnormality.\n\nSINUSES AND MASTOIDS: The paranasal sinuses and mastoid air cells are clear.\n\nBONES: The calvarium and skull base demonstrate normal signal intensity.\n\nIMPRESSION:\n1. Prominent bilateral lentiform Virchow-Robin spaces.\n2. Otherwise normal post-contrast brain MRI.",
 "3fdad783-1d5e-417b-9b42-e1ce3c00a8f8": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Degenerative changes in the form of marginal osteophytes and reduction in the medial tibiofemoral and patellofemoral joint spaces. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Degenerative changes with marginal osteophytes and medial tibiofemoral and patellofemoral joint space narrowing.\n2. No acute osseous abnormality.",
 "42270507-c160-415a-a4d7-0e5b662bc62f": "FINDINGS:\nOSSEOUS STRUCTURES: The adjacent humeral cortex is intact, with no cortical erosion, destruction, or abnormal marrow signal. No cortical remodeling of the adjacent humeral shaft. No associated periosteal reaction or acute osseous abnormality is identified.\n\nMUSCULATURE: Visualized surrounding musculature is otherwise preserved.\n\nSOFT TISSUES: There is a well-defined elongated fat-containing lesion in the intermuscular plane at the level of the mid shaft of the left humerus, measuring approximately 56 \u00d7 20 \u00d7 82 mm (transverse \u00d7 AP \u00d7 craniocaudal). The lesion demonstrates T1 hyperintense signal intensity with suppression/loss of signal on STIR sequences, consistent with its macroscopic fat content. No definite suspicious internal nodular component or non-fatty signal component is described. No significant adjacent soft-tissue edema or fluid collection.\n\nIMPRESSION:\n1. Well-defined intermuscular macroscopic fat-containing lesion along the mid shaft of the left humerus, measuring approximately 56 \u00d7 20 \u00d7 82 mm, most consistent with an intramuscular/intermuscular lipoma.\n2. No adjacent osseous remodeling due to chronic mass effect.\n3. No cortical destruction, bone erosion, or marrow involvement identified.\n4. No definite aggressive imaging features on the described sequences.",
 "430c0318-b10d-428a-8ffe-e40d1b2ae319": "FINDINGS:\nBRAIN: Moderate diffuse cerebral volume loss with prominence of the cortical sulci and ventricular system, suggestive of moderate cerebral atrophy. Moderate multifocal T2/FLAIR hyperintense white-matter signal changes (leukoaraiosis/leukoencephalopathy), predominantly involving the periventricular and deep white matter, likely representing chronic small-vessel ischemic gliotic changes. Chronic infarct/gliotic changes involving bilateral thalami, consistent with old bilateral thalamic infarcts. No evidence of acute intracranial hemorrhage, significant mass effect or midline shift. No obvious acute territorial infarction on the provided sequences.\n\nVENTRICLES: Prominence of the ventricular system in keeping with cerebral volume loss.\n\nORBITS: The globes and retro-orbital structures are symmetric and without abnormality.\n\nSINUSES AND MASTOIDS: The paranasal sinuses and mastoid air cells are clear.\n\nBONES: The calvarium and skull base demonstrate normal signal intensity.\n\nOTHER FINDINGS:\n\nMultilevel cervical spondylotic/degenerative changes. Diffuse disc bulges are noted at C2-C3 and C3-C4, with associated mild indentation of the ventral thecal sac.\n\nIMPRESSION:\n1. Moderate cerebral atrophy with moderate leukoaraiosis.\n2. Bilateral chronic thalamic lacunar infarcts.\n3. Multilevel cervical spondylotic changes with disc bulges at visualized C2-C3 and C3-C4.",
 "436438f8-5e11-497d-80fc-0c9e5a69e66c": "FINDINGS:\nVERTEBRAE: Five non-rib-bearing lumbar-type vertebral bodies identified. Lumbar alignment is maintained with no significant spondylolisthesis. Vertebral body heights are preserved without evidence of acute compression deformity. Mild multilevel degenerative spondylotic changes with small marginal endplate osteophytes. No focal destructive osseous lesion or acute osseous abnormality.\n\nDISC SPACES: Intervertebral disc spaces are relatively maintained.\n\nSOFT TISSUES: Visualized sacroiliac joints are unremarkable.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Mild multilevel degenerative spondylotic changes.\n2. No acute abnormality evident in the lumbar spine.",
 "4433633a-6c7b-408f-a18e-01ed07ea5c9b": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Marginal osteophytes of the knee joint without significant reduction in the joint space. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Marginal osteophytes without significant joint space reduction.\n2. No acute osseous abnormality.",
 "4589378c-5655-494b-b591-b43fd2b519b6": "FINDINGS:\nVERTEBRAE: Degenerative changes of the spine in the form of marginal osteophytes and endplate changes. Grade I anterolisthesis of L5 over S1. Grade I anterolisthesis of L4 over L5. Grade I retrolisthesis of L2 over L3. Loss of normal alignment of the spine. Fusion of L1 and L2. No fracture or osseous lesion. Degenerative changes of the bilateral sacroiliac joints and bilateral hips.\n\nDISC SPACES: Reduction of the disc space at L5-S1 and reduction in all the disc spaces.\n\nSOFT TISSUES: Atherosclerotic changes of the aorta.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Degenerative changes of the lumbar spine with marginal osteophytes, endplate changes and multilevel disc space reduction.\n2. Grade I anterolisthesis of L5 over S1 and of L4 over L5, with grade I retrolisthesis of L2 over L3 and loss of normal spinal alignment.\n3. Fusion of L1 and L2.\n4. Degenerative changes of the bilateral sacroiliac joints and bilateral hips.\n5. Atherosclerotic changes of the aorta.",
 "47866103-dcb8-4a22-9e5e-5393939583e5": "FINDINGS:\nBONES: Mild diffuse osseous demineralization. No acute fracture. No focal erosive or destructive osseous abnormality.\n\nJOINTS: Moderate multifocal interphalangeal joint space narrowing and marginal osteophyte formation, greatest at the distal interphalangeal joints. Moderate degenerative changes of the first carpometacarpal and triscaphe joints. No dislocation. Alignment is maintained.\n\nSOFT TISSUES: Soft tissues are unremarkable.\n\nIMPRESSION:\n1. Multifocal osteoarthritis, greatest at the distal interphalangeal and first carpometacarpal joints.\n2. No acute osseous abnormality of the left hand.",
 "49583957-18c2-43d5-bc4c-acc2a41d6e40": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Minor joint space narrowing and possible osteophytic lipping, consistent with grade 1 osteoarthritis. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Minor joint space narrowing and possible osteophytic lipping, consistent with grade 1 osteoarthritis.\n2. No acute osseous abnormality.",
 "49962b3d-4c21-4517-a91e-c6a5986e6614": "FINDINGS:\nBONES: Mild subchondral sclerosis of the medial tibial plateau. Mild spiking of the tibial spines. No acute fracture. No focal destructive osseous lesion.\n\nJOINTS: Mild degenerative osteoarthritic changes, predominantly involving the medial tibiofemoral compartment, with marginal osteophyte formation and mild joint-space narrowing. No dislocation. No significant suprapatellar joint effusion.\n\nSOFT TISSUES: Periarticular soft tissues are grossly unremarkable.\n\nIMPRESSION:\n1. Mild degenerative osteoarthritic changes, predominantly involving the medial tibiofemoral compartment.\n2. No acute osseous abnormality.",
 "4a66a692-c163-4cca-8d59-206c2670a975": "FINDINGS:\nBRAIN: Mild diffuse cerebral volume loss with prominence of the cortical sulci and ventricular system, consistent with mild age-related cerebral atrophy. No focal area of abnormal signal intensity, acute infarction, intracranial hemorrhage or space-occupying lesion is identified. No significant mass effect or midline shift. No significant focal abnormal signal intensity is seen in the basal ganglia, thalami or periventricular white matter. Brainstem and cerebellum appear unremarkable. Corpus callosum and sellar region are unremarkable. Major intracranial vascular flow voids are preserved.\n\nVENTRICLES: Mild prominence of the ventricles and cortical sulci in keeping with cerebral volume loss. No hydrocephalus or extra-axial collection.\n\nORBITS: No gross abnormality in the visualized portions.\n\nSINUSES AND MASTOIDS: No significant mucosal disease in the visualized paranasal sinuses. Rightward deviation of the nasal septum with a small bony spur, which indents the right inferior turbinate. Mastoid air cells are clear.\n\nBONES: The calvarium and skull base demonstrate normal signal intensity.\n\nIMPRESSION:\n1. Mild diffuse cerebral atrophy.\n2. Rightward deviated nasal septum with a small bony septal spur indenting the right inferior turbinate.",
 "4cee9330-9f62-4489-b475-b3cdd65f3f2b": "FINDINGS:\nBONES: Subcortical cystic changes at the humeral head. No acute fracture or focal lesion.\n\nJOINTS: Moderate acromioclavicular joint space narrowing. Mild to moderate glenohumeral joint space narrowing with subchondral sclerosis along the glenoid. Acromiohumeral interval is intact. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Moderate acromioclavicular and mild to moderate glenohumeral joint space narrowing with subchondral sclerosis.\n2. Subcortical cystic changes at the humeral head.\n3. No acute fracture or dislocation.",
 "4e83994c-4f17-431b-ba2f-4d9347597eaa": "FINDINGS:\nBONES: No acute fracture is demonstrated.\n\nJOINTS: Mild medial compartment joint space narrowing is demonstrated with small marginal osteophytes. Mild patellofemoral degenerative change is also present. No dislocation is demonstrated. No significant knee joint effusion is identified.\n\nSOFT TISSUES: Mild vascular calcifications are present in the posterior soft tissues. Soft tissue fullness is demonstrated posterior to the knee, which may represent a Baker's cyst.\n\nIMPRESSION:\n1. Mild degenerative osteoarthritis of the right knee, greatest in the medial compartment.\n2. Posterior knee soft tissue fullness, likely representing a Baker's cyst.\n3. No acute osseous abnormality.",
 "4f573c02-34e8-4ebf-942e-1b7bf7e2b7b2": "FINDINGS:\nStraightening of the normal cervical lordosis is demonstrated. There is no spondylolisthesis.\n\nVertebral body heights are maintained without acute compression fracture or traumatic malalignment.\n\nMild multilevel degenerative endplate changes are demonstrated with small anterior osteophytes and mild disc space narrowing, greatest at C6-C7.\n\nNo acute fracture is identified.\n\nThe odontoid process is intact.\n\nNo lytic or blastic lesions.\n\nNo prevertebral soft tissue swelling is demonstrated.\n\nIMPRESSION:\n1. Straightening of the normal cervical lordosis.\n2. Mild multilevel cervical spondylosis, greatest at C6-C7.\n3. No acute osseous abnormality.",
 "53b0c08b-e433-4350-9880-c5b5f71dadd0": "FINDINGS:\nLUNGS: Prominent bronchovascular markings.\n\nPLEURAL SPACES: No pleural effusion or pneumothorax.\n\nHEART: Mild cardiomegaly.\n\nMEDIASTINUM/HILA: Atherosclerotic change of the aortic arch. No mediastinal widening or hilar enlargement.\n\nDIAPHRAGM: Diaphragms are unremarkable. No subdiaphragmatic free air.\n\nBONES: No acute osseous abnormality identified on this examination.\n\nSOFT TISSUES: No acute soft tissue abnormality identified on the provided images.\n\nIMPRESSION:\n1. Prominent bronchovascular markings.\n2. Mild cardiomegaly.\n3. Atherosclerotic change of the aortic arch.",
 "5486607a-a7cb-4722-b63e-16022f0b8394": "FINDINGS:\nBRAIN: Bilateral periventricular T2 and FLAIR hyperintensity, with the possibility of chronic ischaemic changes. Generalised atrophy with prominent sulcal spaces seen in the right parietal lobe, with the possibility of focal gliosis. Focal gliosis is seen in the bilateral centrum semiovale. Prominent Virchow-Robin spaces in the bilateral lentiform nuclei. Partial empty sella. No cerebellar tonsillar ectopia. The central arterial and venous flow voids are patent. No restricted diffusion to indicate acute infarction. No intracranial mass or hemorrhage. No midline shift or extra-axial fluid collection.\n\nVENTRICLES: Dilatation of the bilateral lateral ventricles.\n\nORBITS: The globes and retro-orbital structures are symmetric and without abnormality.\n\nSINUSES AND MASTOIDS: The paranasal sinuses are clear. Left mastoiditis. Minimal signal is seen in the right mastoid air cells.\n\nBONES: Hyperostosis frontalis interna.\n\nIMPRESSION:\n1. Bilateral periventricular T2/FLAIR hyperintensity, likely chronic ischaemic change, with focal gliosis in the bilateral centrum semiovale.\n2. Generalised cerebral atrophy with dilatation of the bilateral lateral ventricles.\n3. Left mastoiditis with minimal signal in the right mastoid air cells.\n4. Prominent Virchow-Robin spaces in the bilateral lentiform nuclei.\n5. Partial empty sella and hyperostosis frontalis interna.",
 "557a0d37-7b9d-4ca5-af12-71596dffcc1e": "FINDINGS:\nBONES: No fracture or focal lesion.\n\nJOINTS: There is moderate acromioclavicular osteoarthritis with joint space narrowing and prominent superior marginal osteophyte formation. Glenohumeral alignment is maintained with mild joint space narrowing and small marginal osteophytes. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Moderate acromioclavicular osteoarthritis.\n2. Mild glenohumeral joint space narrowing with small marginal osteophytes.\n3. No acute osseous abnormality.",
 "5658cf75-5a93-4167-bd35-029124d4754d": "FINDINGS:\nBONES: Multiple intramedullary lucent lesions are present within the distal femoral shaft and proximal fibula. Cortical irregularity, remodeling, and periosteal reaction are present along the proximal fibular shaft. No acute fracture.\n\nJOINTS: Moderate medial and patellofemoral compartment joint space narrowing with mild lateral compartment joint space narrowing and small marginal osteophytes. Moderate knee joint effusion. No dislocation.\n\nSOFT TISSUES: Soft tissue swelling of the leg. Calcifications of the femoral artery are noted.\n\nIMPRESSION:\n1. Intramedullary lucent lesions within the distal femur and proximal fibula, with cortical irregularity and periosteal reaction along the proximal fibula. Findings are concerning for osteomyelitis. Multifocal neoplastic lesions is a differential consideration.\n2. Tricompartmental osteoarthritis, moderate in the medial and patellofemoral compartments.\n3. Moderate knee joint effusion and soft tissue swelling of the leg.\n4. Femoral arterial atherosclerotic calcifications.\n5. No acute fracture or dislocation.",
 "56f757ba-a872-4fd4-8b7d-d86399ee3531": "FINDINGS:\nBONES: Diffuse osseous demineralization. No acute fracture.\n\nJOINTS: Mild radiocarpal joint space narrowing and mild degenerative changes of the triscaphe and thumb carpometacarpal joints. No dislocation. Carpal alignment is maintained.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild radiocarpal joint space narrowing and mild degenerative changes of the triscaphe and thumb carpometacarpal joints.\n2. Diffuse osseous demineralization.\n3. No acute osseous abnormality.",
 "579a25f5-63c5-4e5c-b67f-3c98ee406665": "FINDINGS:\nBONES: Oblique fracture of the distal fibula/lateral malleolus at the level of the ankle syndesmosis (Weber B pattern) is seen. Fracture alignment is maintained without significant interval displacement or angulation. Assessment of the fracture margins and osseous healing is limited by overlying plaster cast.\n\nJOINT SPACES AND ALIGNMENT: Ankle mortise remains congruent, without appreciable medial clear-space widening.\n\nSOFT TISSUES: No focal soft tissue swelling. No abnormal calcifications.\n\nIMPRESSION:\n1. Oblique fracture of the distal fibula/lateral malleolus at the level of the ankle syndesmosis (Weber B pattern).\n2. Fracture alignment is maintained without significant interval displacement or angulation.\n3. Ankle mortise remains congruent.",
 "58fdafd6-4edb-4e60-b0df-0821061121a1": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Moderate medial, lateral and patellofemoral compartment joint space narrowing with small marginal osteophytes and mild subchondral sclerosis. Small suprapatellar joint effusion. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Moderate tricompartmental joint space narrowing with small marginal osteophytes and mild subchondral sclerosis.\n2. Small suprapatellar joint effusion.\n3. No acute fracture or dislocation.",
 "5ddc141b-acfd-49f9-bc27-af6f17c55b36": "FINDINGS:\nALIGNMENT: Loss of the normal cervical lordosis with straightening of the cervical spine.\n\nVERTEBRAL BODIES: Vertebral body heights are maintained. No obvious acute compression deformity or focal marrow-replacing lesion on the provided sequences.\n\nSPINAL CORD: The cord is normal in signal intensity and caliber.\n\nC2-3: Mild posterior disc bulge with no significant spinal canal stenosis and no significant neural foraminal narrowing.\n\nC3-4: Mild posterior disc bulge causing mild ventral thecal sac indentation. No significant spinal canal stenosis. No neural foraminal narrowing.\n\nC4-5: Mild posterior central disc prolapse causing mild ventral thecal sac indentation. No significant spinal canal stenosis. No neural foraminal narrowing. AP diameter 11 mm.\n\nC5-6: Mild diffuse posterior disc bulge causing ventral thecal sac indentation with mild spinal canal narrowing. AP diameter of 9 mm. No abnormal signal intensity in the cord. No neural foraminal narrowing.\n\nC6-7: Mild posterior central disc prolapse causing ventral thecal sac indentation with mild spinal canal narrowing. No abnormal signal intensity in the cord. No neural foraminal narrowing. AP diameter 8.8 mm.\n\nC7-T1: No disc herniation or spinal canal stenosis. No neural foraminal narrowing.\n\nPARAVERTEBRAL SOFT TISSUES: Paravertebral soft tissues are normal.\n\nMultilevel degenerative disc disease with disc desiccation and variable reduction in disc heights, associated with marginal endplate osteophyte formation. Multilevel posterior disc-osteophyte complexes, most conspicuous at the C5-C6 and C6-C7 levels, causing ventral thecal sac indentation.\n\nIMPRESSION:\n1. Loss of the normal cervical lordosis with straightening of the cervical spine.\n2. Multilevel cervical degenerative disc disease with posterior disc-osteophyte complexes, most conspicuous at C5-C6 and C6-C7, causing ventral thecal sac indentation and mild spinal canal narrowing.\n3. No abnormal cord signal intensity.",
 "5e6776a6-757a-4193-a7ab-8403c63a911d": "FINDINGS:\nOSSEOUS STRUCTURES: Mild degenerative subchondral changes. No acute fracture or dislocation. No significant bone marrow edema or focal aggressive osseous lesion.\n\nJOINT: Small knee joint effusion. No definite intra-articular loose body.\n\nMENISCI:\n\nMEDIAL MENISCUS: Mild intrasubstance degenerative signal without definite surfacing tear.\n\nLATERAL MENISCUS: Abnormal linear fluid-sensitive signal involving the anterior horn of the lateral meniscus, extending to the articular surface, compatible with an anterior horn lateral meniscal tear. No definite displaced meniscal fragment or significant extrusion.\n\nLIGAMENTS:\n\nANTERIOR CRUCIATE LIGAMENT: Intact with preserved course, caliber, and fiber continuity. No convincing evidence of ACL tear.\n\nPOSTERIOR CRUCIATE LIGAMENT: Intact with preserved morphology and signal.\n\nMEDIAL COLLATERAL LIGAMENT: Intact.\n\nLATERAL COLLATERAL LIGAMENT COMPLEX: Intact. Popliteus tendon is intact.\n\nARTICULAR CARTILAGE: Mild multifocal chondral thinning and surface irregularity involving the medial, lateral, and patellofemoral compartments. No definite full-thickness cartilage defect identified on the provided images.\n\nEXTENSOR MECHANISM: Quadriceps and patellar tendons are intact. Patellar alignment is maintained.\n\nBURSAE: No significant bursitis.\n\nMUSCULATURE: Visualized musculature and tendons are grossly intact.\n\nSOFT TISSUES: No significant Baker's cyst. No focal soft-tissue mass or fluid collection.\n\nIMPRESSION:\n1. Anterior horn tear of the lateral meniscus.\n2. Intact ACL without evidence of tear.\n3. Small knee joint effusion.\n4. No acute osseous abnormality.",
 "62a3b0d3-072c-4cc9-97da-1f6aeb8ec85a": "FINDINGS:\nROTATOR CUFF TENDONS AND MUSCLES: Supraspinatus, infraspinatus and teres minor tendons are maintained in thickness and signal intensity without significant tendinopathy, partial-thickness tear or full-thickness tear. Minimal focal STIR hyperintensity is seen within the distal subscapularis tendon just proximal to its insertion, suggestive of mild tendinopathic/inflammatory change. No tendon retraction or muscle atrophy is noted. Visualized shoulder girdle muscles show preserved bulk and signal intensity without significant muscle edema or atrophy.\n\nLIGAMENTS: Visualized ligaments are normal.\n\nBICEPS ANCHOR COMPLEX: Long head of biceps tendon is maintained in position and shows preserved caliber and signal. Trace fluid is seen along its tendon sheath.\n\nGLENOHUMERAL JOINT: Glenohumeral alignment is maintained. Minimal fluid between the anterior-inferior glenoid labral region and glenoid cavity. No significant glenohumeral joint effusion. The articular cartilage is preserved. The labrum is intact.\n\nACROMIOCLAVICULAR JOINT: Mild degenerative hypertrophic changes at the acromioclavicular joint with mild associated STIR hyperintensity, suggestive of reactive marrow edema. No significant AC joint separation is seen.\n\nBONES: Few small focal subchondral cystic changes are seen within the humeral head, likely degenerative in nature. Focal STIR hyperintensity is also noted along the posterosuperior aspect of the humeral head, suggestive of mild reactive marrow edema. No focal significant osseous abnormality is identified in the glenoid.\n\nPERIARTICULAR SOFT TISSUE: Normal.\n\nIMPRESSION:\n1. Mild acromioclavicular joint osteoarthritic/degenerative changes with mild reactive marrow edema.\n2. Few small degenerative subchondral cystic changes in the humeral head with focal reactive marrow edema at the posterosuperior humeral head.\n3. Minimal distal subscapularis tendinopathic change without definite tear.\n4. Trace fluid along the long head of biceps tendon sheath.\n5. Minimal fluid between the anterior-inferior glenoid labral region and glenoid cavity.",
 "63b5c5a0-0baa-49d3-bf1d-92541d3328da": "FINDINGS:\nBONES: No fracture or focal osseous lesion.\n\nJOINTS: Small suprapatellar joint effusion. No dislocation. No significant degenerative changes.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Small suprapatellar joint effusion.\n2. No fracture or dislocation.",
 "63caa9d4-d265-4dd9-8a0c-13cc556ea5b6": "FINDINGS:\nVERTEBRAE: Five non-rib-bearing lumbar-type vertebral bodies identified. Vertebral body heights are maintained without acute compression deformity or focal destructive osseous lesion. Lower lumbar facet hypertrophic changes, greatest at L4-L5 and L5-S1. Mild lumbar levoconvex curvature. No significant spondylolisthesis identified. Sacroiliac joints are maintained. No acute osseous abnormality identified on these radiographs.\n\nDISC SPACES: Mild multilevel degenerative disc space narrowing and endplate osteophyte formation, greatest in the lower lumbar spine, particularly at L4-L5 and L5-S1.\n\nSOFT TISSUES: Unremarkable.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Multilevel lumbar spondylosis/degenerative disc disease, greatest at L4-L5 and L5-S1.\n2. Lower lumbar facet arthropathy.\n3. Mild lumbar levoconvex curvature.\n4. No acute osseous abnormality.",
 "65d40e7e-7c64-4b71-b874-d2e5fbfcb1ac": "FINDINGS:\nBOWEL: Rectum and visualized bowel show no significant mural thickening or obvious focal abnormality.\n\nBLADDER: Diffuse irregular circumferential mural thickening is noted, measuring up to approximately 2.5 mm. Urinary bladder is mildly distended. No obvious intraluminal mass lesion is seen.\n\nPROSTATE: Mild prostatomegaly, with prostate volume approximately 25 mL. No obvious focal suspicious lesion is identified on the provided sequences.\n\nSEMINAL VESICLES: Seminal vesicles are bilaterally symmetric and unremarkable.\n\nLYMPH NODES: No significant pelvic lymphadenopathy.\n\nBONES: No obvious focal marrow-replacing lesion or significant soft-tissue abnormality.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Mildly distended urinary bladder with diffuse irregular mural thickening, likely inflammatory/cystitic in the appropriate clinical setting.\n2. Mild prostatomegaly, prostate volume approximately 25 mL.",
 "697eea8a-94bf-4e40-9b09-49e1762f5475": "FINDINGS:\nVERTEBRAE: Straightened lumbar lordosis is seen, which may be positional or related to muscle spasm. Subtle levoconvexity of the lumbar spine is present. Small marginal osteophytes are present in the upper lumbar spine. No acute fracture or listhesis is identified. Mild L5-S1 facet joint arthrosis is seen. Mild bilateral sacroiliac joint degenerative changes are present.\n\nDISC SPACES: Mild intervertebral disc space narrowing is seen at T12-L1, L1-L2 and L5-S1, with moderate narrowing at L2-L3.\n\nSOFT TISSUES: Moderate atherosclerotic calcification of the abdominal aorta is present.\n\nSevere osteoarthritic changes of the left hip with avascular necrosis are noted, detailed in a dedicated left hip report.\n\nIMPRESSION:\n1. Multilevel lumbar degenerative disc disease, with moderate narrowing at L2-L3 and mild narrowing at T12-L1, L1-L2 and L5-S1.\n2. Mild L5-S1 facet joint arthrosis and mild bilateral sacroiliac joint degenerative changes.\n3. Straightened lumbar lordosis with subtle levoconvexity.\n4. Severe osteoarthritic changes of the left hip with avascular necrosis.\n5. Moderate atherosclerotic calcification of the abdominal aorta.",
 "69dad284-c63e-432c-afa4-7f0946ee0641": "FINDINGS:\nBONES: Mildly displaced fracture of the distal phalanx.\n\nJOINTS: No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\nMildly displaced fracture of the distal phalanx.",
 "6ac8ae67-2f79-4e80-acf5-b2f914450301": "FINDINGS:\nBONES: There is a small enthesophyte at the Achilles tendon insertion. A well-corticated os peroneum is incidentally noted. No significant plantar calcaneal spur is present. No acute fracture or focal osseous lesion.\n\nJOINTS: No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: Mild soft tissue swelling.\n\nIMPRESSION:\n1. Small enthesophyte at the Achilles tendon insertion.\n2. Mild soft tissue swelling.\n3. Incidental os peroneum.\n4. No acute osseous abnormality.",
 "6acfbe16-5b4b-4fff-a8ae-b897845e8929": "FINDINGS:\nBONES: No acute fracture. No focal destructive osseous lesion.\n\nJOINTS: Mild narrowing of the medial tibiofemoral and patellofemoral joint spaces. Mild marginal osteophyte formation involving the tibial plateau, femoral condyles and patella. Mild subchondral sclerosis. No dislocation.\n\nSOFT TISSUES: No significant joint effusion or other soft-tissue abnormality evident on radiograph.\n\nIMPRESSION:\n1. Mild tricompartmental degenerative osteoarthritic changes of the knee.\n2. No acute osseous abnormality.",
 "6ade6784-c2a3-44c0-87e4-a4648d54b68e": "FINDINGS:\nBONES: There is chronic osseous remodeling about the proximal humerus surrounding the humeral component, corresponding to the site of the previously reported comminuted periprosthetic fracture. There is persistent periprosthetic lucency along the humeral component, including near the distal stem, which may reflect chronic loosening. Prior distal clavicular resection.\n\nJOINTS: Left reverse total shoulder arthroplasty is present. No gross dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Left reverse total shoulder arthroplasty with chronic osseous remodeling about the proximal humerus, corresponding to the site of the previously reported comminuted periprosthetic fracture.\n2. Persistent periprosthetic lucency along the humeral component, including near the distal stem, which may reflect chronic loosening.\n3. No gross dislocation.",
 "6cb899a0-fa0a-4714-a7e0-04cad7de16ae": "FINDINGS:\nVERTEBRAE: Five non-rib-bearing lumbar-type vertebral bodies. Vertebral body heights and alignment are maintained. Mild multilevel degenerative spondylotic changes in the form of small marginal endplate osteophytes and mild endplate sclerosis, slightly more conspicuous in the lower lumbar spine. Lower lumbar facet degenerative changes are most evident at L4-L5 and L5-S1. No acute fracture or significant spondylolisthesis.\n\nDISC SPACES: Intervertebral disc spaces are relatively maintained.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Mild multilevel lumbar spondylosis with lower lumbar facet arthropathy, without significant disc space narrowing.\n2. No acute osseous abnormality.",
 "6ef96248-9538-41dc-95c2-1bee80198349": "FINDINGS:\nLUNGS: No significant interval change in the bilateral perihilar and bibasilar interstitial and airspace opacities. Mild central pulmonary vascular prominence is also unchanged.\n\nPLEURAL SPACES: No pleural effusion or pneumothorax.\n\nHEART: Stable mild cardiomegaly.\n\nMEDIASTINUM/HILA: Cardiomediastinal silhouette is within normal limits. No mediastinal widening or hilar enlargement.\n\nDIAPHRAGM: Diaphragms are unremarkable. No subdiaphragmatic free air.\n\nBONES: Prior median sternotomy. No acute osseous abnormality identified on this examination.\n\nSOFT TISSUES: No acute soft tissue abnormality identified on the provided images.\n\nIMPRESSION:\n1. No significant interval change in the bilateral perihilar and bibasilar interstitial and airspace opacities, which may represent chronic lung changes. A superimposed acute or recurrent infectious process, including atypical/interstitial pneumonia, cannot be excluded in the appropriate clinical setting.\n2. Stable mild cardiomegaly with mild central pulmonary vascular prominence.\n3. Prior median sternotomy.",
 "7212e321-3847-4906-a334-3ed302784c9a": "FINDINGS:\nOSSEOUS STRUCTURES: No focal acute osseous abnormality or marrow edema.\n\nJOINT: Knee joint alignment is maintained. Minimal knee joint effusion.\n\nMENISCI: Minimal extrusion of the meniscal tissue beyond the respective tibial margins. No definite focal meniscal tear is identified.\n\nMEDIAL MENISCUS: Intact.\n\nLATERAL MENISCUS: Intact.\n\nLIGAMENTS:\n\nANTERIOR CRUCIATE LIGAMENT: Diffusely bulky with heterogeneous increased intrasubstance signal intensity and maintained fibre continuity, suggestive of mucoid degeneration/intrasubstance degeneration with sprain. No definite complete ACL disruption is seen.\n\nPOSTERIOR CRUCIATE LIGAMENT: Mildly bulky with heterogeneous signal intensity, suggestive of PCL sprain/degenerative change. Fibre continuity is maintained.\n\nMEDIAL COLLATERAL LIGAMENT: Intact.\n\nLATERAL COLLATERAL LIGAMENT COMPLEX: Intact.\n\nARTICULAR CARTILAGE: Cartilage thickness and signal are preserved.\n\nEXTENSOR MECHANISM: Near full-thickness tear of the distal quadriceps tendon just proximal to its patellar insertion, at the level of the superior pole of the patella. The involved tendon demonstrates heterogeneous increased signal intensity with associated surrounding soft-tissue edema/injury. Focal increased STIR signal intensity within the patellar tendon at its patellar insertion, with mild adjacent soft-tissue edema, suggestive of insertional sprain/tendinous injury.\n\nBURSAE: No significant bursitis.\n\nMUSCULATURE: Normal bulk and signal.\n\nSOFT TISSUES: Soft-tissue edema/injury anteriorly around the distal quadriceps tendon and superior aspect of the patella.\n\nIMPRESSION:\n1. Near full-thickness tear of the distal quadriceps tendon just proximal to its patellar insertion, at the level of the superior pole of the patella, with surrounding soft-tissue edema/injury.\n2. Focal insertional sprain/tendinous injury of the patellar tendon with adjacent soft-tissue edema.\n3. Diffuse mucoid/intrasubstance degeneration with sprain of the ACL; fibres remain continuous.\n4. Mild bulky heterogeneous PCL, suggestive of sprain/degenerative change.\n5. Minimal meniscal extrusion.\n6. Minimal knee joint effusion.",
 "755948c2-f311-4b9c-b1a4-37ac63ba73eb": "FINDINGS:\nVERTEBRAE: A well-defined focal T1/T2 hyperintense lesion is seen within the L1 vertebral body, consistent with a vertebral hemangioma. Another focal lesion involving the L5 vertebral body appears hyperintense on T2/STIR sequences, likely representing a vertebral hemangioma. Vertebral body heights are maintained. No acute compression fracture or significant vertebral collapse. Marrow signal is within normal limits for age.\n\nALIGNMENT: Anatomic lumbar lordosis is maintained. No spondylolisthesis.\n\nSPINAL CORD: The conus medullaris terminates at a normal level, demonstrating normal signal and contour. The cauda equina nerve roots are unremarkable.\n\nDISCS/DEGENERATIVE CHANGES: Multilevel degenerative spondylotic changes are noted in the lumbar spine with multilevel intervertebral disc desiccation and mild degenerative endplate changes. Multilevel ligamentum flavum hypertrophy is also noted.\n\nL1-L2: Diffuse disc bulge/prolapse without significant central canal stenosis or neural foraminal compromise.\n\nL2-L3: Mild diffuse disc bulge with associated degenerative changes. No significant central canal or neural foraminal stenosis.\n\nL3-L4: Mild diffuse disc bulge with facet/ligamentum flavum hypertrophic changes. No significant canal or foraminal stenosis.\n\nL4-L5: Diffuse disc bulge causing bilateral neural foraminal narrowing. No significant exiting nerve root compression or definite nerve root impingement.\n\nL5-S1: Diffuse disc bulge/prolapse with bilateral lateral recess/foraminal narrowing and impingement of the exiting/lateral nerve roots. No significant high-grade central canal stenosis is identified.\n\nPARASPINAL SOFT TISSUES: The visualized paravertebral soft tissues are symmetric and without abnormal signal.\n\nIMPRESSION:\n1. Multilevel lumbar spondylosis and degenerative disc disease with multilevel ligamentum flavum hypertrophy.\n2. L4-L5 diffuse disc bulge causing bilateral neural foraminal narrowing without significant definite nerve root compression.\n3. L5-S1 diffuse disc bulge with bilateral lateral recess/foraminal narrowing and impingement of the bilateral exiting nerve roots.\n4. Vertebral hemangiomas at L1 and L5.\n5. No significant central canal stenosis.",
 "759534d0-c00b-4603-8aa6-b67ca3ca8e15": "FINDINGS:\nDEEP VEINS: The right common femoral, femoral, deep femoral, popliteal and posterior tibial veins demonstrate normal compressibility, color flow and augmentation.\n\nSUPERFICIAL VEINS: The right great saphenous vein is patent.\n\nSOFT TISSUES: No popliteal fossa cyst or other abnormalities.\n\nIMPRESSION:\nNo deep venous thrombosis in the right lower extremity.",
 "7a02839e-1e07-44d4-b41e-638a1615c786": "FINDINGS:\nBONES: Small plantar calcaneal enthesophyte. No acute fracture, dislocation, or focal osseous lesion. Bone mineralization is normal.\n\nJOINT SPACES AND ALIGNMENT: Maintained. Ankle mortise maintained.\n\nSOFT TISSUES: No focal soft tissue swelling. No abnormal calcifications.\n\nIMPRESSION:\n1. Small plantar calcaneal enthesophyte.\n2. No acute osseous abnormality.",
 "7a2dc87b-bf12-4cc8-b54e-9305c69571ce": "FINDINGS:\nBONES: Mild diffuse osseous demineralization. No acute fracture or focal osseous lesion.\n\nJOINTS: Moderate degenerative changes of the first carpometacarpal joint with joint-space narrowing, subchondral sclerosis and marginal osteophyte formation. Mild to moderate multifocal interphalangeal joint space narrowing and osteophyte formation, greatest at the distal interphalangeal joints. Mild degenerative changes of the triscaphe and radiocarpal articulations. No dislocation. Osseous alignment is maintained.\n\nSOFT TISSUES: Soft tissues are unremarkable without radiopaque foreign body.\n\nIMPRESSION:\n1. Moderate first carpometacarpal osteoarthritis with mild to moderate multifocal interphalangeal osteoarthritis.\n2. Mild diffuse osseous demineralization.\n3. No acute osseous abnormality of the left hand/wrist.",
 "7a837822-a72a-4f21-8750-d36d87d3fdbf": "FINDINGS:\nBONES: Diffuse osseous demineralization. No acute fracture.\n\nJOINTS: Mild multifocal interphalangeal joint space narrowing with small marginal osteophytes, as well as mild degenerative change of the first MCP and thumb CMC articulations. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Diffuse osseous demineralization.\n2. Mild multifocal interphalangeal, first MCP and thumb CMC degenerative changes.\n3. No acute osseous abnormality.",
 "7c981ae5-1672-412f-912f-92793cea174d": "FINDINGS:\nROTATOR CUFF TENDONS AND MUSCLES: STIR hyperintensity and increased signal are noted within the subscapularis tendon at its humeral insertion, suggestive of tendinopathic/strain-related changes. Supraspinatus and infraspinatus tendons are grossly maintained without definite full-thickness tear. Teres minor tendon is intact. No significant muscle atrophy or fatty infiltration.\n\nLIGAMENTS: Visualized ligaments are normal.\n\nBICEPS ANCHOR COMPLEX: No gross tendon retraction is seen.\n\nGLENOHUMERAL JOINT: Minimal focal discontinuity/irregularity is noted involving the anterior-inferior glenoid labrum, suggestive for a small partial labral tear. No significant joint effusion. Articular cartilage is grossly preserved. No focal osteochondral defect identified.\n\nACROMIOCLAVICULAR JOINT: Minimal hyperintense joint fluid is seen within the acromioclavicular joint. No gross cortical disruption or displaced fracture fragment is identified.\n\nBONES: No acute fracture, dislocation, or aggressive osseous lesion. Normal bone marrow signal.\n\nPERIARTICULAR SOFT TISSUE: Normal.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. Small partial tear of the anterior-inferior glenoid labrum.\n2. Tendinopathic/strain-related changes of the subscapularis tendon at its humeral insertion.\n3. Minimal acromioclavicular joint fluid, suggestive of acute/subacute stress/traumatic changes of the AC joint.\n4. No definite full-thickness rotator cuff tear.",
 "7ccf69b7-3f3d-4f9d-b47e-ebafd62e7db8": "FINDINGS:\nBRAIN: No enhancing focal lesions in the bilateral cerebello-pontine angles. Mild diffuse cerebral atrophy in the form of prominent cortical sulci. No restricted diffusion to indicate acute infarction. No intracranial mass or hemorrhage. No midline shift or extra-axial fluid collection. No cerebellar tonsillar ectopia. The central arterial and venous flow voids are patent.\n\nVENTRICLES: The ventricular system is prominent.\n\nORBITS: The globes and retro-orbital structures are symmetric and without abnormality.\n\nSINUSES AND MASTOIDS: The paranasal sinuses and mastoid air cells are clear.\n\nBONES: The calvarium and skull base demonstrate normal signal intensity.\n\nIMPRESSION:\n1. No enhancing focal lesions in the bilateral cerebello-pontine angles.\n2. Mild diffuse cerebral atrophy.",
 "7d1b1d15-8fb5-4c42-8088-bb1b678bdb72": "FINDINGS:\nBONES: Mild diffuse osseous demineralization. No acute fracture.\n\nJOINTS: Mild degenerative joint space narrowing of the left hip with marginal osteophyte formation. Mild degenerative changes of the visualized sacroiliac joints and lower lumbar spine. No dislocation.\n\nSOFT TISSUES: Soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild left hip osteoarthritis.\n2. No acute osseous abnormality.",
 "82c296eb-836f-42a6-a697-f5c57d111b65": "FINDINGS:\nBONES: There is a small Achilles insertional enthesophyte. No acute fracture. No focal osseous erosion or destructive lesion is identified.\n\nJOINTS: Moderate hallux valgus deformity is present with medial first metatarsal head prominence and moderate first metatarsophalangeal joint space narrowing, subchondral sclerosis, and marginal osteophyte formation. Mild degenerative changes involve the interphalangeal and midfoot articulations. No dislocation. Alignment of the remaining metatarsals and tarsometatarsal joints is maintained without Lisfranc malalignment.\n\nSOFT TISSUES: Soft tissues demonstrate no focal acute abnormality.\n\nIMPRESSION:\n1. Moderate hallux valgus deformity with moderate first metatarsophalangeal degenerative changes.\n2. Mild degenerative changes of the interphalangeal and midfoot articulations.\n3. Small Achilles insertional enthesophyte.\n4. No acute osseous abnormality.",
 "839da5a3-d213-474e-bb61-f5326f917106": "FINDINGS:\nNormal cervical mineralization and architecture.\n\nStraightening of the cervical lordosis.\n\nThe vertebral body and disc space heights are preserved.\n\nNo acute fracture is identified.\n\nNo evidence of spondylolisthesis or prevertebral soft tissue swelling.\n\nThe odontoid process is intact.\n\nThe craniocervical junction is intact.\n\nNo facet dislocation.\n\nNo lytic or blastic lesions.\n\nThe paravertebral soft tissues are normal.\n\nIMPRESSION:\n1. Straightening of the cervical lordosis, which may be positional or related to muscle spasm.\n2. No acute fracture or spondylolisthesis.",
 "8475ae7b-8d0b-4756-9772-b20b98459143": "FINDINGS:\nBRAIN: Cerebral hemispheres demonstrate preserved gray-white matter differentiation. No focal parenchymal T2 signal abnormality identified on the provided images. No evidence of acute diffusion restriction to suggest acute infarction. No focal mass lesion or significant mass effect. No obvious traumatic cerebral contusion identified on the provided sequences. No evidence of acute intracranial hemorrhage identified on the provided images. No obvious intraparenchymal, subarachnoid, intraventricular, or subdural hemorrhage. No midline shift or significant sulcal effacement. No obvious extra-axial fluid collection. Cerebellum and brainstem are grossly unremarkable on the provided sequences. No gross abnormality of the sellar/parasellar region identified on the provided images. No cerebellar tonsillar ectopia. The central arterial and venous flow voids are patent.\n\nVENTRICLES: Ventricular system is normal in size and configuration. No hydrocephalus. Fourth ventricle is normal in caliber.\n\nORBITS: Visualized orbital structures are grossly unremarkable.\n\nSINUSES AND MASTOIDS: Visualized paranasal sinuses and mastoid air cells demonstrate no significant abnormality.\n\nBONES: No gross calvarial abnormality identified on the provided images. No significant scalp soft tissue collection evident.\n\nOTHER FINDINGS:\n\nIMPRESSION:\n1. No acute intracranial abnormality identified on the provided MRI sequences.\n2. No acute infarct, intracranial hemorrhage, extra-axial collection, mass effect, or midline shift.\n3. No obvious MRI evidence of traumatic cerebral contusion.",
 "856fb1a7-3dd5-49a1-bf6d-9ed2cc551ec9": "FINDINGS:\nROTATOR CUFF TENDONS AND MUSCLES: Supraspinatus tendinosis. The infraspinatus, subscapularis and teres minor tendons are normal.\n\nLIGAMENTS: Visualized ligaments are normal.\n\nBICEPS ANCHOR COMPLEX: Normal\n\nGLENOHUMERAL JOINT: Mild fluid within the glenohumeral joint. The articular cartilage is preserved. The labrum is intact.\n\nACROMIOCLAVICULAR JOINT: Mild degenerative changes of the acromioclavicular joint.\n\nBONES: Edema at the anteroinferior aspect of the humeral head. No acute fracture, dislocation, or aggressive osseous lesion.\n\nPERIARTICULAR SOFT TISSUE: Edema within the anterior deltoid muscle. Subdeltoid and subacromial fluid.\n\nIMPRESSION:\n1. Supraspinatus tendinosis with subdeltoid and subacromial fluid.\n2. Edema at the anteroinferior aspect of the humeral head and within the anterior deltoid muscle.\n3. Mild glenohumeral joint fluid.\n4. Mild degenerative changes of the acromioclavicular joint.",
 "866f4dfa-c60a-4cce-9511-42fbb1389644": "FINDINGS:\nLUNGS: No focal airspace consolidation.\n\nPLEURAL SPACES: No pleural effusion. No pneumothorax.\n\nBONES: No visible acute rib fracture.\n\nIMPRESSION:\nNo visible acute rib fracture. No pneumothorax.",
 "881922d3-fd6c-4f33-a5dd-6df23632220d": "FINDINGS:\nBONES: Mild chronic-appearing irregularity/remodeling of the distal phalangeal tufts, without a definite acute fracture line. No acute fracture. No focal osseous erosion or aggressive osseous lesion.\n\nJOINTS: Mild multifocal interphalangeal joint space narrowing and marginal osteophyte formation, consistent with osteoarthritis. Mild degenerative changes are most evident at the distal interphalangeal joints. No dislocation identified. Alignment is maintained.\n\nSOFT TISSUES: Soft tissues are unremarkable.\n\nIMPRESSION:\n1. No acute fracture or dislocation.\n2. Mild multifocal interphalangeal osteoarthritis.\n3. Chronic-appearing distal phalangeal tuft irregularity without definite acute osseous abnormality.",
 "88f779c0-bcea-4f26-b8e0-9b718325eb4b": "FINDINGS:\nBRAIN: Bilateral cerebral sulci and cisterns are prominent, suggestive of generalized cerebral atrophy. Punctate non-diffusion restricting FLAIR hyperintensities are seen in bilateral periventricular and subcortical white matter, suggestive of Fazekas 1 leukoaraiosis. No intracranial mass or hemorrhage. No midline shift or extra-axial fluid collection. No cerebellar tonsillar ectopia. The central arterial and venous flow voids are patent.\n\nVENTRICLES: The ventricles are normal in size and configuration.\n\nORBITS: The globes and retro-orbital structures are symmetric and without abnormality.\n\nSINUSES AND MASTOIDS: Mucosal thickening in the left maxillary, sphenoid and ethmoid sinuses, suggestive of infective/inflammatory etiology.\n\nBONES: The calvarium and skull base demonstrate normal signal intensity.\n\nIMPRESSION:\n1. Generalized cerebral atrophy.\n2. Fazekas 1 leukoaraiosis.\n3. Mucosal thickening in the left maxillary, sphenoid and ethmoid sinuses, suggestive of infective/inflammatory etiology.",
 "8986b83e-6f00-4beb-8a7c-a057441a151e": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\nNo acute osseous abnormality.",
 "89cb3fe2-9aee-4e4a-85c3-e0e5a099b911": "FINDINGS:\nVERTEBRAE: No acute fracture, subluxation, or aggressive osseous lesion. Vertebral body heights are maintained. Normal marrow signal throughout.\n\nALIGNMENT: Kyphotic deformity of the thoracic spine.\n\nSPINAL CORD: The thoracic spinal cord is normal in signal intensity and caliber. The conus medullaris terminates at the L1-L2 level.\n\nDISCS/DEGENERATIVE CHANGES: There is pronounced multilevel thoracic spondylosis with kyphotic deformity and multilevel disc space narrowing, endplate remodeling, and osteophyte formation, and facet arthropathy, greatest in the mid to lower thoracic spine. Multilevel neural foraminal narrowing is present, right predominant in the upper thoracic spine and left predominant at several mid/lower thoracic levels. At T4-T5, a left-predominant disc-osteophyte complex produces mild-to-moderate spinal canal stenosis and mild ventral cord indentation. Advanced degenerative disc/endplate disease is also present at T12-L1 with broad posterior disc-osteophyte complex but no significant canal stenosis.\n\nPARASPINAL SOFT TISSUES: The paravertebral soft tissues demonstrate normal signal and morphology.\n\nIMPRESSION:\n1. Pronounced multilevel thoracic spondylosis with kyphotic deformity, greatest in the mid to lower thoracic spine.\n2. Multilevel neural foraminal narrowing, right predominant in the upper thoracic spine and left predominant at several mid/lower thoracic levels.\n3. Mild-to-moderate spinal canal stenosis with mild ventral cord indentation at T4-T5.\n4. Advanced degenerative disc/endplate disease at T12-L1 with broad posterior disc-osteophyte complex.\n5. No high-grade thoracic spinal canal stenosis.",
 "8ab1a654-3318-4ec3-803a-5bf97d466f91": "FINDINGS:\nVERTEBRAE: Multilevel degenerative changes are noted in the visualized vertebral bodies with marginal osteophyte formation and endplate sclerosis. There is reduction in the height of the L4 vertebral body with anterior wedging, suggestive of a compression deformity. Multilevel osteophytes are noted. No obvious acute displaced fracture or significant spondylolisthesis is identified.\n\nDISC SPACES: Intervertebral disc spaces are relatively maintained except for mild multilevel degenerative narrowing.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Reduction in the height of the L4 vertebral body with anterior wedging, suggestive of a compression deformity.\n2. Multilevel lumbar degenerative changes.\n3. No obvious acute displaced fracture or significant spondylolisthesis.",
 "8b404fa8-569c-43c9-8785-bd31382ed36a": "FINDINGS:\nSINUSES: The bilateral frontal sinuses are hypoplastic. The remaining visualized paranasal sinuses appear clear, without definite air-fluid level or significant mucosal thickening.\n\nORBITS: No obvious orbital abnormality.\n\nSOFT TISSUES: Unremarkable.\n\nBONES: No acute osseous abnormality identified.\n\nIMPRESSION:\n1. Bilateral hypoplastic frontal sinuses.\n2. No acute paranasal sinus abnormality evident on plain film examination.",
 "956291b1-faba-4c9f-991b-7cc282287ed9": "FINDINGS:\nSINUSES: No air-fluid level or paranasal opacification identified.\n\nORBITS: No obvious orbital abnormality.\n\nSOFT TISSUES: Unremarkable.\n\nBONES: No fracture or lesion\n\nIMPRESSION:\nNo acute paranasal sinus abnormality evident on plain film examination.",
 "959135c4-6c14-4342-ab67-d63ffd6ed6fe": "FINDINGS:\nLUNGS/AIRWAYS: Subpleural reticulation with associated intra- and interlobular septal thickening is seen in both lungs, predominantly involving the lower lobes with a symmetrical distribution and apicobasal gradient. Associated traction bronchiectatic and bronchiolectatic changes are present. No honeycombing is identified. No significant ground-glass opacities are seen. No focal consolidation, suspicious pulmonary nodule, or mass lesion is identified.\n\nPLEURA: No pleural effusion or pleural thickening. No pneumothorax.\n\nCARDIOVASCULAR: Cardiac chambers are within normal size limits. No pericardial effusion. No coronary artery calcification is identified. Thoracic aorta is normal in caliber, measuring approximately 24.6 mm. Main pulmonary artery is normal in caliber, measuring approximately 23.4 mm.\n\nMEDIASTINAL/HILAR LYMPH NODES: No enlarged mediastinal or hilar lymph nodes. Few subcentimeter mediastinal lymph nodes are present, including prevascular, pretracheal, and precarinal nodes, the largest precarinal node measuring approximately 8.8 mm in short-axis diameter.\n\nCHEST WALL/MUSCULOSKELETAL: Stable calcific foci are scattered in both breasts. Diffuse osteopenia. Mild multilevel degenerative changes of the thoracic spine. No acute osseous abnormality identified.\n\nLINES/TUBES/SUPPORT DEVICES: None.\n\nOTHER FINDINGS:\n\nVisualized lower neck structures are unremarkable. Visualized upper abdominal structures reveal no significant abnormality on this examination. Small hiatal hernia is seen.\n\nIMPRESSION:\n1. Bilateral lower lobe predominant subpleural reticulation, intra- and interlobular septal thickening, and traction bronchiectatic/bronchiolectatic changes with apicobasal gradient, without honeycombing or significant ground-glass opacities, consistent with fibrotic interstitial lung disease. Overall imaging appearance is most compatible with a probable UIP pattern.\n2. No suspicious pulmonary nodule, focal consolidation, pleural effusion, or mediastinal lymphadenopathy.\n3. No significant interval change compared to the CT chest, likely benign.\n4. Osteopenia and mild thoracic spondylotic degenerative changes.",
 "9b0e5e9c-5c6d-4391-898b-44be789e0b6c": "FINDINGS:\nLIGAMENTS:\n\nSCAPHOLUNATE: Scapholunate interval is maintained without gross widening. No definite disruption of the scapholunate ligament identified on the provided sequences.\n\nLUNOTRIQUETRAL: No definite disruption of the lunotriquetral ligament identified on the provided sequences.\n\nTENDONS:\n\nFLEXOR COMPARTMENTS: Visualized flexor tendons are intact without significant tenosynovitis or tendon tear.\n\nEXTENSOR COMPARTMENTS: Visualized extensor tendons are intact without significant tenosynovitis or tendon tear.\n\nNERVES:\n\nMEDIAN: The median nerve demonstrates normal course and signal intensity within the carpal tunnel. No gross abnormality of the visualized carpal tunnel.\n\nULNAR: The ulnar nerve demonstrates normal course and signal intensity within Guyon's canal. No gross abnormality of the visualized Guyon canal.\n\nBONES: No acute fracture or dislocation identified on the provided images. No focal bone marrow edema or suspicious osseous lesion evident.\n\nMUSCLES: The musculature is of normal bulk and signal intensity.\n\nJOINTS: Carpal alignment is maintained. Joint spaces are relatively maintained without significant degenerative change. No significant wrist joint effusion.\n\nCARTILAGE: The articular cartilage is preserved in thickness and signal intensity.\n\nTRIANGULAR FIBROCARTILAGE: TFCC demonstrates mild heterogeneous increased signal, particularly along its ulnar-sided attachment, compatible with degenerative/intrasubstance signal. No definite full-thickness TFCC tear or significant DRUJ effusion identified on the available images.\n\nOTHER FINDINGS:\n\nNo focal soft tissue mass or significant fluid collection identified.\n\nIMPRESSION:\n1. No acute fracture or dislocation identified.\n2. Mild heterogeneous/intrasubstance signal within the TFCC, without a definite full-thickness tear on the provided images.\n3. No gross scapholunate or lunotriquetral ligament disruption.\n4. No significant tendon abnormality.",
 "9e441726-c216-4eb4-bcb9-5cf9969e0f70": "FINDINGS:\nVERTEBRAE: Vertebral body heights are maintained. No focal suspicious marrow signal abnormality or acute compression deformity.\n\nALIGNMENT: Grade 1 retrolisthesis of L5 over S1. Vertebral body alignment otherwise maintained.\n\nSPINAL CORD: Conus medullaris terminates at an appropriate level and demonstrates normal signal and morphology. Visualized cauda equina nerve roots are unremarkable.\n\nDISCS/DEGENERATIVE CHANGES: Disc desiccation changes, involving the lower lumbar levels, with associated reduction in disc height at L4-L5 and L5-S1.\n\nL1-L2: No significant disc bulge. No significant spinal canal or neural foraminal stenosis.\n\nL2-L3: No significant disc bulge. No significant spinal canal or neural foraminal stenosis.\n\nL3-L4: Mild diffuse posterior disc bulge with mild ventral thecal sac indentation and mild bilateral lateral recess narrowing. Bilateral facet arthrosis. No significant central canal stenosis.\n\nL4-L5: Diffuse posterior disc bulge with moderate central canal and bilateral lateral recess stenosis. Superimposed left foraminal disc herniation with bilateral facet arthrosis, greater on the left, resulting in moderate left neural foraminal narrowing. There is impingement of the exiting left L4 and traversing left L5 nerve roots.\n\nL5-S1: Diffuse posterior disc bulge resulting in mild central canal narrowing. Bilateral facet arthrosis, greater on the left, with moderate left neural foraminal and lateral recess narrowing, resulting in impingement of the exiting left L5 nerve roots.\n\nPARASPINAL SOFT TISSUES: No significant abnormality.\n\nIMPRESSION:\n1. Degenerative disc disease predominantly at L4-L5 and L5-S1, with associated disc space narrowing and disc desiccation.\n2. L4-L5 diffuse disc bulge with superimposed left foraminal disc herniation and left-predominant facet arthrosis, resulting in moderate central canal stenosis and moderate left foraminal/lateral recess stenosis with impingement of the exiting left L4 and traversing left L5 nerve roots.\n3. L5-S1 diffuse disc bulge and left-predominant facet arthrosis causing mild central canal and moderate left foraminal/lateral recess stenosis, with impingement of the exiting left L5 nerve roots.\n4. Grade 1 retrolisthesis of L5 over S1.",
 "9f238d30-dbd0-48d2-b327-3528f6542fb5": "FINDINGS:\nVERTEBRAE: Transitional lumbosacral anatomy with sacralisation of the L5 vertebra, resulting in four non-rib-bearing lumbar-type vertebral bodies. Multilevel early degenerative spondylotic changes with marginal endplate osteophyte formation and endplate sclerosis. No fracture or osseous lesion.\n\nDISC SPACES: Preserved.\n\nSOFT TISSUES: Unremarkable.\n\nIMPRESSION:\n1. Transitional lumbosacral anatomy with sacralisation of the L5 vertebra.\n2. Multilevel early degenerative spondylotic changes.\n3. No acute abnormality evident in the lumbar spine.",
 "a2771d2d-a706-4501-a0df-11045de0b0df": "FINDINGS:\nLOWER THORAX: No pleural effusion or basilar consolidation.\n\nLIVER: Liver is normal in size and contour with preserved parenchymal signal intensity. No definite focal hepatic lesion is seen.\n\nGALLBLADDER AND BILE DUCTS: Gallbladder is adequately distended. No definite intraluminal calculus or focal mural abnormality is identified. Common hepatic duct and common bile duct are normal in caliber with smooth distal tapering. No definite intraductal filling defect or obstructing lesion is seen. No intrahepatic biliary radicle dilatation is noted.\n\nPANCREAS: Pancreas is normal in size and signal intensity. Main pancreatic duct is not dilated. No definite focal pancreatic lesion or peripancreatic collection is identified.\n\nSPLEEN: A cystic lesion is noted within the splenic parenchyma. Spleen is normal in size.\n\nADRENALS: Visualized adrenal glands are unremarkable.\n\nKIDNEYS: Left kidney is not visualized, consistent with the known history of solitary kidney. Right kidney is visualized and shows a large well-defined cyst arising from its lower pole, measuring approximately 66 \u00d7 60 mm. No right hydronephrosis is seen.\n\nSTOMACH AND BOWEL: The visualized portions of the stomach and bowel are unremarkable. No bowel wall thickening or obstruction.\n\nLYMPH NODES: No significant upper abdominal lymphadenopathy or free fluid is seen.\n\nVASCULATURE: The abdominal aorta and major visceral vessels are patent. No evidence of aneurysm or dissection.\n\nIMPRESSION:\n1. Non-visualization of the left kidney, consistent with solitary right kidney.\n2. Large right renal lower-pole cyst measuring approximately 66 \u00d7 60 mm.\n3. Small splenic cyst.\n4. No intrahepatic or extrahepatic biliary dilatation or definite choledocholithiasis. No pancreatic ductal dilatation.",
 "a67db807-aa38-4995-892f-231f079c570f": "FINDINGS:\nVERTEBRAE: Vertebral body heights are maintained. No acute compression fracture or significant focal marrow signal abnormality is seen.\n\nALIGNMENT: Lumbar lordosis is mildly straightened. No significant spondylolisthesis is identified.\n\nSPINAL CORD: Significant crowding of the cauda equina nerve roots is seen at L4-L5 secondary to severe canal stenosis. Conus medullaris terminates at approximately the L1 level and demonstrates normal signal intensity.\n\nDISCS/DEGENERATIVE CHANGES: Multilevel intervertebral disc desiccation is noted with multilevel diffuse disc bulges, predominantly involving the lower lumbar levels. Multilevel ligamentum flavum hypertrophy, most pronounced at the lower lumbar levels, contributing to central canal and lateral recess narrowing. Multilevel facet joint hypertrophic/degenerative changes are noted.\n\nL1-L2: Diffuse disc bulge causing mild left neural foraminal narrowing. No significant central canal stenosis.\n\nL2-L3: Diffuse disc bulge with left neural foraminal narrowing, abutting the left exiting nerve root.\n\nL3-L4: Diffuse disc bulge with bilateral neural foraminal narrowing, abutting the bilateral exiting nerve roots. Associated ligamentum flavum hypertrophy contributes to thecal sac narrowing.\n\nL4-L5: Diffuse disc bulge with superimposed posterior central disc protrusion. There is significant central spinal canal stenosis, with AP canal diameter measuring approximately 5 mm. Marked crowding/compression of the cauda equina nerve roots is noted. Bilateral lateral recess and neural foraminal narrowing is seen with compression of the bilateral traversing and exiting nerve roots. Associated ligamentum flavum hypertrophy and posterior longitudinal ligament elevation/lifting contribute to the canal stenosis.\n\nL5-S1: Diffuse disc bulge with superimposed posterior central disc protrusion. Central canal AP diameter measures approximately 9.8 mm, suggestive of mild central canal narrowing. Bilateral neural foraminal narrowing is noted with compression of the bilateral exiting nerve roots.\n\nPARASPINAL SOFT TISSUES: No significant abnormality identified.\n\nIMPRESSION:\n1. Multilevel lumbar spondylosis with multilevel intervertebral disc desiccation and diffuse disc bulges.\n2. L4-L5: Posterior central disc protrusion with severe central canal stenosis (AP diameter ~5 mm), associated with marked crowding/compression of the cauda equina nerve roots, bilateral lateral recess and neural foraminal narrowing with compression of bilateral traversing and exiting nerve roots.\n3. L5-S1: Posterior central disc protrusion with mild central canal narrowing (AP diameter ~9.8 mm) and bilateral neural foraminal narrowing with compression of bilateral exiting nerve roots.\n4. L2-L3 left neural foraminal narrowing with abutment of the left exiting nerve root.\n5. L3-L4 bilateral neural foraminal narrowing with abutment of bilateral exiting nerve roots.\n6. Multilevel ligamentum flavum hypertrophy and facet arthropathy, contributing to spinal canal and foraminal stenosis.\n7. Posterior longitudinal ligament elevation/lifting at the lower lumbar level related to the posterior disc protrusion.",
 "a8abd8d5-b136-4415-96cf-f08729c3bf2c": "FINDINGS:\nBONES: No acute fracture or focal osseous lesion.\n\nJOINTS: Degenerative changes with multiple reduced joint spaces. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Degenerative changes with multiple reduced joint spaces.\n2. No acute osseous abnormality.",
 "a8d31bb3-825c-46fe-ab14-2c038ec47778": "FINDINGS:\nROTATOR CUFF TENDONS AND MUSCLES: Supraspinatus and subscapularis tendinosis. The infraspinatus and teres minor tendons are normal.\n\nLIGAMENTS: Visualized ligaments are normal.\n\nBICEPS ANCHOR COMPLEX: Mild fluid along the biceps tendon.\n\nGLENOHUMERAL JOINT: Mild fluid in the glenohumeral joint. The articular cartilage is preserved. The labrum is intact.\n\nACROMIOCLAVICULAR JOINT: Mild degenerative changes in the acromioclavicular joint.\n\nBONES: No acute fracture, dislocation, or aggressive osseous lesion. Normal bone marrow signal.\n\nPERIARTICULAR SOFT TISSUE: Mild subdeltoid and subacromial fluid.\n\nIMPRESSION:\n1. Supraspinatus and subscapularis tendinosis.\n2. Mild subdeltoid and subacromial fluid with mild fluid along the biceps tendon.\n3. Mild fluid in the glenohumeral joint.\n4. Mild degenerative changes in the acromioclavicular joint.",
 "a9f7e486-1730-4312-93df-cc9e7049b2ed": "FINDINGS:\nBONES: Complete mildly displaced midshaft acute fracture of the right clavicle with superior displacement and overriding of the proximal fragment. No humeral fracture. No focal lesion.\n\nJOINTS: Mild glenohumeral osteoarthritic changes. No dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Complete mildly displaced midshaft acute fracture of the right clavicle with superior displacement and overriding of the proximal fragment.\n2. Mild glenohumeral osteoarthritic changes.\n3. No humeral fracture or dislocation.",
 "aa9fc78f-f9c6-4a25-85ec-86ec0c66b216": "FINDINGS:\nBONES: Calcaneal spur is noted, likely at the plantar aspect of the calcaneus. No acute fracture is evident. No focal acute osseous abnormality identified.\n\nJOINTS: Degenerative osteoarthritic changes are noted in the visualized joint. Multiple joint spaces are reduced, with associated marginal osteophyte formation and subchondral degenerative changes. No dislocation is evident.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Multicompartmental degenerative osteoarthritic changes with multiple reduced joint spaces.\n2. Calcaneal spur.\n3. No acute osseous abnormality.",
 "b33e48c9-3967-42e7-813d-45a72a2a57aa": "FINDINGS:\nCOMMON FEMORAL ARTERY: The bilateral common femoral and profunda femoris arteries are patent, with preserved color Doppler flow. No focal significant elevation in peak systolic velocity to suggest hemodynamically significant stenosis.\n\nSUPERFICIAL FEMORAL ARTERY: The proximal, mid and distal superficial femoral arteries are patent, with preserved color Doppler flow.\n\nPOPLITEAL ARTERY: The popliteal arteries are patent, with preserved color Doppler flow.\n\nCALF ARTERIES: The posterior tibial and dorsalis pedis arteries are patent, with preserved color Doppler flow.\n\nOTHER FINDINGS:\n\nBilateral lower extremity subcutaneous edema is present.\n\nIMPRESSION:\n1. Bilateral lower extremity subcutaneous edema.\n2. No flow limiting stenosis or occlusion in the bilateral lower extremity.",
 "b7fc16f0-51ec-425f-bb3d-79fa950b00af": "FINDINGS:\nTENDONS:\n\nFLEXORS: Iliopsoas tendon is intact.\n\nEXTENSORS/HAMSTRING: Rectus femoris and proximal hamstring tendons are grossly intact. Gluteus maximus tendon is intact.\n\nABDUCTORS: Mild insertional tendinopathy of the gluteus medius/minimus tendons at the greater trochanter.\n\nADDUCTORS: The adductor tendons are intact at their origin.\n\nROTATORS: The external rotator tendons are intact.\n\nBONES: Mild subchondral marrow signal abnormality along the superolateral acetabular margin. Mild degenerative osteophyte formation along the femoral head-neck junction. No acute fracture or dislocation. Femoral head maintains normal contour without evidence of avascular necrosis. No femoral head collapse or osteonecrosis. No focal aggressive osseous lesion identified. Visualized portions of the sacrum and pelvic bones are grossly intact. Visualized sacroiliac joints are maintained. No significant marrow edema or erosive change. Pubic symphysis is unremarkable.\n\nMUSCLES: Mild adjacent peritrochanteric soft-tissue edema. No focal muscle tear or significant muscle atrophy.\n\nFLUID: Small left hip joint effusion. Mild fluid within the greater trochanteric bursal region, compatible with mild trochanteric bursitis. No significant iliopsoas bursitis. No intra-articular loose body identified.\n\nLABRUM: Degenerative signal and irregularity involving the anterosuperior/superior acetabular labrum, suspicious for a degenerative labral tear. No large paralabral cyst identified.\n\nCARTILAGE: Mild degenerative changes of the left hip with mild chondral thinning/irregularity, predominantly superiorly.\n\nOTHER FINDINGS:\n\nNo significant femoroacetabular impingement morphology identified on the provided images. No focal soft-tissue mass or drainable fluid collection identified on the provided images. Visualized pelvic structures are grossly unremarkable. No gross abnormality of the visualized neurovascular structures.\n\nIMPRESSION:\n1. Mild left hip osteoarthrosis with superior chondral thinning/irregularity and small joint effusion.\n2. Degenerative tearing/irregularity of the anterosuperior acetabular labrum.\n3. Mild gluteus medius/minimus insertional tendinopathy with mild adjacent greater trochanteric bursitis.\n4. No acute fracture or avascular necrosis of the left femoral head.\n5. No focal aggressive osseous lesion or significant muscle/tendon tear.",
 "b92efe0b-9644-4f69-833f-64eded759dfb": "FINDINGS:\nLUNGS: Lungs are clear. No focal air-space consolidation or pulmonary edema. No discrete pulmonary mass or suspicious focal parenchymal opacity identified on radiograph.\n\nPLEURAL SPACES: No pleural effusion or pneumothorax.\n\nHEART: Cardiac silhouette is within normal size limits.\n\nMEDIASTINUM/HILA: There is bilateral hilar prominence, appearing relatively symmetric, suspicious for bilateral hilar lymphadenopathy. Cardiomediastinal silhouette is mildly prominent.\n\nDIAPHRAGM: Diaphragms are unremarkable. No subdiaphragmatic free air.\n\nBONES: No acute osseous abnormality identified on this examination.\n\nSOFT TISSUES: No acute soft tissue abnormality identified on the provided images.\n\nIMPRESSION:\nBilateral hilar prominence, suspicious for bilateral hilar lymphadenopathy.",
 "bcc23ba7-8291-4c7d-8693-6eb23712a447": "FINDINGS:\nBONES: Minimal enthesopathic change is present at the Achilles tendon insertion. A well-corticated os peroneum is incidentally noted. There is no acute fracture or focal destructive osseous abnormality. No significant plantar calcaneal spur is identified. Calcaneal morphology is maintained.\n\nJOINTS: Subtalar alignment is maintained. No dislocation. The joint spaces are normal.\n\nSOFT TISSUES: Mild soft tissue edema over the heel.\n\nIMPRESSION:\n1. Minimal enthesopathic change at the Achilles tendon insertion.\n2. Incidental os peroneum.\n3. No acute fracture.",
 "c28fe8d5-1b78-4f89-beaf-c5bf1adb4115": "FINDINGS:\nBONES: Degenerative changes. No acute fracture or focal osseous lesion. No sternoclavicular dislocation.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Degenerative changes.\n2. No acute osseous abnormality.",
 "cae63587-9491-47ca-8aaf-d4166a856ffe": "FINDINGS:\nBRAIN: No acute intracranial hemorrhage, acute large territorial infarction, mass effect, midline shift or extra-axial fluid collection. The gray-white matter differentiation is maintained. The basal cisterns are patent. Brain parenchyma appears normal in attenuation and morphology.\n\nVENTRICLES: The ventricles and cortical sulci are within normal limits.\n\nORBITS: The globes and retrobulbar soft tissues are symmetric and unremarkable.\n\nSINUSES AND MASTOIDS: Moderate mucosal thickening is present in the right sphenoid sinus. Mild mucosal thickening is present in the bilateral frontal and ethmoid sinuses and left sphenoid sinus. The visualized mastoid air cells are clear.\n\nSOFT TISSUES: No significant facial or scalp soft tissue swelling evident.\n\nBONES: No acute calvarial abnormality.\n\nIMPRESSION:\n1. Paranasal sinus mucosal thickening, moderate in the right sphenoid sinus and mild in the bilateral frontal and ethmoid sinuses and left sphenoid sinus.\n2. No acute intracranial abnormality.",
 "cdf174d7-ba12-40ab-99b0-e9d90e55a9fd": "FINDINGS:\nLUNGS: Nodule observed in right upper and middle zones. Consolidation observed in right lower zone. The left lung is clear.\n\nPLEURAL SPACES: Pleural effusion is observed on the right. No pneumothorax is seen. There is no left pleural effusion.\n\nHEART: Cardiac silhouette is within normal size limits.\n\nMEDIASTINUM/HILA: The cardiomediastinal silhouette is within normal limits. The trachea is centrally located. Both hila appear normal. No mediastinal widening or hilar enlargement.\n\nDIAPHRAGM: No hiatus hernia. Diaphragms are otherwise unremarkable. No subdiaphragmatic free air.\n\nBONES: The bony thorax appears unremarkable. Degenerative changes of the spine.\n\nSOFT TISSUES: No acute soft tissue abnormality identified on the provided images.\n\nIMPRESSION:\n1. Nodule in the right upper and middle zones.\n2. Consolidation in the right lower zone.\n3. Right pleural effusion.",
 "d14307e6-9059-4dc2-bbd4-6cc58e71f54c": "FINDINGS:\nLIVER: Liver is normal in size and demonstrates homogeneous parenchymal signal intensity. No focal hepatic lesion is identified.\n\nGALLBLADDER: Gallbladder is adequately distended with no obvious intraluminal calculus or wall thickening.\n\nBILIARY TREE: The intrahepatic biliary radicles are not dilated. Common hepatic duct and common bile duct are normal in calibre, with no obvious intraductal filling defect or focal stricture. The CBD is seen to drain normally into the second part of the duodenum.\n\nPANCREATIC DUCT: Main pancreatic duct is normal in calibre and course.\n\nPANCREAS: Normal in size and signal. No obvious focal pancreatic lesion or peripancreatic collection is seen.\n\nSPLEEN: Spleen and adrenal glands are unremarkable.\n\nKIDNEYS: Both kidneys are normal in size and morphology. Few small simple cortical cysts are noted in both kidneys, without suspicious internal septation or solid component. No hydronephrosis is seen.\n\nOTHER FINDINGS:\n\nNo significant ascites or enlarged upper abdominal lymph nodes are identified.\n\nIMPRESSION:\n1. No evidence of intrahepatic or extrahepatic biliary dilatation.\n2. No obvious choledocholithiasis or biliary stricture.\n3. Few small simple bilateral renal cortical cysts.",
 "d8c7357a-d804-4a1b-9eb7-4de840baf9f7": "FINDINGS:\nBONES: Few small subchondral cystic/geode-like lucencies are seen in the lateral tibial plateau and medial femoral condyle. No acute fracture. No focal aggressive osseous lesion.\n\nJOINTS: Mild narrowing of the medial tibiofemoral compartment with small marginal osteophyte formation. Lateral tibiofemoral and patellofemoral joint spaces are relatively maintained. No dislocation. No significant joint effusion evident on the provided radiograph.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild degenerative changes with small subchondral cysts.\n2. No acute osseous abnormality.",
 "ddac2ed8-a4f4-4f37-960a-49b27be1d7e5": "FINDINGS:\nBONES: Moderate plantar calcaneal enthesophyte is present. No acute fracture is identified.\n\nJOINTS: Mild degenerative changes are present in the midfoot with small dorsal marginal osteophytes. No dislocation is identified.\n\nSOFT TISSUES: The soft tissues are unremarkable.\n\nIMPRESSION:\n1. Mild degenerative changes in the midfoot.\n2. Moderate plantar calcaneal enthesophyte.\n3. No acute osseous abnormality.",
 "e9721e29-b1d3-4b38-9369-30cec1e63bc0": "FINDINGS:\nBOWEL: Nonobstructive bowel gas pattern without evidence of dilated small or large bowel loops. Moderate colonic stool burden. No abnormal bowel wall thickening or pneumoperitoneum evident on this single supine view.\n\nPERITONEUM/SOFT TISSUES: Multiple pelvic phleboliths are noted. No appreciable free air.\n\nORGANS OUTLINE: No definite radiopaque calculus identified overlying the expected locations of either kidney or along the expected course of the ureters. Liver, spleen, and renal outlines are unremarkable where visualized.\n\nBONES: Visualized osseous structures demonstrate mild degenerative changes of the lower lumbar spine and sacroiliac joints. No acute osseous abnormality identified.\n\nOTHER FINDINGS:\n\nVisualized lung bases are grossly clear.\n\nIMPRESSION:\n1. No definite radiopaque renal or ureteral calculus identified.\n2. Nonobstructive bowel gas pattern with moderate colonic stool burden.\n3. Pelvic phleboliths.",
 "f6b3620a-65d6-49f7-9fe8-13a3ed4681fc": "FINDINGS:\nUTERUS/CERVIX: Anteverted and measures 9.0 x 5.08 x 4.1 cm, with a volume of 100.9 cm3. Myometrium is homogeneous. No focal or diffuse lesion.\nThe cervix is normal in appearance.\n\nENDOMETRIUM: Central and measures ~ 1.2 cm in thickness.\n\nRIGHT OVARY: Not visualized due to bowel.\n\nLEFT OVARY: Not visualized due to bowel.\n\nFREE FLUID: No free fluid in the pelvis.\n\nOTHER FINDINGS:\n\nUrinary bladder appears normal.\n\nIMPRESSION:\n1. Unremarkable pelvic ultrasound.\n2. Ovaries and bilateral adnexae not visualized due to bowel.",
 "f7b1aa6a-9ef5-41b0-9175-55c9b80d5cd8": "FINDINGS:\nBRAIN: There is a left cerebral convexity subdural collection, predominantly low attenuation with a small residual higher-attenuation component, measuring up to approximately 12-13 mm in maximal thickness on the provided images. This likely represents a subacute/chronic subdural hematoma with possible small acute/subacute component. Mild associated mass effect upon the underlying left cerebral hemisphere with mild effacement of adjacent cortical sulci. No significant midline shift is identified on the provided images. No large territorial acute infarction. No intraparenchymal hemorrhage identified. Basal cisterns remain patent. Gray-white matter differentiation is preserved.\n\nVENTRICLES: Mild generalized cerebral volume loss with prominence of the ventricles and cortical sulci. No hydrocephalus.\n\nORBITS: The globes and retrobulbar soft tissues are symmetric and unremarkable.\n\nSINUSES AND MASTOIDS: Visualized paranasal sinuses and mastoid air cells are without significant acute abnormality.\n\nSOFT TISSUES: No significant facial or scalp soft tissue swelling evident.\n\nBONES: Postsurgical changes of the calvarium are present, compatible with prior evacuation/craniotomy. No acute calvarial fracture identified.\n\nIMPRESSION:\n1. Left cerebral convexity subdural hematoma/collection, measuring up to approximately 12-13 mm in thickness, with mixed attenuation suggesting subacute/chronic blood products with a small acute/subacute component.\n2. Mild associated mass effect without significant midline shift.\n3. Postsurgical changes related to prior subdural hematoma evacuation.\n4. No hydrocephalus or large acute territorial infarction."
}

## 7. Write `submission.csv` and re-validate

In [ ]:
CACHE = "refined_reports.json"
cached = json.load(open(CACHE, encoding="utf-8")) if os.path.exists(CACHE) else {}

refined = {}
for row in test.to_dict("records"):
    cid = row["case_id"]
    refined[cid] = refine(row, drafts[cid]) if USE_LLM else cached.get(cid, drafts[cid])
if USE_LLM:
    json.dump(refined, open(CACHE, "w", encoding="utf-8"), indent=1)

# every emitted report goes back through the validator
issue_counts = {}
for row in test.to_dict("records"):
    cid = row["case_id"]
    res = validate(row, refined[cid], traces[cid], vocab=generator.model.vocab)
    for i in res.issues:
        # `untouched_field` compares against the deterministic router's routing
        # decisions, which the refinement stage is allowed to correct
        if i.kind == "untouched_field":
            continue
        issue_counts[i.kind] = issue_counts.get(i.kind, 0) + 1

order = sample["case_id"].tolist()
assert set(order) == set(refined), "case_id set differs from sample_submission"
assert len(order) == len(set(order)) == len(test) == 132, "row count / duplicate check failed"

with open("submission.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh, quoting=csv.QUOTE_ALL, lineterminator="\n")
    w.writerow(["case_id", "report"])
    for cid in order:
        w.writerow([cid, refined[cid].strip()])

print("submission.csv rows:", len(order))
print("validation issues  :", json.dumps(issue_counts, sort_keys=True) or "none")
print()
print(refined[order[0]])
